# FastCCC PFC70 CPDBv5 healthy-aging CCC analysis Mainline v2

**Goal**: identify age-associated cell-cell communication (CCC) changes in 70 multi-cohort human PFC donors.

**Tools**: FastCCC + CellPhoneDB v5.0.0 | **statistical unit**: donor (N=70)

**Main design**: continuous-age Spearman discovery as the primary analysis; Young adult/Adult/Elderly grouped views as companion visualization; OLS, leave-one-cohort-out, cohort-block permutation, and residual Spearman as cohort-aware sensitivity annotations.

**Data**: age 27-89 years, multi-cohort; Young adult (<40), Adult (40-69), Elderly (>=70).

---
**Analysis layers**: Stage 0 metadata -> Stage 1 FastCCC -> Stage 2 CPDB annotation -> Stage 3 donor aggregation -> Stage 4 broad cell pair -> Stage 5 fine cell pair -> Stage 6 classification/function -> Stage 7 LR driver -> Stage 8 network centrality -> Stage 9 sensitivity and evidence summary


In [ ]:
import os, gc, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as st
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import fastccc
from IPython.display import display, Markdown

sys.path.insert(0, str(Path.home()))
from ccc_models import (fit_age_model, fit_group_stats,
                         run_models_for_panel, direction_consistent)
from ccc_cpdb import (load_cpdb_annotations, build_lr_annotation_map,
                       load_receptor_tf, build_fastccc_long, parse_cellpair_index)
from ccc_network import (build_donor_graph, compute_centrality_metrics,
                          build_network_metrics_panel, build_group_network)
from ccc_viz import (set_pub_style, save_fig, AGE_PAL, CURVE_PAL,
                      plot_donor_composition, plot_loess_trajectory,
                      plot_violin_strip, plot_forest,
                      plot_lr_bubble, plot_classification_triptych,
                      plot_young_old_networks, plot_centrality_heatmap,
                      plot_sender_receiver_scatter, plot_receptor_tf_sankey)

warnings.filterwarnings('ignore')
set_pub_style()
CURVE_PAL.update({
    'monotonic_up': '#C84E4E',
    'monotonic_down': '#3B74AF',
    'middle_high': '#D39C32',
    'middle_low': '#4A9B74',
    'late_onset': '#8A5FBF',
    'early_shift': '#E07A5F',
    'flat_or_uncertain': '#9AA0A6',
    'insufficient_groups': '#C7C7C7',
})
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 60)
print('Imports OK')


## Stage 0 — 路径 & 配置

In [ ]:
# ============================================================
# Paths and configuration
# ============================================================
BASE_DIR   = Path('/bigdata/disk1/HDD1-Data/aya/brainscope')
SAMPLE_TSV = BASE_DIR / 'metaq_clean_pipeline' / 'final_target_samples.tsv'
H5AD_DIR   = BASE_DIR / 'metaq_clean_pipeline' / 'input_h5ad'
DB_PATH    = Path('/bigdata/disk1/HDD1-Data/aya/FastCCC-main/db/CPDBv5.0.0')

OUT     = BASE_DIR / 'fastccc_pfc70_cpdbv5_v2'
TMP     = OUT / 'tmp_fastccc_input_h5ad'
RESULTS = OUT / 'per_sample_results'
FIG_DIR = Path('/home/aya/图片/70')  # manuscript-ready vector figures
TAB_DIR = OUT / 'tables'

for p in [OUT, TMP, RESULTS, FIG_DIR, TAB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

VECTOR_FIG_FORMATS = ('pdf', 'svg')

def save_fig(fig, name, outdir=FIG_DIR, formats=VECTOR_FIG_FORMATS, dpi=300):
    """Save manuscript figures as vector PDF/SVG in the manuscript figure directory."""
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    saved = []
    for fmt in formats:
        path = outdir / f'{name}.{fmt}'
        fig.savefig(path, format=fmt, dpi=dpi, bbox_inches='tight', pad_inches=0.05)
        saved.append(path)
    return saved

META_KEY           = 'celltype'
CONVERT_TYPE       = 'hgnc_symbol'
PVAL_SIG_CUTOFF    = 0.05
MIN_DONORS_PRESENT = 8
RUN_FASTCCC_IF_MISSING = True   # Set False to skip running FastCCC
RERUN_EXISTING     = False

# Age group thresholds (tertile-based from data: cuts at 48 and 66)
AGE_YOUNG_THRESH = 48.0
AGE_OLD_THRESH   = 66.0

# Group small cohorts (n<5) as 'Small_cohort'
MIN_COHORT_SIZE  = 5

print('OUT:', OUT)
print('FIG_DIR:', FIG_DIR)
print('Figure formats:', ', '.join(VECTOR_FIG_FORMATS))
assert SAMPLE_TSV.exists(), f'Missing: {SAMPLE_TSV}'
assert H5AD_DIR.exists(),   f'Missing: {H5AD_DIR}'
assert DB_PATH.exists(),    f'Missing: {DB_PATH}'
print('All paths OK')

## Stage 0 - Metadata preparation

Build donor_base metadata with literature-aligned age groups and cohort labels.
- Young adult: age < 40
- Adult: 40 <= age < 70
- Elderly: age >= 70
- Small cohorts (n < 5) are collapsed to `Small_cohort`.


In [ ]:
# ============================================================
# Stage 0: Sample table + donor_base
# ============================================================
raw = pd.read_csv(SAMPLE_TSV, sep='\t', dtype={'sample_id': str})
raw.columns = [c.strip() for c in raw.columns]
raw['h5ad_path'] = raw['sample_id'].apply(lambda s: str(H5AD_DIR / f'{s}.h5ad'))
raw = raw[raw['h5ad_path'].apply(lambda p: Path(p).exists())].copy()
print(f'Matched {len(raw)} donors')

raw['age'] = pd.to_numeric(raw['age'], errors='coerce')
raw = raw.dropna(subset=['age']).copy()

def assign_age_group(age):
    # Literature-aligned age bins: Young adult <40, Adult 40-69, Elderly >=70.
    if age < 40:
        return 'Young adult'
    elif age < 70:
        return 'Adult'
    else:
        return 'Elderly'

raw['age_group'] = raw['age'].apply(assign_age_group)

cohort_sizes = raw['cohort'].value_counts()
raw['cohort_group'] = raw['cohort'].apply(
    lambda c: c if cohort_sizes.get(c, 0) >= MIN_COHORT_SIZE else 'Small_cohort')

print('Age groups:', raw['age_group'].value_counts().to_dict())
print('Cohort groups:', raw['cohort_group'].value_counts().to_dict())

# Cell type fractions from h5ad
ct_frac_records = []
for _, row in raw.iterrows():
    sid = row['sample_id']
    try:
        adata = sc.read_h5ad(row['h5ad_path'])
        n = int(adata.n_obs)
        ct_counts = adata.obs[META_KEY].value_counts()
        rec = (ct_counts / ct_counts.sum()).to_dict()
        rec['sample_id'] = sid
        rec['n_cells_obs'] = n
        ct_frac_records.append(rec)
        del adata; gc.collect()
    except Exception as e:
        print(f'  WARNING {sid}: {e}')

ct_frac_df = pd.DataFrame(ct_frac_records).fillna(0)

# Build donor_base
donor_base = raw[['sample_id','age','age_group','cohort','cohort_group','n_cells']].copy()
donor_base = donor_base.merge(
    ct_frac_df[['sample_id','n_cells_obs']], on='sample_id', how='left')
donor_base['n_cells'] = donor_base['n_cells_obs'].fillna(donor_base['n_cells'])
donor_base.drop(columns=['n_cells_obs'], errors='ignore', inplace=True)
donor_base['log_total_cells'] = np.log1p(donor_base['n_cells'].fillna(0))

samples = raw.copy()
donor_base.to_csv(TAB_DIR / 'S1_donor_metadata.tsv', sep='\t', index=False)
ct_frac_df.to_csv(TAB_DIR / 'S2_celltype_abundance_per_donor.tsv', sep='\t', index=False)
print(f'donor_base: {len(donor_base)} rows')
display(donor_base.head(3))

## Stage 1 — FastCCC Per-Donor

对每个donor独立运行FastCCC，**使用默认16-CS Cauchy组合检验**（不传single_unit_summary / complex_aggregation / LR_combination）。
结果缓存到 `per_sample_results/{sample_id}/`。

In [ ]:
# ============================================================
# Stage 1: FastCCC per-donor with caching
# ============================================================
FORCE_NORMALIZE_LOG1P = True
MIN_CELLS_PER_SAMPLE  = 200

def sample_result_paths(sample_id):
    save_dir = RESULTS / str(sample_id)
    return {'save_dir': save_dir,
            'strength': save_dir / 'interactions_strength.tsv',
            'pvals':    save_dir / 'pvals.tsv',
            'percent':  save_dir / 'percents_analysis.tsv'}

def has_fastccc_cache(sid):
    rp = sample_result_paths(sid)
    return rp['strength'].exists() and rp['pvals'].exists() and rp['percent'].exists()

def run_fastccc_one(row):
    sid = str(row['sample_id'])
    rp  = sample_result_paths(sid)
    rp['save_dir'].mkdir(parents=True, exist_ok=True)
    adata = sc.read_h5ad(str(row['h5ad_path']))
    adata.obs_names_make_unique(); adata.var_names_make_unique()
    if META_KEY not in adata.obs.columns:
        raise KeyError(f'{sid}: missing obs column {META_KEY!r}')
    adata.obs[META_KEY] = adata.obs[META_KEY].astype(str).str.strip()
    adata = adata[adata.obs[META_KEY].notna() & (adata.obs[META_KEY] != '')].copy()
    if adata.n_obs < MIN_CELLS_PER_SAMPLE:
        raise ValueError(f'{sid}: only {adata.n_obs} cells')
    if FORCE_NORMALIZE_LOG1P:
        sc.pp.normalize_total(adata, target_sum=1e4); sc.pp.log1p(adata)
    input_path = TMP / f'{sid}.fastccc_input.h5ad'
    adata.write_h5ad(str(input_path), compression='gzip')
    rec = {'n_cells': int(adata.n_obs), 'n_celltypes': int(adata.obs[META_KEY].nunique())}
    del adata; gc.collect()
    # Default 16-CS Cauchy: no single_unit_summary / complex_aggregation / LR_combination
    strength, pvals, percents = fastccc.statistical_analysis_method(
        str(DB_PATH), None, str(input_path), CONVERT_TYPE,
        meta_key=META_KEY, save_path=str(rp['save_dir']))
    strength.to_csv(rp['strength'], sep='\t')
    pvals.to_csv(rp['pvals'],    sep='\t')
    percents.to_csv(rp['percent'], sep='\t')
    return rec

run_records = []
for i, (_, row) in enumerate(samples.iterrows(), 1):
    sid = str(row['sample_id'])
    cached = has_fastccc_cache(sid)
    print(f'[{i}/{len(samples)}] {sid}  age={row["age"]:.0f}  cached={cached}', end='')
    if cached and not RERUN_EXISTING:
        run_records.append({'sample_id': sid, 'status': 'cached'}); print()
        continue
    if not RUN_FASTCCC_IF_MISSING:
        run_records.append({'sample_id': sid, 'status': 'missing_not_run'}); print(' [SKIP]')
        continue
    try:
        rec = run_fastccc_one(row)
        run_records.append({'sample_id': sid, 'status': 'computed', **rec}); print(' [DONE]')
    except Exception as e:
        run_records.append({'sample_id': sid, 'status': f'error: {repr(e)}'})
        print(f' [ERROR: {e}]')

run_summary = pd.DataFrame(run_records)
run_summary.to_csv(OUT / 'fastccc_run_summary.tsv', sep='\t', index=False)
print(f"\nCached:{(run_summary.status=='cached').sum()}  "
      f"Computed:{(run_summary.status=='computed').sum()}  "
      f"Errors:{run_summary.status.str.startswith('error').sum()}")
display(run_summary)

## Stage 2 — 显著互作提取 + CPDB 注释

加载CPDBv5.0.0注释，从每个donor的FastCCC输出中提取p<0.05的互作，
构建 `fastccc_long` 长表，关联 classification、directionality、is_ppi。

In [ ]:
# ============================================================
# Stage 2: CPDB annotations + fastccc_long
# ============================================================
print('Loading CPDBv5.0.0 annotations...')
cpdb_tables = load_cpdb_annotations(str(DB_PATH))
annot_map   = build_lr_annotation_map(cpdb_tables)
receptor_tf = load_receptor_tf(cpdb_tables)
print(f'  Interactions: {len(annot_map)}')
print(f'  Receptor-TF pairs: {len(receptor_tf)}')

result_dir_dict = {}
for _, row in run_summary.iterrows():
    sid = str(row['sample_id'])
    if has_fastccc_cache(sid):
        rp = sample_result_paths(sid)
        result_dir_dict[sid] = {k: str(v) for k, v in rp.items() if k != 'save_dir'}

print(f'Building fastccc_long from {len(result_dir_dict)} donors...')
fastccc_long = build_fastccc_long(result_dir_dict, annot_map,
                                   pval_cutoff=PVAL_SIG_CUTOFF)
print(f'fastccc_long: {len(fastccc_long)} rows, '
      f'{fastccc_long["cellpair"].nunique() if not fastccc_long.empty else 0} cellpairs, '
      f'{fastccc_long["lr_pair"].nunique() if not fastccc_long.empty else 0} LR pairs')

if not fastccc_long.empty:
    fastccc_long.to_csv(TAB_DIR / 'fastccc_long_sig.tsv', sep='\t', index=False)
    display(fastccc_long.groupby('classification')['lr_pair'].count().sort_values(ascending=False).head(10))
else:
    print('WARNING: fastccc_long is empty!')

## Stage 3 — Donor-Level 聚合 + 支持度过滤

三种metric：sum_strength（通讯总量）、mean_strength（单互作强度）、sig_lr_count（多样性）。
支持度过滤：cellpair在 ≥8 个donor中出现才保留（~11%阈值）。

In [ ]:
# ============================================================
# Stage 3: Donor-level aggregation
# ============================================================

def aggregate_metrics(fl, pair_col, donor_meta):
    if fl.empty: return pd.DataFrame()
    agg = (fl.groupby(['sample_id', pair_col])
             .agg(sum_strength=('strength','sum'),
                  mean_strength=('strength','mean'),
                  sig_lr_count=('lr_pair','nunique'))
             .reset_index())
    n_present = agg.groupby(pair_col)['sample_id'].nunique()
    valid = n_present[n_present >= MIN_DONORS_PRESENT].index
    agg = agg[agg[pair_col].isin(valid)].copy()
    # Zero-fill all donor x valid pairs
    all_sids = donor_meta['sample_id'].unique()
    full_idx = pd.MultiIndex.from_product([all_sids, valid],
                                           names=['sample_id', pair_col])
    agg = (agg.set_index(['sample_id', pair_col])
              .reindex(full_idx, fill_value=0)
              .reset_index())
    agg = agg.merge(donor_meta[['sample_id','age','age_group','cohort_group','log_total_cells']],
                    on='sample_id', how='left')
    return agg

def add_fractions(df, pair_col, ct_frac):
    if df.empty: return df
    # Parse sender/receiver
    parts = df[pair_col].str.split('|', n=1, expand=True)
    df = df.copy()
    df['sender']   = parts[0].str.strip()
    df['receiver'] = parts[1].str.strip() if parts.shape[1] > 1 else parts[0]
    meta_cols = ['sample_id']
    frac_cols = [c for c in ct_frac.columns if c not in ['sample_id','n_cells_obs','n_cells']]
    fl = ct_frac[meta_cols + frac_cols].melt(id_vars='sample_id',
                                              var_name='celltype', value_name='frac')
    fmap = fl.set_index(['sample_id','celltype'])['frac'].to_dict()
    df['sender_fraction']   = df.apply(lambda r: fmap.get((r['sample_id'],r['sender']), 0.0), axis=1)
    df['receiver_fraction'] = df.apply(lambda r: fmap.get((r['sample_id'],r['receiver']), 0.0), axis=1)
    return df

if not fastccc_long.empty:
    fine_pair_full  = aggregate_metrics(fastccc_long, 'cellpair',   donor_base)
    fine_pair_full  = add_fractions(fine_pair_full, 'cellpair', ct_frac_df)
    broad_pair_full = aggregate_metrics(fastccc_long, 'broad_pair', donor_base)
    broad_pair_full = add_fractions(broad_pair_full, 'broad_pair', ct_frac_df)
    global_comm = (fastccc_long.groupby('sample_id')
                   .agg(total_sum_strength=('strength','sum'),
                        n_sig_lr=('lr_pair','nunique'),
                        n_sig_cellpairs=('cellpair','nunique'))
                   .reset_index()
                   .merge(donor_base, on='sample_id', how='left'))
    fine_pair_full.to_csv(TAB_DIR / 'fine_pair_full.tsv', sep='\t', index=False)
    broad_pair_full.to_csv(TAB_DIR / 'broad_pair_full.tsv', sep='\t', index=False)
    global_comm.to_csv(TAB_DIR / 'S3_global_comm_per_donor.tsv', sep='\t', index=False)
    print(f'fine pairs: {fine_pair_full["cellpair"].nunique()}')
    print(f'broad pairs: {broad_pair_full["broad_pair"].nunique()}')
    display(fine_pair_full.head(3))
else:
    fine_pair_full = broad_pair_full = global_comm = pd.DataFrame()
    print('WARNING: fastccc_long empty - Stage 3 skipped')

## Stage 4 — Broad Cell-Pair 分析

OLS模型：`metric ~ age + age² + sender_fraction + receiver_fraction + log_total_cells + C(cohort_group)`

注：无sex协变量（数据中不可得），并在文中标注此局限。

In [ ]:
import zlib

# ============================================================
# Stage 4: Broad cell-pair Spearman discovery + sensitivity analysis
# ============================================================

AGE_ORDER = ['Young adult', 'Adult', 'Elderly']

# Literature-aligned grouped-age palette and compatibility aliases.
AGE_PAL = globals().get('AGE_PAL', {})
AGE_PAL.update({
    'Young adult': '#3B74AF',
    'Adult': '#D39C32',
    'Elderly': '#C84E4E',
})
YOUNG_COLOR = AGE_PAL['Young adult']
OLD_COLOR = AGE_PAL['Elderly']
SPEARMAN_P_DISCOVERY = 0.05
SPEARMAN_RHO_DISCOVERY = 0.30
CLASS_SPEARMAN_P_MAIN = 0.01
LR_SPEARMAN_P_STRICT = 0.001
LR_SPEARMAN_P_RELAXED = 0.01
COHORT_BLOCK_PERMUTATIONS = 0
COHORT_SENSITIVITY_P = 0.10
LOO_STABILITY_MIN = 0.75


def first_existing_col(df, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback


def ordered_group_pattern(values, groups, order=AGE_ORDER, tol_frac=0.08):
    """Summarize ordered age-bin means without replacing continuous-age inference."""
    tmp = pd.DataFrame({'value': values, 'age_group': groups}).dropna()
    means = tmp.groupby('age_group')['value'].mean().reindex(order)
    if means.isna().any():
        return pd.Series({
            'ordered_curve_pattern': 'insufficient_groups',
            'young_mean_ordered': means.get('Young adult', np.nan),
            'middle_mean_ordered': means.get('Adult', np.nan),
            'old_mean_ordered': means.get('Elderly', np.nan),
            'early_delta': np.nan,
            'late_delta': np.nan,
        })
    y, m, o = means.loc['Young adult'], means.loc['Adult'], means.loc['Elderly']
    rng = max(float(means.max() - means.min()), 1e-12)
    tol = tol_frac * rng
    early_delta = m - y
    late_delta = o - m

    if early_delta > tol and late_delta > tol:
        pat = 'monotonic_up'
    elif early_delta < -tol and late_delta < -tol:
        pat = 'monotonic_down'
    elif early_delta > tol and late_delta < -tol:
        pat = 'middle_high'
    elif early_delta < -tol and late_delta > tol:
        pat = 'middle_low'
    elif abs(early_delta) <= tol and abs(late_delta) > tol:
        pat = 'late_onset'
    elif abs(early_delta) > tol and abs(late_delta) <= tol:
        pat = 'early_shift'
    else:
        pat = 'flat_or_uncertain'
    return pd.Series({
        'ordered_curve_pattern': pat,
        'young_mean_ordered': y,
        'middle_mean_ordered': m,
        'old_mean_ordered': o,
        'early_delta': early_delta,
        'late_delta': late_delta,
    })


def jonckheere_terpstra_trend(values, groups, order=AGE_ORDER):
    """Approximate Jonckheere-Terpstra ordered trend test for Young adult < Adult < Elderly."""
    tmp = pd.DataFrame({'value': values, 'age_group': groups}).dropna()
    vals_by_group = [tmp.loc[tmp['age_group'] == g, 'value'].to_numpy(dtype=float) for g in order]
    ns = np.array([len(v) for v in vals_by_group], dtype=float)
    if (ns < 2).any() or ns.sum() < 6:
        return pd.Series({'jt_stat': np.nan, 'jt_z': np.nan, 'jt_p': np.nan, 'jt_direction': 'insufficient'})

    j_inc = 0.0
    total_pairs = 0.0
    for i in range(len(vals_by_group) - 1):
        xi = vals_by_group[i]
        for j in range(i + 1, len(vals_by_group)):
            xj = vals_by_group[j]
            diffs = xj[:, None] - xi[None, :]
            j_inc += np.sum(diffs > 0) + 0.5 * np.sum(diffs == 0)
            total_pairs += diffs.size

    n = ns.sum()
    mean_j = total_pairs / 2.0
    var_j = (n**2 * (2*n + 3) - np.sum(ns**2 * (2*ns + 3))) / 72.0
    if var_j <= 0:
        return pd.Series({'jt_stat': j_inc, 'jt_z': np.nan, 'jt_p': np.nan, 'jt_direction': 'flat'})

    z = (j_inc - mean_j) / np.sqrt(var_j)
    p_two = 2 * st.norm.sf(abs(z))
    direction = 'increasing' if z > 0 else ('decreasing' if z < 0 else 'flat')
    return pd.Series({'jt_stat': j_inc, 'jt_z': z, 'jt_p': p_two, 'jt_direction': direction})


def spearman_age_trend(values, ages):
    tmp = pd.DataFrame({'value': values, 'age': ages}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 5 or tmp['value'].nunique() < 2 or tmp['age'].nunique() < 2:
        return pd.Series({'spearman_rho': np.nan, 'spearman_p': np.nan})
    rho, pval = st.spearmanr(tmp['age'].values, tmp['value'].values)
    return pd.Series({'spearman_rho': rho, 'spearman_p': pval})


def build_shared_model_group(grp, metric):
    """Build one donor subset shared by raw Spearman, OLS, and sensitivity checks."""
    required = ['sample_id','age','age_group','cohort_group','log_total_cells',
                'sender_fraction','receiver_fraction', metric]
    missing = [c for c in required if c not in grp.columns]
    if missing:
        return pd.DataFrame(columns=['sample_id','age','age_group','cohort_group','log_total_cells',
                                     'sender_fraction','receiver_fraction','value'])
    model_grp = grp[required].copy().rename(columns={metric: 'value'})
    model_grp = model_grp.replace([np.inf, -np.inf], np.nan)
    return model_grp.dropna(subset=['sample_id','age','age_group','cohort_group','log_total_cells',
                                    'sender_fraction','receiver_fraction','value']).copy()


def effect_support_summary(values, beta_age=np.nan, beta_age_se=np.nan):
    """Add display-friendly effect sizes and guards against sparse/degenerate entities."""
    vals = pd.Series(values, dtype='float64').replace([np.inf, -np.inf], np.nan).dropna()
    n_total = int(vals.shape[0])
    if n_total == 0:
        return pd.Series({
            'beta_10y': np.nan, 'beta_10y_se': np.nan,
            'std_beta_age': np.nan, 'std_beta_10y': np.nan,
            'n_donors_modeled': 0, 'n_nonzero_donors': 0,
            'donor_frac': np.nan, 'value_mean': np.nan,
            'value_sd': np.nan, 'value_nunique': 0,
        })
    value_sd = float(vals.std(ddof=0))
    n_nonzero = int((vals.abs() > 0).sum())
    beta_10y = beta_age * 10 if pd.notna(beta_age) else np.nan
    beta_10y_se = beta_age_se * 10 if pd.notna(beta_age_se) else np.nan
    std_beta_age = beta_age / value_sd if (pd.notna(beta_age) and value_sd > 0) else np.nan
    return pd.Series({
        'beta_10y': beta_10y,
        'beta_10y_se': beta_10y_se,
        'std_beta_age': std_beta_age,
        'std_beta_10y': std_beta_age * 10 if pd.notna(std_beta_age) else np.nan,
        'n_donors_modeled': n_total,
        'n_nonzero_donors': n_nonzero,
        'donor_frac': n_nonzero / n_total if n_total else np.nan,
        'value_mean': float(vals.mean()),
        'value_sd': value_sd,
        'value_nunique': int(vals.nunique(dropna=True)),
    })


def raw_spearman_discovery_mask(row):
    n_modeled = row.get('n_donors_modeled', np.nan)
    n_nonzero = row.get('n_nonzero_donors', 0)
    donor_frac = row.get('donor_frac', np.nan)
    min_nonzero = np.ceil(0.35 * n_modeled) if pd.notna(n_modeled) else np.inf
    return (
        pd.notna(row.get('spearman_p', np.nan)) and
        pd.notna(row.get('spearman_rho', np.nan)) and
        row.get('spearman_p', np.inf) < SPEARMAN_P_DISCOVERY and
        abs(row.get('spearman_rho', 0.0)) > SPEARMAN_RHO_DISCOVERY and
        pd.notna(donor_frac) and donor_frac >= 0.35 and
        n_nonzero >= min_nonzero and
        row.get('value_sd', 0) > 0
    )


def leave_one_cohort_out_spearman(model_grp, full_rho=np.nan):
    """Check whether Spearman direction survives removing each cohort once."""
    base = model_grp.replace([np.inf, -np.inf], np.nan).dropna(subset=['value','age','cohort_group']).copy()
    base['cohort_group'] = base['cohort_group'].astype(str)

    if base.empty or base['cohort_group'].nunique() < 2 or pd.isna(full_rho) or full_rho == 0:
        return pd.Series({'loo_direction_stability': np.nan, 'rho_min_loo': np.nan,
                          'rho_max_loo': np.nan, 'rho_median_loo': np.nan, 'loo_n_cohorts': 0})

    rhos = []
    for cohort in sorted(base['cohort_group'].dropna().astype(str).unique()):
        sub = base[base['cohort_group'] != cohort]
        if len(sub) < 8 or sub['value'].nunique() < 2 or sub['age'].nunique() < 2:
            continue
        rho, _ = st.spearmanr(sub['age'].values, sub['value'].values)
        if pd.notna(rho):
            rhos.append(float(rho))

    if not rhos:
        return pd.Series({'loo_direction_stability': np.nan, 'rho_min_loo': np.nan,
                          'rho_max_loo': np.nan, 'rho_median_loo': np.nan, 'loo_n_cohorts': 0})

    full_sign = np.sign(full_rho)
    signs = [np.sign(r) for r in rhos if r != 0]
    stability = np.mean([s == full_sign for s in signs]) if signs else np.nan

    return pd.Series({'loo_direction_stability': float(stability) if pd.notna(stability) else np.nan,
                      'rho_min_loo': float(np.min(rhos)),
                      'rho_max_loo': float(np.max(rhos)),
                      'rho_median_loo': float(np.median(rhos)),
                      'loo_n_cohorts': int(len(rhos))})


def cohort_block_permutation_spearman(model_grp, observed_rho=np.nan, n_perm=COHORT_BLOCK_PERMUTATIONS, seed=0):
    """Empirical Spearman p after permuting age within cohort blocks."""
    base = model_grp.replace([np.inf, -np.inf], np.nan).dropna(subset=['value','age','cohort_group']).copy()
    base['cohort_group'] = base['cohort_group'].astype(str)

    if (base.empty or len(base) < 8 or base['value'].nunique() < 2 or base['age'].nunique() < 2 or
            pd.isna(observed_rho) or base['cohort_group'].nunique() < 2):
        return pd.Series({'block_perm_p': np.nan, 'block_perm_n': 0})

    rng = np.random.default_rng(seed)
    obs = abs(float(observed_rho))
    ge = 0
    valid = 0
    values = base['value'].to_numpy(dtype=float)
    cohorts = base['cohort_group'].astype(str).to_numpy()
    age = base['age'].to_numpy(dtype=float)

    for _ in range(int(n_perm)):
        perm_age = age.copy()
        for cohort in sorted(pd.unique(cohorts)):
            idx = np.where(cohorts == cohort)[0]
            if len(idx) > 1:
                perm_age[idx] = rng.permutation(perm_age[idx])
        if pd.Series(perm_age).nunique() < 2:
            continue
        rho, _ = st.spearmanr(perm_age, values)
        if pd.notna(rho):
            valid += 1
            ge += abs(float(rho)) >= obs

    if valid == 0:
        return pd.Series({'block_perm_p': np.nan, 'block_perm_n': 0})

    return pd.Series({'block_perm_p': (ge + 1) / (valid + 1), 'block_perm_n': int(valid)})


def residual_spearman(model_grp):
    """Residualize CCC value against cohort/composition covariates, then correlate residuals with age."""
    cols = ['value','age','cohort_group','log_total_cells','sender_fraction','receiver_fraction']
    base = model_grp[cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    base['cohort_group'] = base['cohort_group'].astype(str)

    if len(base) < 8 or base['value'].nunique() < 2 or base['age'].nunique() < 2:
        return pd.Series({'residual_spearman_rho': np.nan, 'residual_spearman_p': np.nan})

    pieces = [pd.Series(1.0, index=base.index, name='intercept')]
    if base['cohort_group'].nunique() > 1:
        pieces.append(pd.get_dummies(base['cohort_group'], prefix='cohort', drop_first=True, dtype=float))
    for c in ['log_total_cells','sender_fraction','receiver_fraction']:
        if c in base.columns and base[c].nunique(dropna=True) > 1:
            pieces.append(base[[c]].astype(float))

    X = pd.concat(pieces, axis=1).astype(float).to_numpy()
    y = base['value'].astype(float).to_numpy()

    try:
        coef, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ coef
        if pd.Series(resid).nunique() < 2:
            return pd.Series({'residual_spearman_rho': np.nan, 'residual_spearman_p': np.nan})
        rho, pval = st.spearmanr(base['age'].values, resid)
        return pd.Series({'residual_spearman_rho': rho, 'residual_spearman_p': pval})
    except Exception:
        return pd.Series({'residual_spearman_rho': np.nan, 'residual_spearman_p': np.nan})


def add_discovery_trend_labels(df):
    if df.empty:
        return df
    df = df.copy()
    min_nonzero = np.ceil(0.35 * df['n_donors_modeled'].astype(float)) if 'n_donors_modeled' in df.columns else np.inf
    support_mask = (
        (df['spearman_p'] < SPEARMAN_P_DISCOVERY) &
        (df['spearman_rho'].abs() > SPEARMAN_RHO_DISCOVERY) &
        (df['donor_frac'] >= 0.35) &
        (df['n_nonzero_donors'] >= min_nonzero) &
        (df['value_sd'] > 0)
    )
    df['discovery_trend_label'] = np.where(
        support_mask,
        'nominal Spearman trend',
        'not selected'
    )
    df['discovery_selected'] = support_mask
    df['abs_spearman_rho'] = df['spearman_rho'].abs() if 'spearman_rho' in df.columns else np.nan
    return df


def add_cohort_sensitivity_labels(df):
    if df.empty:
        return df
    df = add_discovery_trend_labels(df)
    loo_ok = df['loo_direction_stability'].ge(LOO_STABILITY_MIN) if 'loo_direction_stability' in df.columns else pd.Series(False, index=df.index)
    if 'block_perm_p' in df.columns and df['block_perm_p'].notna().any():
        block_ok = df['block_perm_p'].lt(COHORT_SENSITIVITY_P)
        block_bad = df['block_perm_p'].notna() & ~block_ok
    else:
        block_ok = pd.Series(True, index=df.index)
        block_bad = pd.Series(False, index=df.index)
    loo_bad = df['loo_direction_stability'].notna() & ~loo_ok if 'loo_direction_stability' in df.columns else pd.Series(False, index=df.index)
    selected = df['discovery_selected']
    robust = selected & loo_ok & block_ok
    sensitive = selected & ~robust & (loo_bad | block_bad)
    exploratory = selected & ~robust & ~sensitive
    df['cohort_sensitivity_label'] = np.select(
        [robust, sensitive, exploratory],
        ['robust trend', 'cohort-sensitive trend', 'exploratory trend'],
        default='not selected'
    )
    return df



CLASS_MAIN_MIN_MODELED = 50
CLASS_MAIN_MIN_NONZERO = 15
CLASS_MAIN_MIN_FRAC = 0.35
CLASS_SPARSE_MIN_NONZERO = 10
CLASS_SPARSE_MIN_FRAC = 0.15

LR_MAIN_MIN_MODELED = 50
LR_MAIN_MIN_NONZERO = 21
LR_MAIN_MIN_FRAC = 0.35
LR_SPARSE_MIN_NONZERO = 15
LR_SPARSE_MIN_FRAC = 0.20


def evidence_thresholds(layer='general'):
    """Layer-specific prevalence thresholds for main-text interpretation."""
    layer = str(layer).lower()
    if layer in ['classification', 'class', 'cpdb_classification']:
        return {
            'main_min_modeled': CLASS_MAIN_MIN_MODELED,
            'main_min_nonzero': CLASS_MAIN_MIN_NONZERO,
            'main_min_frac': CLASS_MAIN_MIN_FRAC,
            'sparse_min_nonzero': CLASS_SPARSE_MIN_NONZERO,
            'sparse_min_frac': CLASS_SPARSE_MIN_FRAC,
            'main_p': CLASS_SPEARMAN_P_MAIN,
        }
    if layer in ['lr', 'lr_driver', 'lri', 'ligand_receptor']:
        return {
            'main_min_modeled': LR_MAIN_MIN_MODELED,
            'main_min_nonzero': LR_MAIN_MIN_NONZERO,
            'main_min_frac': LR_MAIN_MIN_FRAC,
            'sparse_min_nonzero': LR_SPARSE_MIN_NONZERO,
            'sparse_min_frac': LR_SPARSE_MIN_FRAC,
            'main_p': LR_SPEARMAN_P_RELAXED,
        }
    return {
        'main_min_modeled': 50,
        'main_min_nonzero': 8,
        'main_min_frac': 0.35,
        'sparse_min_nonzero': 3,
        'sparse_min_frac': 0.05,
        'main_p': SPEARMAN_P_DISCOVERY,
    }


def sign_nozero(x):
    if pd.isna(x) or x == 0:
        return np.nan
    return float(np.sign(x))


def covariate_sensitivity_from_row(row):
    """Summarize OLS/residual direction checks without making them main gates."""
    rho_sign = sign_nozero(row.get('spearman_rho', np.nan))
    if pd.isna(rho_sign):
        return 'covariate-not-assessable'
    assessed = []
    for col in ['beta_age', 'residual_spearman_rho']:
        val = row.get(col, np.nan)
        val_sign = sign_nozero(val)
        if pd.notna(val_sign):
            assessed.append(val_sign == rho_sign)
    if not assessed:
        return 'covariate-not-assessable'
    if all(assessed):
        return 'covariate-consistent'
    return 'covariate-sensitive'


def group_pattern_inconsistent_from_row(row):
    """Flag only clear monotonic group-pattern conflicts with continuous Spearman."""
    pattern = str(row.get('ordered_curve_pattern', '')).lower()
    rho = row.get('spearman_rho', np.nan)
    if pd.isna(rho) or rho == 0:
        return False
    if rho > 0 and pattern == 'monotonic_down':
        return True
    if rho < 0 and pattern == 'monotonic_up':
        return True
    return False


def assign_interpretation_tier(row, layer='general'):
    th = evidence_thresholds(layer)
    n_modeled = row.get('n_donors_modeled', np.nan)
    n_nonzero = row.get('n_nonzero_donors', 0)
    donor_frac = row.get('donor_frac', np.nan)
    if pd.isna(donor_frac) and pd.notna(n_modeled) and n_modeled:
        donor_frac = n_nonzero / n_modeled

    # Tier 4 has highest priority: sparse signals are not interpreted, even with tiny p values.
    if (pd.isna(n_nonzero) or pd.isna(donor_frac) or
            n_nonzero < th['sparse_min_nonzero'] or donor_frac < th['sparse_min_frac']):
        return 'Tier 4: sparse signal; not interpreted'

    if not bool(row.get('discovery_selected', False)) and row.get('discovery_trend_label', '') != 'nominal Spearman trend':
        return 'not selected'

    if (pd.isna(n_modeled) or n_modeled < th['main_min_modeled'] or
            n_nonzero < th['main_min_nonzero'] or donor_frac < th['main_min_frac']):
        return 'Tier 3: supplementary exploratory trend'

    if row.get('spearman_p', np.inf) >= th.get('main_p', SPEARMAN_P_DISCOVERY):
        return 'Tier 3: supplementary exploratory trend'

    if row.get('cohort_sensitivity_label', '') != 'robust trend':
        return 'Tier 3: supplementary exploratory trend'

    covar = row.get('covariate_sensitivity_label', 'covariate-not-assessable')
    if covar in ['covariate-sensitive', 'covariate-not-assessable']:
        return 'Tier 2: main-caution; covariate-sensitive nominal trend'

    return 'Tier 1: main; robust nominal trend'


def recommended_location_from_tier(row):
    tier = str(row.get('interpretation_tier', ''))
    if tier.startswith('Tier 1'):
        return 'main text only'
    if tier.startswith('Tier 2'):
        return 'main text only'
    if tier.startswith('Tier 3'):
        return 'supplementary figure'
    if tier.startswith('Tier 4'):
        return 'not interpreted'
    return 'supplementary table'


def add_interpretation_annotations(df, layer='general'):
    """Add manuscript-facing interpretation tiers without altering primary statistics."""
    if df is None or df.empty:
        return df
    out = df.copy()
    if 'n_donors_modeled' in out.columns and 'n_nonzero_donors' in out.columns:
        denom = out['n_donors_modeled'].replace(0, np.nan).astype(float)
        out['donor_frac'] = out['n_nonzero_donors'].astype(float) / denom
    out['covariate_sensitivity_label'] = out.apply(covariate_sensitivity_from_row, axis=1)
    out['group_pattern_inconsistent'] = out.apply(group_pattern_inconsistent_from_row, axis=1)

    th = evidence_thresholds(layer)
    out['sparse_signal_flag'] = (
        out['n_nonzero_donors'].fillna(0).astype(float).lt(th['sparse_min_nonzero']) |
        out['donor_frac'].fillna(0).astype(float).lt(th['sparse_min_frac'])
    )
    out['interpretation_tier'] = out.apply(lambda row: assign_interpretation_tier(row, layer=layer), axis=1)
    out['main_text_candidate'] = out['interpretation_tier'].isin([
        'Tier 1: main; robust nominal trend',
        'Tier 2: main-caution; covariate-sensitive nominal trend',
    ])
    out['recommended_location'] = out.apply(recommended_location_from_tier, axis=1)
    return out



def _entity_sender_receiver_for_balance(entity):
    text = str(entity)
    if '::' in text:
        text = text.split('::', 1)[0]
    if '|' in text:
        left, right = text.split('|', 1)
    elif '->' in text:
        left, right = text.split('->', 1)
    else:
        left, right = text, text
    return left.strip(), right.strip()


def _take_balanced_rows(rows_df, entity_col, n, max_per_sender=None, max_per_receiver=None,
                        sender_counts=None, receiver_counts=None):
    sender_counts = {} if sender_counts is None else dict(sender_counts)
    receiver_counts = {} if receiver_counts is None else dict(receiver_counts)
    selected = []
    if rows_df is None or rows_df.empty or n <= 0:
        return selected, sender_counts, receiver_counts
    for _, row in rows_df.drop_duplicates(entity_col).iterrows():
        ent = row[entity_col]
        sender, receiver = _entity_sender_receiver_for_balance(ent)
        if max_per_sender is not None and sender_counts.get(sender, 0) >= max_per_sender:
            continue
        if max_per_receiver is not None and receiver_counts.get(receiver, 0) >= max_per_receiver:
            continue
        selected.append(row)
        sender_counts[sender] = sender_counts.get(sender, 0) + 1
        receiver_counts[receiver] = receiver_counts.get(receiver, 0) + 1
        if len(selected) >= n:
            break
    return selected, sender_counts, receiver_counts


def select_balanced_discovery_entities(model_df, entity_col, top_n=18, min_up=None, min_down=None,
                                       max_per_sender=None, max_per_receiver=None,
                                       label='balanced discovery entities'):
    """Select continuous-age discovery entities while preserving aging-up/down when available."""
    if model_df is None or model_df.empty or entity_col not in model_df.columns:
        print(f'{label}: no model table available')
        return [], pd.DataFrame()
    df = model_df.copy()
    if 'discovery_selected' not in df.columns:
        df = add_cohort_sensitivity_labels(df)
    pool = df[df['discovery_selected'].fillna(False)].copy()
    if pool.empty:
        print(f'{label}: no entities passed strict nominal Spearman discovery')
        return [], pool
    pool['abs_spearman_rho'] = pool['spearman_rho'].abs()
    pool['aging_direction'] = np.where(pool['spearman_rho'] > 0, 'aging-up',
                                np.where(pool['spearman_rho'] < 0, 'aging-down', 'flat'))
    sort_cols = ['spearman_p', 'abs_spearman_rho']
    asc = [True, False]
    if 'donor_frac' in pool.columns:
        sort_cols.append('donor_frac'); asc.append(False)
    pool = pool.sort_values(sort_cols, ascending=asc)
    target_up = min_up if min_up is not None else top_n // 2
    target_down = min_down if min_down is not None else top_n - target_up
    if target_up + target_down > top_n:
        target_down = max(0, top_n - target_up)
    sender_counts, receiver_counts = {}, {}
    up_rows, sender_counts, receiver_counts = _take_balanced_rows(
        pool[pool['spearman_rho'] > 0], entity_col, target_up,
        max_per_sender, max_per_receiver, sender_counts, receiver_counts)
    down_rows, sender_counts, receiver_counts = _take_balanced_rows(
        pool[pool['spearman_rho'] < 0], entity_col, target_down,
        max_per_sender, max_per_receiver, sender_counts, receiver_counts)
    rows = []
    for i in range(max(len(up_rows), len(down_rows))):
        if i < len(up_rows): rows.append(up_rows[i])
        if i < len(down_rows): rows.append(down_rows[i])
    selected_entities = [r[entity_col] for r in rows]
    if len(selected_entities) < top_n:
        remaining = pool[~pool[entity_col].isin(selected_entities)].copy()
        fill_rows, sender_counts, receiver_counts = _take_balanced_rows(
            remaining, entity_col, top_n - len(selected_entities),
            max_per_sender, max_per_receiver, sender_counts, receiver_counts)
        selected_entities.extend([r[entity_col] for r in fill_rows])
    if len(selected_entities) < top_n:
        remaining = pool[~pool[entity_col].isin(selected_entities)].drop_duplicates(entity_col)
        selected_entities.extend(remaining[entity_col].head(top_n - len(selected_entities)).tolist())
    selected_entities = list(dict.fromkeys(selected_entities))[:top_n]
    selected = pool[pool[entity_col].isin(selected_entities)].copy()
    n_up = int((selected['spearman_rho'] > 0).sum()) if not selected.empty else 0
    n_down = int((selected['spearman_rho'] < 0).sum()) if not selected.empty else 0
    print(f'{label}: selected {len(selected_entities)} entities ({n_up} aging-up, {n_down} aging-down)')
    if n_up == 0:
        print(f'{label}: no aging-up entities passed p < 0.05, |rho| > 0.30, donor_frac >= 0.35')
    if n_down == 0:
        print(f'{label}: no aging-down entities passed p < 0.05, |rho| > 0.30, donor_frac >= 0.35')
    return selected_entities, selected


def select_main_text_entities(model_df, entity_col, top_n=6, layer='general', allow_tier2=True):
    """Select a compact Tier 1/Tier 2 set for main figures."""
    if model_df is None or model_df.empty:
        return []
    df = add_interpretation_annotations(model_df, layer=layer)
    tier_keep = ['Tier 1: main; robust nominal trend']
    if allow_tier2:
        tier_keep.append('Tier 2: main-caution; covariate-sensitive nominal trend')
    df = df[df['interpretation_tier'].isin(tier_keep)].copy()
    if df.empty:
        return []
    df['abs_spearman_rho'] = df['spearman_rho'].abs() if 'spearman_rho' in df.columns else np.nan
    tier_order = {'Tier 1: main; robust nominal trend': 0, 'Tier 2: main-caution; covariate-sensitive nominal trend': 1}
    df['_tier_order'] = df['interpretation_tier'].map(tier_order).fillna(9)
    sort_cols = ['_tier_order', 'donor_frac']
    asc = [True, False]
    if 'spearman_p' in df.columns:
        sort_cols.append('spearman_p'); asc.append(True)
    sort_cols.append('abs_spearman_rho'); asc.append(False)
    return df.sort_values(sort_cols, ascending=asc)[entity_col].drop_duplicates().head(top_n).tolist()



def add_visual_evidence(df):
    """Compatibility wrapper: keep OLS/residual as sensitivity annotations, not main gates."""
    if df.empty:
        return df
    df = df.copy()
    p_ols = first_existing_col(df, ['p_overall', 'p_linear'], None)
    df['OLS_p_display'] = df[p_ols] if p_ols in df.columns else np.nan
    beta_sign = np.sign(df['beta_age']) if 'beta_age' in df.columns else pd.Series(np.nan, index=df.index)
    rho_sign = np.sign(df['spearman_rho']) if 'spearman_rho' in df.columns else pd.Series(np.nan, index=df.index)
    df['ols_spearman_direction_consistent'] = (
        pd.notna(beta_sign) & pd.notna(rho_sign) & (beta_sign != 0) & (rho_sign != 0) & (beta_sign == rho_sign)
    )
    df['OLS_sensitivity_label'] = np.select(
        [df['ols_spearman_direction_consistent'], pd.notna(beta_sign) & pd.notna(rho_sign) & (beta_sign != 0) & (rho_sign != 0)],
        ['OLS direction consistent', 'OLS direction differs'],
        default='OLS unavailable'
    )
    df['abs_spearman_rho'] = df['spearman_rho'].abs() if 'spearman_rho' in df.columns else np.nan
    df = add_cohort_sensitivity_labels(df)
    return add_interpretation_annotations(df, layer='general')


def save_spearman_discovery_summary(df, entity_col, prefix, metric=None, top_n=120):
    if df.empty:
        return pd.DataFrame()
    out = df.copy()
    if metric is not None and 'metric' in out.columns:
        out = out[out['metric'] == metric].copy()
    if out.empty:
        return out
    out = add_cohort_sensitivity_labels(out)
    selected = out[out['discovery_selected']].copy()
    if selected.empty:
        selected = out.copy()
        selected['discovery_trend_label'] = 'top nominal trends only; no p<0.05 hits'
        selected['cohort_sensitivity_label'] = 'top nominal trends only; no p<0.05 hits'
    order = {
        'robust trend': 0,
        'cohort-sensitive trend': 1,
        'exploratory trend': 2,
        'top nominal trends only; no p<0.05 hits': 3,
        'not selected': 4,
    }
    selected['_sensitivity_order'] = selected['cohort_sensitivity_label'].map(order).fillna(9)
    selected['abs_spearman_rho'] = selected['spearman_rho'].abs()
    keep = [c for c in [entity_col, 'metric', 'cohort_sensitivity_label', 'discovery_trend_label',
                        'spearman_rho', 'spearman_p',
                        'loo_direction_stability', 'rho_min_loo', 'rho_max_loo', 'rho_median_loo',
                        'block_perm_p', 'block_perm_n', 'residual_spearman_rho', 'residual_spearman_p',
                        'n_donors_modeled', 'n_nonzero_donors', 'donor_frac', 'value_sd',
                        'beta_age', 'beta_10y', 'OLS_p_display', 'ols_spearman_direction_consistent',
                        'OLS_sensitivity_label', 'ordered_curve_pattern', 'group_pattern_inconsistent', 'trend_support', 'sparse_signal_flag', 'covariate_sensitivity_label', 'interpretation_tier', 'main_text_candidate', 'recommended_location'] if c in selected.columns]
    selected = selected.sort_values(['_sensitivity_order','spearman_p','abs_spearman_rho','donor_frac'],
                                    ascending=[True, True, False, False]).drop_duplicates(entity_col).head(top_n)
    selected[keep].to_csv(TAB_DIR / f'{prefix}_spearman_discovery_summary.tsv', sep='\t', index=False)
    return selected[keep]


def save_cohort_sensitivity_tables(df, entity_col, prefix, metric=None, top_n=120):
    """Save cohort-aware main candidates and OLS-only sensitivity tables."""
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()
    out = df.copy()
    if metric is not None and 'metric' in out.columns:
        out = out[out['metric'] == metric].copy()
    if out.empty:
        return out, out
    out = add_cohort_sensitivity_labels(out)
    out['abs_spearman_rho'] = out['spearman_rho'].abs() if 'spearman_rho' in out.columns else np.nan
    keep_main = [c for c in [entity_col, 'metric', 'cohort_sensitivity_label', 'discovery_trend_label',
                             'spearman_rho', 'spearman_p',
                             'loo_direction_stability', 'rho_min_loo', 'rho_max_loo', 'rho_median_loo',
                             'block_perm_p', 'block_perm_n', 'residual_spearman_rho', 'residual_spearman_p',
                             'n_donors_modeled', 'n_nonzero_donors', 'donor_frac', 'value_sd',
                             'ordered_curve_pattern', 'group_pattern_inconsistent', 'trend_support', 'sparse_signal_flag', 'covariate_sensitivity_label', 'interpretation_tier', 'main_text_candidate', 'recommended_location'] if c in out.columns]
    keep_ols = [c for c in [entity_col, 'metric', 'spearman_rho', 'spearman_p',
                            'beta_age', 'beta_10y', 'std_beta_age', 'std_beta_10y', 'OLS_p_display',
                            'ols_spearman_direction_consistent', 'OLS_sensitivity_label',
                            'n_donors_modeled', 'n_nonzero_donors', 'donor_frac', 'value_sd', 'sparse_signal_flag', 'covariate_sensitivity_label', 'interpretation_tier', 'main_text_candidate', 'recommended_location'] if c in out.columns]
    order = {'robust trend': 0, 'cohort-sensitive trend': 1, 'exploratory trend': 2, 'not selected': 9}
    main = out[out['discovery_selected']].copy()
    if not main.empty:
        main['_sensitivity_order'] = main['cohort_sensitivity_label'].map(order).fillna(9)
        main = main.sort_values(['_sensitivity_order','spearman_p','abs_spearman_rho','donor_frac'],
                                ascending=[True, True, False, False]).drop_duplicates(entity_col).head(top_n)
        main[keep_main].to_csv(TAB_DIR / f'{prefix}_cohort_sensitivity_summary.tsv', sep='\t', index=False)
    ols = out.copy()
    if not ols.empty:
        ols = ols.sort_values(['spearman_p','abs_spearman_rho'], ascending=[True, False]).drop_duplicates(entity_col).head(top_n)
        ols[keep_ols].to_csv(TAB_DIR / f'{prefix}_OLS_sensitivity.tsv', sep='\t', index=False)
    return main[keep_main] if not main.empty else pd.DataFrame(columns=keep_main), ols[keep_ols] if not ols.empty else pd.DataFrame(columns=keep_ols)


def save_spearman_summary(df, entity_col, outfile, metric=None, top_n=80):
    if df.empty:
        return pd.DataFrame()
    if 'spearman_rho' not in df.columns or 'spearman_p' not in df.columns:
        return pd.DataFrame()
    out = df.copy()
    if metric is not None and 'metric' in out.columns:
        out = out[out['metric'] == metric].copy()
    if out.empty:
        return out
    out = add_cohort_sensitivity_labels(out)
    out['abs_spearman_rho'] = out['spearman_rho'].abs()
    keep = [c for c in [entity_col, 'metric', 'cohort_sensitivity_label', 'discovery_trend_label',
                        'beta_age', 'beta_10y', 'std_beta_age', 'std_beta_10y',
                        'spearman_rho', 'spearman_p', 'OLS_p_display',
                        'loo_direction_stability', 'rho_min_loo', 'rho_max_loo', 'block_perm_p',
                        'residual_spearman_rho', 'residual_spearman_p', 'trend_support',
                        'ordered_curve_pattern', 'n_donors_modeled', 'n_nonzero_donors',
                        'donor_frac', 'value_sd', 'value_nunique', 'sparse_signal_flag', 'covariate_sensitivity_label', 'interpretation_tier', 'main_text_candidate', 'recommended_location'] if c in out.columns]
    out = out.sort_values(['spearman_p', 'abs_spearman_rho'], ascending=[True, False]).drop_duplicates(entity_col).head(top_n)
    out[keep].to_csv(TAB_DIR / outfile, sep='\t', index=False)
    return out[keep]


def mark_trend_support(df):
    if df.empty:
        return df
    df = df.copy()
    min_nonzero = np.ceil(0.35 * df['n_donors_modeled'].astype(float)) if 'n_donors_modeled' in df.columns else np.inf
    raw = (
        (df['spearman_p'] < SPEARMAN_P_DISCOVERY) &
        (df['spearman_rho'].abs() > SPEARMAN_RHO_DISCOVERY) &
        (df['donor_frac'] >= 0.35) &
        (df['n_nonzero_donors'] >= min_nonzero) &
        (df['value_sd'] > 0)
    )
    df['trend_support'] = np.where(raw, 'Spearman nominal discovery', 'weak')
    return df


def empty_age_model_series():
    return pd.Series({
        'beta_age': np.nan, 'beta_age_se': np.nan, 'p_linear': np.nan,
        'beta_age2': np.nan, 'p_nonlinear': np.nan, 'p_overall': np.nan,
        'model_n': np.nan,
    })


def empty_group_stats_series():
    return pd.Series({
        'young_mean': np.nan, 'middle_mean': np.nan, 'old_mean': np.nan,
        'kw_p': np.nan, 'mwu_yo_p': np.nan, 'direction': 'not_tested'
    })


def record_dict(x):
    """Convert pandas Series or dict-like model outputs into a plain record dict."""
    if hasattr(x, 'to_dict'):
        return x.to_dict()
    if isinstance(x, dict):
        return dict(x)
    return {}


def dual_track(full_df, entity_col, metrics=None):
    """Fast donor-level modeling: Spearman/support first, sensitivity only for strict discovery hits."""
    if full_df.empty:
        return pd.DataFrame()
    if metrics is None:
        metrics = ['sum_strength','mean_strength','sig_lr_count']
    records = []
    for metric in metrics:
        if metric not in full_df.columns:
            continue
        for entity, grp in full_df.groupby(entity_col):
            model_grp = build_shared_model_group(grp.copy(), metric)
            sp_s = spearman_age_trend(model_grp['value'].values, model_grp['age'].values)
            eff_s = effect_support_summary(model_grp['value'].values, beta_age=np.nan, beta_age_se=np.nan)
            tmp_row = {**record_dict(sp_s), **record_dict(eff_s)}
            is_discovery = raw_spearman_discovery_mask(tmp_row)

            if is_discovery:
                cont = fit_age_model(
                    y=model_grp['value'].values,
                    meta_df=model_grp[['sample_id','age','cohort_group','log_total_cells',
                                       'sender_fraction','receiver_fraction']])
                grp_s = fit_group_stats(model_grp['value'].values, model_grp['age_group'].values)
                ordered_s = ordered_group_pattern(model_grp['value'].values, model_grp['age_group'].values)
                jt_s = jonckheere_terpstra_trend(model_grp['value'].values, model_grp['age_group'].values)
                eff_s = effect_support_summary(
                    model_grp['value'].values,
                    beta_age=cont.get('beta_age', np.nan),
                    beta_age_se=cont.get('beta_age_se', np.nan))
                seed = zlib.adler32(f'{entity}::{metric}'.encode('utf-8'))
                loo_s = leave_one_cohort_out_spearman(model_grp, full_rho=sp_s.get('spearman_rho', np.nan))
                block_s = (cohort_block_permutation_spearman(model_grp, observed_rho=sp_s.get('spearman_rho', np.nan), seed=seed)
                           if COHORT_BLOCK_PERMUTATIONS else pd.Series({'block_perm_p': np.nan, 'block_perm_n': 0}))
                resid_s = residual_spearman(model_grp)
                consist = direction_consistent(cont.get('beta_age', np.nan), grp_s.get('direction', np.nan))
            else:
                cont = empty_age_model_series()
                grp_s = empty_group_stats_series()
                ordered_s = ordered_group_pattern(model_grp['value'].values, model_grp['age_group'].values)
                jt_s = pd.Series({'jt_stat': np.nan, 'jt_z': np.nan, 'jt_p': np.nan})
                loo_s = pd.Series({'loo_direction_stability': np.nan, 'rho_min_loo': np.nan,
                                   'rho_max_loo': np.nan, 'rho_median_loo': np.nan, 'loo_n_cohorts': 0})
                block_s = pd.Series({'block_perm_p': np.nan, 'block_perm_n': 0})
                resid_s = pd.Series({'residual_spearman_rho': np.nan, 'residual_spearman_p': np.nan})
                consist = np.nan

            records.append({entity_col: entity, 'metric': metric,
                            **record_dict(cont), **record_dict(eff_s), **record_dict(grp_s), **record_dict(ordered_s),
                            **record_dict(jt_s), **record_dict(sp_s), **record_dict(loo_s),
                            **record_dict(block_s), **record_dict(resid_s),
                            'direction_consistent': consist})
    if not records:
        return pd.DataFrame()
    df = pd.DataFrame(records)
    df = mark_trend_support(df)
    return add_visual_evidence(df)


if not broad_pair_full.empty:
    print('Running broad cellpair models...')
    broad_models = dual_track(broad_pair_full, 'broad_pair')
    broad_models.to_csv(TAB_DIR / 'S5_broad_cellpair_models.tsv', sep='\t', index=False)
    broad_cohort_summary, broad_ols_sensitivity = save_cohort_sensitivity_tables(
        broad_models, 'broad_pair', 'S5_broad', metric='sum_strength')
    broad_discovery_summary = save_spearman_discovery_summary(
        broad_models, 'broad_pair', 'S5_broad', metric='sum_strength')
    broad_spearman_summary = save_spearman_summary(
        broad_models, 'broad_pair', 'S5_broad_spearman_trend_summary.tsv', metric='sum_strength')
    broad_sum = (broad_models.sort_values(['spearman_p','abs_spearman_rho'], ascending=[True, False])
                 .groupby('broad_pair').first().reset_index())
    print(f'broad_models: {len(broad_models)} rows')
    pcol = 'spearman_p'
    display_cols = [c for c in ['broad_pair','metric','cohort_sensitivity_label','discovery_trend_label',
                                'spearman_rho','spearman_p',
                                'loo_direction_stability','block_perm_p','residual_spearman_rho',
                                'n_donors_modeled','n_nonzero_donors','donor_frac',
                                'ordered_curve_pattern','trend_support','OLS_p_display','beta_10y'] if c in broad_sum.columns]
    display(broad_sum.nsmallest(10, pcol)[display_cols])
    if not broad_discovery_summary.empty:
        print('Top broad Spearman discovery trends:')
        display(broad_discovery_summary.head(8))
else:
    broad_models = broad_sum = broad_spearman_summary = broad_cohort_summary = broad_ols_sensitivity = broad_discovery_summary = pd.DataFrame()
    print('WARNING: broad_pair_full empty')


## Stage 5 — Fine Cell-Pair 分析

粒度到细胞亚型对。Fine pairs are selected independently at fine resolution using donor-level continuous-age Spearman discovery and cohort-aware sensitivity, rather than being forced to nest under significant broad pairs.
跨 3 种 metric 的一致性作为辅助信息；正文解释以后续 `interpretation_tier` 和 donor coverage 为准。


In [ ]:
# ============================================================
# Stage 5: Fine cell-pair analysis
# ============================================================
if not fine_pair_full.empty:
    print('Running fine cellpair models...')
    fine_models = dual_track(fine_pair_full, 'cellpair')
    fine_models.to_csv(TAB_DIR / 'S4_fine_cellpair_models.tsv', sep='\t', index=False)
    print(f'fine_models: {len(fine_models)} rows')

    if not fine_models.empty:
        pcol5 = first_existing_col(fine_models, ['spearman_p'], 'spearman_p')
        # Cross-metric consistency
        discovery_fine = fine_models[fine_models['discovery_selected']] if 'discovery_selected' in fine_models.columns else fine_models.iloc[0:0]
        sig_by_metric = (discovery_fine
                         .groupby('cellpair')['metric'].nunique()
                         .rename('n_metrics_discovery'))
        fine_sum = (fine_models.sort_values(pcol5)
                    .groupby('cellpair').first().reset_index()
                    .merge(sig_by_metric, on='cellpair', how='left'))
        fine_sum['n_metrics_discovery'] = fine_sum['n_metrics_discovery'].fillna(0).astype(int)
        metric_count_col = 'n_metrics_discovery' if 'n_metrics_discovery' in fine_sum.columns else pcol5
        fine_sum.sort_values([metric_count_col, pcol5], ascending=[False, True], inplace=True)
        fine_cohort_summary, fine_ols_sensitivity = save_cohort_sensitivity_tables(
            fine_models, 'cellpair', 'S4_fine', metric='mean_strength')
        fine_discovery_summary = save_spearman_discovery_summary(
            fine_models, 'cellpair', 'S4_fine', metric='mean_strength')
        fine_spearman_summary = save_spearman_summary(
            fine_models, 'cellpair', 'S4_fine_spearman_trend_summary.tsv', metric='mean_strength')
        selected_fine_pairs = (fine_discovery_summary['cellpair'].head(18).tolist()
                               if not fine_discovery_summary.empty else [])
        # LR driver discovery should not be gated by sum_strength alone.
        # sum_strength captures total communication burden, mean_strength captures per-LR intensity,
        # and sig_lr_count captures communication diversity. Use their union to preserve both aging-up
        # and aging-down fine-pair candidates before nesting into LR-level analysis.
        lr_entry_metrics = ['sum_strength', 'mean_strength', 'sig_lr_count']
        fine_for_lr_source = fine_models[fine_models['metric'].isin(lr_entry_metrics)].copy()
        if not fine_for_lr_source.empty:
            fine_for_lr_source['abs_spearman_rho'] = fine_for_lr_source['spearman_rho'].abs()
            fine_for_lr_source = fine_for_lr_source.sort_values(
                ['spearman_p', 'abs_spearman_rho', 'donor_frac'],
                ascending=[True, False, False]
            )
        selected_fine_pairs_for_lr, selected_fine_pairs_for_lr_table = select_balanced_discovery_entities(
            fine_for_lr_source, 'cellpair', top_n=24, min_up=12, min_down=12,
            max_per_sender=3, max_per_receiver=3,
            label='Stage 7 continuous LR fine-pair pool across sum/mean/count metrics')
        if not selected_fine_pairs_for_lr:
            selected_fine_pairs_for_lr = selected_fine_pairs.copy()
            selected_fine_pairs_for_lr_table = fine_for_lr_source[fine_for_lr_source['cellpair'].isin(selected_fine_pairs_for_lr)].copy()
        if not selected_fine_pairs_for_lr_table.empty and 'metric' in selected_fine_pairs_for_lr_table.columns:
            print('Stage 7 LR fine-pair entry metrics:')
            display(selected_fine_pairs_for_lr_table['metric'].value_counts().to_frame('n'))
        if not selected_fine_pairs_for_lr_table.empty:
            selected_fine_pairs_for_lr_table.to_csv(TAB_DIR / 'S4_selected_fine_pairs_for_LR_continuous.tsv', sep='\t', index=False)
        print(f'Selected {len(selected_fine_pairs)} display fine pairs')
        print(f'Selected {len(selected_fine_pairs_for_lr)} fine pairs for continuous LR driver analysis')
        display_cols = [c for c in ['cellpair','metric','cohort_sensitivity_label','discovery_trend_label',
                                    'beta_age','beta_10y','spearman_rho','spearman_p','loo_direction_stability','block_perm_p','residual_spearman_rho','OLS_p_display',
                                    'n_metrics_discovery','n_donors_modeled','n_nonzero_donors','donor_frac',
                                    'ordered_curve_pattern','trend_support'] if c in fine_sum.columns]
        display(fine_sum.head(8)[display_cols])
        if not fine_discovery_summary.empty:
            print('Top fine Spearman discovery trends:')
            display(fine_discovery_summary.head(8))
    else:
        selected_fine_pairs = []
        selected_fine_pairs_for_lr = []
        selected_fine_pairs_for_lr_table = pd.DataFrame()
        fine_sum = fine_spearman_summary = fine_cohort_summary = fine_ols_sensitivity = fine_discovery_summary = pd.DataFrame()
else:
    fine_models = fine_sum = fine_spearman_summary = fine_cohort_summary = fine_ols_sensitivity = fine_discovery_summary = pd.DataFrame()
    selected_fine_pairs = []
    selected_fine_pairs_for_lr = []
    selected_fine_pairs_for_lr_table = pd.DataFrame()
    print('WARNING: fine_pair_full empty')

## Stage 6 — CPDB Classification 信息流分析

按CPDBv5.0.0的 `classification` 字段聚合到donor水平，分析哪些功能家族（Chemokine, Growth factor等）的通讯在衰老中变化。

In [ ]:
# ============================================================
# Stage 6: Classification-level analysis
# ============================================================
if not fastccc_long.empty:
    cls_base = (fastccc_long.groupby(['sample_id','classification'])
                .agg(sum_strength=('strength','sum'),
                     mean_strength=('strength','mean'),
                     sig_lr_count=('lr_pair','nunique'))
                .reset_index()
                .merge(donor_base[['sample_id','age','age_group','cohort_group','log_total_cells']],
                       on='sample_id', how='left'))
    cls_base['sender_fraction'] = 0.0
    cls_base['receiver_fraction'] = 0.0
    # Zero-fill
    all_cls = cls_base['classification'].unique()
    all_sids = donor_base['sample_id'].unique()
    full_idx = pd.MultiIndex.from_product([all_sids, all_cls],
                                           names=['sample_id','classification'])
    cls_base = (cls_base.set_index(['sample_id','classification'])
                .reindex(full_idx, fill_value=0)
                .reset_index()
                .merge(donor_base[['sample_id','age','age_group','cohort_group','log_total_cells']],
                       on='sample_id', how='left', suffixes=('','_m')))
    for col in ['age','age_group','cohort_group','log_total_cells']:
        if col+'_m' in cls_base.columns:
            cls_base[col] = cls_base[col].fillna(cls_base[col+'_m'])
            cls_base.drop(columns=[col+'_m'], inplace=True)
    classification_donor_df = cls_base.copy()
    classification_donor_df.to_csv(TAB_DIR / 'S6_classification_donor.tsv', sep='\t', index=False)

    print('Running classification models...')
    class_models = dual_track(classification_donor_df, 'classification')
    class_models = add_interpretation_annotations(class_models, layer='classification')
    class_models.to_csv(TAB_DIR / 'S6_classification_models.tsv', sep='\t', index=False)
    pcol6 = first_existing_col(class_models, ['spearman_p'], 'spearman_p')
    sig_classifications = (class_models[class_models['discovery_selected']]['classification'].unique().tolist()
                           if (not class_models.empty and 'discovery_selected' in class_models.columns) else [])
    print(f'Discovery classifications: {len(sig_classifications)}')
    class_cohort_summary, class_ols_sensitivity = save_cohort_sensitivity_tables(
        class_models, 'classification', 'S6_classification', metric='sum_strength')
    class_discovery_summary = save_spearman_discovery_summary(
        class_models, 'classification', 'S6_classification', metric='sum_strength')
    class_spearman_summary = save_spearman_summary(
        class_models, 'classification', 'S6_classification_spearman_trend_summary.tsv', metric='sum_strength')
    display_cols = [c for c in ['classification','metric','cohort_sensitivity_label','discovery_trend_label',
                                'spearman_rho','spearman_p','loo_direction_stability','block_perm_p','residual_spearman_rho',
                                'n_donors_modeled','n_nonzero_donors','donor_frac',
                                'sparse_signal_flag','covariate_sensitivity_label','interpretation_tier','main_text_candidate','recommended_location',
                                'ordered_curve_pattern','group_pattern_inconsistent','trend_support'] if c in class_models.columns]
    class_display = class_models.copy()
    class_display['abs_spearman_rho'] = class_display['spearman_rho'].abs()
    tier_order = {
        'Tier 1: main; robust nominal trend': 0,
        'Tier 2: main-caution; covariate-sensitive nominal trend': 1,
        'Tier 3: supplementary exploratory trend': 2,
        'not selected': 3,
        'Tier 4: sparse signal; not interpreted': 4,
    }
    class_display['_tier_order'] = class_display['interpretation_tier'].map(tier_order).fillna(9)
    class_display = class_display.sort_values(
        ['_tier_order', 'donor_frac', 'spearman_p', 'abs_spearman_rho'],
        ascending=[True, False, True, False])
    display(class_display.head(8)[display_cols])
    if not class_discovery_summary.empty:
        print('Top classification Spearman discovery trends:')
        display(class_discovery_summary.head(8))
else:
    classification_donor_df = class_models = class_spearman_summary = class_cohort_summary = class_ols_sensitivity = class_discovery_summary = pd.DataFrame()
    sig_classifications = []
    print('WARNING: fastccc_long empty - Stage 6 skipped')

## Stage 7A/B - Continuous LR Driver Discovery + Enrichment Inputs

Use continuous-age Spearman-selected fine cell pairs as the LR search space. Grouped Young adult/Adult/Elderly Fig.4h-style plots are companion visualizations only and do not define the continuous LR driver set.


In [ ]:
# ============================================================
# Stage 7A: Continuous LR driver discovery within continuous-age selected fine pairs
# ============================================================
def first_existing_col(df, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback

def build_lr_entity_panel(lr_sub, donor_meta, ct_frac):
    if lr_sub.empty:
        return pd.DataFrame(), pd.DataFrame()
    lr_obs = (lr_sub.groupby(['sample_id','cellpair','lr_pair',
                              'classification','directionality','is_ppi'])
              .agg(sum_strength=('strength','sum'))
              .reset_index())
    lr_obs['entity'] = lr_obs['cellpair'] + '::' + lr_obs['lr_pair']
    entity_meta = lr_obs[['entity','cellpair','lr_pair','classification','directionality','is_ppi']].drop_duplicates('entity')
    full_idx = pd.MultiIndex.from_product(
        [donor_meta['sample_id'].unique(), entity_meta['entity'].values],
        names=['sample_id','entity']
    ).to_frame(index=False)
    lr_full = (full_idx.merge(entity_meta, on='entity', how='left')
               .merge(lr_obs[['sample_id','entity','sum_strength']],
                      on=['sample_id','entity'], how='left')
               .merge(donor_meta[['sample_id','age','age_group','cohort_group','log_total_cells']],
                      on='sample_id', how='left'))
    lr_full['sum_strength'] = lr_full['sum_strength'].fillna(0.0)
    lr_full = add_fractions(lr_full, 'cellpair', ct_frac)
    return lr_full, entity_meta


def compute_lr_rank_shift(lr_panel):
    if lr_panel.empty:
        return pd.DataFrame(columns=['entity','young_rank','old_rank','rank_shift','abs_rank_shift'])
    mean_by_group = (lr_panel[lr_panel['age_group'].isin(['Young adult','Elderly'])]
                     .groupby(['entity','age_group'])['sum_strength']
                     .mean().reset_index())
    piv = mean_by_group.pivot(index='entity', columns='age_group', values='sum_strength').fillna(0.0)
    if 'Young adult' not in piv.columns or 'Elderly' not in piv.columns:
        return pd.DataFrame(columns=['entity','young_rank','old_rank','rank_shift','abs_rank_shift'])
    piv['young_rank'] = piv['Young adult'].rank(ascending=False, method='average')
    piv['old_rank'] = piv['Elderly'].rank(ascending=False, method='average')
    piv['rank_shift'] = piv['young_rank'] - piv['old_rank']
    piv['abs_rank_shift'] = piv['rank_shift'].abs()
    return piv[['young_rank','old_rank','rank_shift','abs_rank_shift']].reset_index()


lr_fine_pair_pool = selected_fine_pairs_for_lr if 'selected_fine_pairs_for_lr' in globals() and selected_fine_pairs_for_lr else selected_fine_pairs
if not fastccc_long.empty and lr_fine_pair_pool:
    print(f'Continuous LR driver search uses {len(lr_fine_pair_pool)} fine cell pairs')
    lr_sub = fastccc_long[fastccc_long['cellpair'].isin(lr_fine_pair_pool)].copy()
    lr_agg, lr_entity_meta = build_lr_entity_panel(lr_sub, donor_base, ct_frac_df)
    lr_rank_shift = compute_lr_rank_shift(lr_agg)
    lr_models_raw = dual_track(lr_agg, 'entity', metrics=['sum_strength'])
    if not lr_models_raw.empty:
        lr_models_raw[['cellpair','lr_pair']] = (
            lr_models_raw['entity'].str.split('::', expand=True).iloc[:, :2])
        ent_meta = lr_entity_meta[['entity','classification','directionality','is_ppi']].drop_duplicates()
        lr_models_raw = lr_models_raw.merge(ent_meta, on='entity', how='left')
        lr_models_raw = lr_models_raw.merge(lr_rank_shift, on='entity', how='left')
        lr_models_raw = add_interpretation_annotations(lr_models_raw, layer='lr_driver')
        lr_models_raw.to_csv(TAB_DIR / 'S7_lr_level_continuous_models_all.tsv', sep='	', index=False)

        # Discovery and main-text LR pools are intentionally separated.
        lr_cohort_summary, lr_ols_sensitivity = save_cohort_sensitivity_tables(
            lr_models_raw, 'entity', 'S7_lr', metric='sum_strength')
        lr_spearman_summary = save_spearman_summary(
            lr_models_raw, 'entity', 'S7_lr_spearman_trend_summary.tsv', metric='sum_strength')
        lr_discovery_summary = save_spearman_discovery_summary(
            lr_models_raw, 'entity', 'S7_lr', metric='sum_strength')

        lr_sum = lr_models_raw[lr_models_raw['metric'] == 'sum_strength'].copy()
        lr_sum['aging_direction'] = np.where(lr_sum['spearman_rho'] > 0, 'aging-up',
                                             np.where(lr_sum['spearman_rho'] < 0, 'aging-down', 'flat'))
        selected_lr_discovery = lr_sum[lr_sum['discovery_selected'].fillna(False)].copy()
        if not selected_lr_discovery.empty:
            selected_lr_discovery['abs_spearman_rho'] = selected_lr_discovery['spearman_rho'].abs()
            selected_lr_discovery = selected_lr_discovery.sort_values(
                ['spearman_p','abs_spearman_rho','donor_frac'], ascending=[True, False, False])
        selected_lr_main = lr_sum[lr_sum['main_text_candidate'].fillna(False)].copy()
        selected_lr_continuous = selected_lr_discovery.copy()
        selected_lr = selected_lr_continuous.copy()
        selected_lr_discovery.to_csv(TAB_DIR / 'S7_lr_driver_discovery_models.tsv', sep='\t', index=False)
        selected_lr_continuous.to_csv(TAB_DIR / 'S7_lr_driver_continuous_models.tsv', sep='\t', index=False)
        selected_lr_main.to_csv(TAB_DIR / 'S7_lr_driver_main_text_models.tsv', sep='\t', index=False)
        selected_lr.to_csv(TAB_DIR / 'S7_lr_driver_models.tsv', sep='\t', index=False)

        print(f'LR-level continuous models: {len(lr_models_raw)}')
        print(f'Selected continuous LR drivers: {len(selected_lr_continuous)}')
        if not selected_lr_continuous.empty:
            print('Continuous LR driver direction counts:')
            direction_counts = selected_lr_continuous['aging_direction'].value_counts()
            display(direction_counts.to_frame('n'))
            for _direction in ['aging-up', 'aging-down']:
                if int(direction_counts.get(_direction, 0)) == 0:
                    print(f'No {_direction} LR passed strict continuous discovery filter: nominal P < 0.05, |rho| > 0.30, donor_frac >= 0.35.')
        display_cols = [c for c in ['entity','aging_direction','cohort_sensitivity_label','discovery_trend_label','beta_age','beta_10y',
                                    'spearman_rho','spearman_p','loo_direction_stability','block_perm_p','residual_spearman_rho','OLS_p_display',
                                    'jt_p','ordered_curve_pattern','rank_shift','classification','is_ppi',
                                    'n_donors_modeled','n_nonzero_donors','donor_frac','sparse_signal_flag','covariate_sensitivity_label','interpretation_tier','main_text_candidate','recommended_location'] if c in selected_lr.columns]
        display(selected_lr[display_cols].head(8))
        if not lr_discovery_summary.empty:
            print('Top LR Spearman discovery trends:')
            display(lr_discovery_summary.head(8))
    else:
        selected_lr = selected_lr_continuous = selected_lr_discovery = selected_lr_main = lr_models_raw = lr_spearman_summary = lr_cohort_summary = lr_ols_sensitivity = lr_discovery_summary = pd.DataFrame()
else:
    selected_lr = selected_lr_continuous = selected_lr_discovery = selected_lr_main = lr_models_raw = lr_spearman_summary = lr_cohort_summary = lr_ols_sensitivity = lr_discovery_summary = pd.DataFrame()
    print('WARNING: skipping Stage 7')

# ============================================================
# Stage 7B: AD risk gene anchored LR analysis
# ============================================================
AD_GOAL_GENES = [
    'APOE','TREM2','ABCA1','APP','PSEN1','PSEN2','CLU',
    'SEMA4A','NRP1',
    'CSF1','CSF1R','TGFB1','TGFB2','TGFB3','TGFBR1','TGFBR2','CX3CL1','CX3CR1'
]
AD_GOAL_SET = set(AD_GOAL_GENES)


def split_lr_pair_genes(lr_pair):
    text = str(lr_pair)
    if text in ['nan', 'None', '']:
        return set(), set()
    parts = text.split('_', 1)
    lig = parts[0] if len(parts) >= 1 else ''
    rec = parts[1] if len(parts) == 2 else ''
    lig_genes = {g.strip().upper() for g in lig.split('+') if g.strip()}
    rec_genes = {g.strip().upper() for g in rec.split('+') if g.strip()}
    return lig_genes, rec_genes


def detect_goal_gene_hits(lr_pair):
    lig_genes, rec_genes = split_lr_pair_genes(lr_pair)
    hit = sorted((lig_genes | rec_genes) & AD_GOAL_SET)
    if not hit:
        return '', ''
    sides = []
    if any(g in lig_genes for g in hit):
        sides.append('ligand')
    if any(g in rec_genes for g in hit):
        sides.append('receptor')
    return ';'.join(hit), '|'.join(sides)


def detect_ad_axis(lr_pair, classification):
    lr_u = str(lr_pair).upper()
    cls = str(classification).lower()
    tags = []
    if ('SEMA4A' in lr_u) and ('NRP1' in lr_u):
        tags.append('SEMA4A_NRP1')
    if any(g in lr_u for g in ['APOE','APP','PSEN1','PSEN2','CLU','ABCA1','TREM2']):
        tags.append('APOE_APP_TREM2_axis')
    if any(g in lr_u for g in ['CSF1','CSF1R','TGFB1','TGFB2','TGFB3','TGFBR1','TGFBR2','CX3CL1','CX3CR1']):
        tags.append('Microglia_homeostasis_axis')
    if 'microglia' in cls or 'interferon' in cls:
        tags.append('Microglia_related_classification')
    if not tags:
        tags.append('Other_AD_related')
    return ';'.join(sorted(set(tags)))


if not fastccc_long.empty:
    goal_hits = fastccc_long['lr_pair'].apply(detect_goal_gene_hits)
    goal_mask = goal_hits.apply(lambda x: bool(x[0]))
    lr_goal_sub = fastccc_long[goal_mask].copy()

    if not lr_goal_sub.empty:
        hit_df = pd.DataFrame(goal_hits[goal_mask].tolist(), columns=['goal_genes','goal_hit_side'], index=lr_goal_sub.index)
        lr_goal_sub[['goal_genes','goal_hit_side']] = hit_df
        lr_goal_sub['ad_axis'] = lr_goal_sub.apply(
            lambda r: detect_ad_axis(r.get('lr_pair', ''), r.get('classification', '')),
            axis=1
        )

        lr_goal_agg = (lr_goal_sub.groupby([
                        'sample_id','cellpair','lr_pair','classification','directionality','is_ppi',
                        'goal_genes','goal_hit_side','ad_axis'
                    ])
                    .agg(sum_strength=('strength','sum'))
                    .reset_index())
        lr_goal_agg['entity'] = lr_goal_agg['cellpair'] + '::' + lr_goal_agg['lr_pair']

        # Zero-fill donor x entity for stable continuous-age modeling
        entity_meta = lr_goal_agg[[
            'entity','cellpair','lr_pair','classification','directionality','is_ppi',
            'goal_genes','goal_hit_side','ad_axis'
        ]].drop_duplicates('entity')
        all_sids = donor_base['sample_id'].unique()
        full_idx = pd.MultiIndex.from_product([all_sids, entity_meta['entity'].values],
                                              names=['sample_id','entity']).to_frame(index=False)
        lr_goal_full = (full_idx.merge(entity_meta, on='entity', how='left')
                        .merge(lr_goal_agg[['sample_id','entity','sum_strength']],
                               on=['sample_id','entity'], how='left')
                        .merge(donor_base[['sample_id','age','age_group','cohort_group','log_total_cells']],
                               on='sample_id', how='left'))
        lr_goal_full['sum_strength'] = lr_goal_full['sum_strength'].fillna(0.0)
        lr_goal_full = add_fractions(lr_goal_full, 'cellpair', ct_frac_df)

        goal_models_raw = dual_track(lr_goal_full, 'entity', metrics=['sum_strength'])
        if not goal_models_raw.empty:
            goal_models_raw[['cellpair','lr_pair']] = (
                goal_models_raw['entity'].str.split('::', expand=True).iloc[:, :2])
            goal_models_raw = goal_models_raw.merge(entity_meta, on=['entity','cellpair','lr_pair'], how='left')

            p_goal = first_existing_col(goal_models_raw, ['p_linear'], 'p_linear')
            goal_models_raw['is_microglia_receiver'] = goal_models_raw['cellpair'].str.split('|').str[-1].str.contains('micro', case=False, na=False)
            goal_models_raw['abs_beta_age'] = goal_models_raw['beta_age'].abs()

            selected_lr_anchor = (goal_models_raw
                                  .sort_values([p_goal, 'abs_beta_age'], ascending=[True, False])
                                  .drop_duplicates('entity')
                                  .head(40)
                                  .copy())
            goal_models_raw.to_csv(TAB_DIR / 'S7b_goal_gene_anchor_models.tsv', sep='	', index=False)
            selected_lr_anchor.to_csv(TAB_DIR / 'S7b_goal_gene_anchor_top.tsv', sep='	', index=False)
            lr_goal_sub.to_csv(TAB_DIR / 'S7b_goal_gene_anchor_long.tsv', sep='	', index=False)
            print(f'AD goal-gene anchored LR entities: {goal_models_raw["entity"].nunique()}')
            print(f'Selected anchored LR for display: {len(selected_lr_anchor)}')
            display(selected_lr_anchor[[
                'entity','goal_genes','goal_hit_side','ad_axis','cohort_sensitivity_label',
                'beta_age','beta_10y','spearman_rho','loo_direction_stability','block_perm_p','residual_spearman_rho','OLS_p_display',p_goal,'classification'
            ]].head(10))
        else:
            goal_models_raw = selected_lr_anchor = pd.DataFrame()
            print('WARNING: goal_models_raw empty')
    else:
        lr_goal_agg = goal_models_raw = selected_lr_anchor = pd.DataFrame()
        print('WARNING: no LR pairs matched AD goal genes')
else:
    lr_goal_agg = goal_models_raw = selected_lr_anchor = pd.DataFrame()
    print('WARNING: fastccc_long empty - Stage 7B skipped')


## Stage 8 — 网络中心性分析

对每个donor构建有向加权CCC网络，计算6种中心性指标，对每个(celltype, metric)做连续+分组分析。

In [ ]:
# ============================================================
# Stage 8: Network centrality per donor
# ============================================================
if not fastccc_long.empty:
    print('Building network centrality panel...')
    network_panel = build_network_metrics_panel(fastccc_long, donor_base)
    network_panel.to_csv(TAB_DIR / 'network_metrics_panel.tsv', sep='\t', index=False)
    print(f'network_panel: {len(network_panel)} rows')

    # Models per (celltype, metric_name), using one donor subset for OLS and Spearman.
    net_records = []
    for (ct, mn), grp in network_panel.groupby(['celltype','metric_name']):
        model_grp = (grp[['sample_id','age','age_group','cohort_group','log_total_cells','metric_value']]
                     .replace([np.inf, -np.inf], np.nan)
                     .dropna()
                     .rename(columns={'metric_value': 'value'})
                     .copy())
        model_grp['sender_fraction'] = 0.0
        model_grp['receiver_fraction'] = 0.0
        cont = fit_age_model(model_grp['value'].values,
                             model_grp[['sample_id','age','cohort_group','log_total_cells']])
        grp_s = fit_group_stats(model_grp['value'].values, model_grp['age_group'].values)
        sp_s = spearman_age_trend(model_grp['value'].values, model_grp['age'].values)
        eff_s = effect_support_summary(
            model_grp['value'].values,
            beta_age=cont.get('beta_age', np.nan),
            beta_age_se=cont.get('beta_age_se', np.nan))
        tmp_row = {**sp_s.to_dict(), **eff_s.to_dict()}
        should_perm = raw_spearman_discovery_mask(tmp_row)
        seed = zlib.adler32(f'{ct}::{mn}'.encode('utf-8'))
        loo_s = leave_one_cohort_out_spearman(model_grp, full_rho=sp_s.get('spearman_rho', np.nan))
        block_s = (cohort_block_permutation_spearman(model_grp, observed_rho=sp_s.get('spearman_rho', np.nan), seed=seed)
                   if should_perm else pd.Series({'block_perm_p': np.nan, 'block_perm_n': 0}))
        resid_s = residual_spearman(model_grp) if should_perm else pd.Series({'residual_spearman_rho': np.nan, 'residual_spearman_p': np.nan})
        net_records.append({'celltype': ct, 'metric_name': mn,
                            **cont, **eff_s.to_dict(), **grp_s, **sp_s.to_dict(),
                            **loo_s.to_dict(), **block_s.to_dict(), **resid_s.to_dict(),
                            'direction_consistent': direction_consistent(cont['beta_age'], grp_s['direction'])})
    network_models = pd.DataFrame(net_records)
    network_models = mark_trend_support(network_models)
    network_models = add_visual_evidence(network_models)
    network_models.to_csv(TAB_DIR / 'S8_network_centrality_models.tsv', sep='\t', index=False)
    network_models_with_entity = network_models.assign(
        role_entity=network_models['celltype'].astype(str) + '::' + network_models['metric_name'].astype(str))
    network_cohort_summary, network_ols_sensitivity = save_cohort_sensitivity_tables(
        network_models_with_entity, 'role_entity', 'S8_network')
    network_discovery_summary = save_spearman_discovery_summary(
        network_models_with_entity, 'role_entity', 'S8_network')
    network_spearman_summary = save_spearman_summary(
        network_models_with_entity, 'role_entity', 'S8_network_spearman_trend_summary.tsv')
    print(f'network_models: {len(network_models)} rows')
    if not network_discovery_summary.empty:
        print('Top network-role Spearman discovery trends:')
        display(network_discovery_summary.head(8))

    # Young/Old aggregate networks
    g_young = build_group_network(fastccc_long, donor_base, 'Young adult', top_n_edges=16)
    g_old   = build_group_network(fastccc_long, donor_base, 'Elderly',   top_n_edges=16)
    print(f'Young network: {g_young.number_of_nodes()} nodes, {g_young.number_of_edges()} edges')
    print(f'Old network:   {g_old.number_of_nodes()} nodes, {g_old.number_of_edges()} edges')
else:
    network_panel = network_models = network_spearman_summary = network_cohort_summary = network_ols_sensitivity = network_discovery_summary = pd.DataFrame()
    g_young = g_old = None
    print('WARNING: fastccc_long empty - Stage 8 skipped')

## Stage 9 - Cohort-aware sensitivity and evidence summary

Main evidence uses donor-level nominal Spearman age trends. Main candidates are selected by nominal p, rho, donor coverage, and cohort sensitivity. Classification uses p < 0.01; LR drivers prioritize p < 0.001. OLS is retained only as cohort/composition sensitivity. Leave-one-cohort-out, cohort-block permutation, and residual Spearman are used to label cohort sensitivity.


In [ ]:
import re
# ============================================================
# Stage 9: Cohort-aware sensitivity and manuscript evidence summary
# ============================================================

ev_rows = []

def layer_for_thresholds(layer):
    if layer == 'classification':
        return 'classification'
    if layer == 'lr_driver':
        return 'lr_driver'
    return 'general'


def add_rows(df, layer, entity_col):
    if df is None or df.empty:
        return
    df = add_interpretation_annotations(df.copy(), layer=layer_for_thresholds(layer))
    rt_recs = set(receptor_tf['Receptor'].tolist()) if not receptor_tf.empty else set()
    for _, row in df.iterrows():
        ent = row.get(entity_col, '')
        if (pd.isna(ent) or ent == '') and {'celltype', 'metric_name'}.issubset(df.columns):
            ent = f"{row.get('celltype', '')}::{row.get('metric_name', '')}"
        lr = row.get('lr_pair', '')
        receptor_tf_flag = False
        if isinstance(lr, str) and '_' in lr:
            receptor_part = lr.split('_', 1)[1]
            receptor_tf_flag = any(g in rt_recs for g in re.split(r'[+;&|,\s]+', receptor_part))
        ev_rows.append({
            'layer': layer,
            'entity': ent,
            'metric': row.get('metric', row.get('metric_name', np.nan)),
            'cohort_sensitivity_label': row.get('cohort_sensitivity_label', 'not selected'),
            'discovery_trend_label': row.get('discovery_trend_label', 'not selected'),
            'interpretation_tier': row.get('interpretation_tier', np.nan),
            'main_text_candidate': row.get('main_text_candidate', False),
            'recommended_location': row.get('recommended_location', np.nan),
            'sparse_signal_flag': row.get('sparse_signal_flag', np.nan),
            'covariate_sensitivity_label': row.get('covariate_sensitivity_label', np.nan),
            'group_pattern_inconsistent': row.get('group_pattern_inconsistent', np.nan),
            'spearman_rho': row.get('spearman_rho', np.nan),
            'spearman_p': row.get('spearman_p', np.nan),
            'loo_direction_stability': row.get('loo_direction_stability', np.nan),
            'rho_min_loo': row.get('rho_min_loo', np.nan),
            'rho_max_loo': row.get('rho_max_loo', np.nan),
            'block_perm_p': row.get('block_perm_p', np.nan),
            'residual_spearman_rho': row.get('residual_spearman_rho', np.nan),
            'residual_spearman_p': row.get('residual_spearman_p', np.nan),
            'beta_10y': row.get('beta_10y', np.nan),
            'OLS_p_display': row.get('OLS_p_display', np.nan),
            'ols_spearman_direction_consistent': row.get('ols_spearman_direction_consistent', np.nan),
            'OLS_sensitivity_label': row.get('OLS_sensitivity_label', np.nan),
            'n_donors_modeled': row.get('n_donors_modeled', np.nan),
            'n_nonzero_donors': row.get('n_nonzero_donors', np.nan),
            'donor_frac': row.get('donor_frac', np.nan),
            'classification': row.get('classification', np.nan),
            'directionality': row.get('directionality', np.nan),
            'receptor_tf_flag': receptor_tf_flag,
        })

if 'broad_models' in globals() and not broad_models.empty:
    add_rows(broad_models, 'broad_cellpair', 'broad_pair')
if 'fine_models' in globals() and not fine_models.empty:
    add_rows(fine_models, 'fine_cellpair', 'cellpair')
if 'class_models' in globals() and not class_models.empty:
    add_rows(class_models, 'classification', 'classification')
if 'lr_models_raw' in globals() and not lr_models_raw.empty:
    add_rows(lr_models_raw, 'lr_driver', 'entity')
if 'network_models' in globals() and not network_models.empty:
    add_rows(network_models, 'network_role', 'role_entity')

unified_evidence = pd.DataFrame(ev_rows)
if not unified_evidence.empty:
    unified_evidence['abs_spearman_rho'] = unified_evidence['spearman_rho'].abs()
    tier_order = {
        'Tier 1: main; robust nominal trend': 0,
        'Tier 2: main-caution; covariate-sensitive nominal trend': 1,
        'Tier 3: supplementary exploratory trend': 2,
        'not selected': 3,
        'Tier 4: sparse signal; not interpreted': 4,
    }
    unified_evidence['_tier_order'] = unified_evidence['interpretation_tier'].map(tier_order).fillna(9)
    unified_evidence = unified_evidence.sort_values(
        ['_tier_order', 'main_text_candidate', 'spearman_p', 'abs_spearman_rho', 'donor_frac'],
        ascending=[True, False, True, False, False])
    unified_evidence.drop(columns=['_tier_order']).to_csv(
        TAB_DIR / 'S9_cohort_aware_evidence_summary.tsv', sep='\t', index=False)

    ols_cols = [c for c in ['layer','entity','metric','spearman_rho','spearman_p',
                            'beta_10y','OLS_p_display','ols_spearman_direction_consistent','OLS_sensitivity_label',
                            'covariate_sensitivity_label','interpretation_tier','main_text_candidate','recommended_location',
                            'n_donors_modeled','n_nonzero_donors','donor_frac'] if c in unified_evidence.columns]
    unified_evidence[ols_cols].to_csv(TAB_DIR / 'S9_OLS_sensitivity_summary.tsv', sep='\t', index=False)

    print('Interpretation tier distribution:')
    display(unified_evidence['interpretation_tier'].value_counts(dropna=False).to_frame('n'))
    print('Recommended location distribution:')
    display(unified_evidence['recommended_location'].value_counts(dropna=False).to_frame('n'))
    print('Cohort sensitivity label distribution:')
    display(unified_evidence['cohort_sensitivity_label'].value_counts(dropna=False).to_frame('n'))
    print('Discovery trend label distribution:')
    display(unified_evidence['discovery_trend_label'].value_counts(dropna=False).to_frame('n'))
    display_cols = [c for c in ['layer','entity','metric','interpretation_tier','recommended_location','main_text_candidate',
                                'cohort_sensitivity_label','covariate_sensitivity_label','spearman_rho','spearman_p',
                                'loo_direction_stability','block_perm_p','residual_spearman_rho',
                                'n_nonzero_donors','donor_frac'] if c in unified_evidence.columns]
    display(unified_evidence[display_cols].head(25))
else:
    print('No evidence rows assembled.')


## Figure 1 - Dataset Overview + Continuous Age Context

Show donor age, cohort structure, and cell composition first. Composition is treated as context and a covariate, not as the main biological endpoint.


In [ ]:
set_pub_style()

# Load all obs for composition plot
all_obs = []
for _, row in samples.iterrows():
    try:
        adata = sc.read_h5ad(row['h5ad_path'])
        obs = adata.obs[[META_KEY]].copy()
        obs['sample_id'] = row['sample_id']
        obs['age'] = row['age']
        all_obs.append(obs)
        del adata; gc.collect()
    except Exception:
        pass

if all_obs:
    obs_all = pd.concat(all_obs, ignore_index=True).rename(columns={META_KEY: 'celltype'})
    fig1a, _ = plot_donor_composition(obs_all, figsize=(14, 4))
    fig1a.suptitle('Figure 1A  Donor cell composition ordered by age', fontweight='bold')
    save_fig(fig1a, 'Figure1A_donor_composition', FIG_DIR)
    plt.show()

# Demographics
fig1c, axes1c = plt.subplots(1, 3, figsize=(12, 4))
donor_base['age'].hist(bins=15, ax=axes1c[0], color='#4C8DC0', edgecolor='white')
axes1c[0].set(xlabel='Age (years)', ylabel='N donors', title='Age distribution')
donor_base['age_group'].value_counts().reindex(['Young adult', 'Adult', 'Elderly']).plot(
    kind='bar', ax=axes1c[1], color=[AGE_PAL[g] for g in ['Young adult', 'Adult', 'Elderly']],
    edgecolor='white')
axes1c[1].set(xlabel='', ylabel='N donors', title='Age groups')
axes1c[1].set_xticklabels(axes1c[1].get_xticklabels(), rotation=0)
donor_base['cohort'].value_counts().plot(kind='bar', ax=axes1c[2],
    color='#9B6DA5', edgecolor='white')
axes1c[2].set(xlabel='', ylabel='N donors', title='Cohort structure')
axes1c[2].set_xticklabels(axes1c[2].get_xticklabels(), rotation=35, ha='right', fontsize=12)
plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
save_fig(fig1c, 'Figure1C_demographics', FIG_DIR)
plt.show()

# Global communication trajectories
if not global_comm.empty:
    fig1de, axes1de = plt.subplots(1, 2, figsize=(10, 4))

    plot_loess_trajectory(global_comm['age'], global_comm['total_sum_strength'],
                          global_comm['age_group'], label='1D  Global CCC strength across age',
                          ylabel='Sum strength (all pairs)', n_boot=200, ax=axes1de[0])

    plot_loess_trajectory(global_comm['age'], global_comm['n_sig_lr'],
                          global_comm['age_group'], label='1E  Global LRI diversity across age',
                          ylabel='Count', n_boot=200, ax=axes1de[1])

    # 只保留 Young adult / Adult / Elderly，去除重复的 Young / Middle / Old
    KEEP_LABELS = {'Young adult', 'Adult', 'Elderly'}
    for ax in axes1de:
        handles, labels = ax.get_legend_handles_labels()
        filtered = [(h, l) for h, l in zip(handles, labels) if l in KEEP_LABELS]
        if ax.get_legend():
            ax.get_legend().remove()
        if filtered:
            fh, fl = zip(*filtered)
            ax.legend(fh, fl, frameon=False, fontsize=12, loc='best', title=None)

    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig1de, 'Figure1DE_global_trajectories', FIG_DIR)
    plt.show()

print('Figure 1 saved')

## Figure 2 - Broad Cell-Pair Continuous Age Remodeling

Use age-effect forest plots and LOESS trajectories as the main panels. Young adult/Adult/Elderly networks or violin plots are kept as auxiliary views for early, late, and cumulative changes.


In [ ]:

# ============================================================
# Shared Figure Utilities - Spearman-first, cohort-aware visuals
# ============================================================
import re

UP_COLOR = '#C44E52'
DOWN_COLOR = '#3B74AF'
OLD_COLOR = '#D95F5F'
YOUNG_COLOR = '#315EAA'
NEUTRAL_COLOR = '#8A8A8A'

set_pub_style()
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 12,
    'axes.linewidth': 0.9,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'axes.titleweight': 'regular',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'xtick.major.width': 0.9,
    'ytick.major.width': 0.9,
    'xtick.major.size': 4.2,
    'ytick.major.size': 4.2,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'legend.fontsize': 12,
    'legend.frameon': False,
    'legend.borderpad': 0.2,
    'lines.linewidth': 1.0,
    'lines.markersize': 8,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'savefig.transparent': False,
    'figure.dpi': 150,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
})

def first_existing_col(df, candidates, fallback=None):
    for c in candidates:
        if c in df.columns:
            return c
    return fallback


def choose_trend_pcol(df):
    for c in ['spearman_p', 'p_overall', 'p_linear']:
        if c in df.columns:
            return c
    return df.columns[0]


def p_label(p_col):
    return 'pS' if p_col == 'spearman_p' else 'p'


def trend_support_mask(df, min_nonzero=8, min_sd=0.0):
    if df.empty:
        return pd.Series(dtype=bool)
    mask = pd.Series(True, index=df.index)
    if 'n_nonzero_donors' in df.columns:
        mask &= df['n_nonzero_donors'].fillna(0) >= min_nonzero
    if 'value_sd' in df.columns:
        mask &= df['value_sd'].fillna(0) > min_sd
    if 'spearman_rho' in df.columns:
        mask &= df['spearman_rho'].notna()
    return mask


def visual_candidate_df(model_df, min_nonzero=8, min_sd=0.0, allow_nominal=True, allow_nominal_fallback=True):
    if model_df.empty:
        return model_df.copy()
    df0 = model_df.copy()
    df = df0[trend_support_mask(df0, min_nonzero=min_nonzero, min_sd=min_sd)].copy()
    if df.empty:
        return df
    df = add_cohort_sensitivity_labels(df) if 'cohort_sensitivity_label' not in df.columns else df
    selected = df[df.get('discovery_selected', pd.Series(False, index=df.index)).fillna(False)].copy()
    if selected.empty and allow_nominal_fallback:
        selected = df.copy()
        selected['discovery_trend_label'] = 'top nominal trends only; no p<0.05 hits'
        selected['cohort_sensitivity_label'] = 'top nominal trends only; no p<0.05 hits'
    elif selected.empty:
        return selected
    selected['abs_spearman_rho'] = selected['spearman_rho'].abs() if 'spearman_rho' in selected.columns else 0.0
    order = {'robust trend': 0, 'cohort-sensitive trend': 1, 'exploratory trend': 2,
             'top nominal trends only; no p<0.05 hits': 3, 'not selected': 9}
    selected['_sensitivity_order'] = selected['cohort_sensitivity_label'].map(order).fillna(9)
    sort_cols = ['_sensitivity_order']
    asc = [True]
    if 'spearman_p' in selected.columns:
        sort_cols.append('spearman_p'); asc.append(True)
    sort_cols.append('abs_spearman_rho'); asc.append(False)
    if 'donor_frac' in selected.columns:
        sort_cols.append('donor_frac'); asc.append(False)
    return selected.sort_values(sort_cols, ascending=asc)


def select_visual_entities(model_df, entity_col, top_n=12, min_nonzero=8, min_sd=0.0, allow_nominal=True):
    df = visual_candidate_df(model_df, min_nonzero=min_nonzero, min_sd=min_sd, allow_nominal=allow_nominal)
    if df.empty:
        return []
    return df[entity_col].drop_duplicates().head(top_n).tolist()


STRICT_VISUAL_P = 0.05
STRICT_VISUAL_RHO = 0.30
STRICT_VISUAL_DONOR_FRAC = 0.35


def strict_nominal_visual_pool(model_df, min_p=STRICT_VISUAL_P, min_abs_rho=STRICT_VISUAL_RHO,
                               min_donor_frac=STRICT_VISUAL_DONOR_FRAC, min_sd=0.0):
    """Strict nominal discovery pool used by main figures. No multiple-testing filters."""
    if model_df is None or model_df.empty:
        return pd.DataFrame()
    df = model_df.copy()
    if 'cohort_sensitivity_label' not in df.columns:
        df = add_cohort_sensitivity_labels(df)
    needed = {'spearman_rho', 'spearman_p'}
    if not needed.issubset(df.columns):
        return pd.DataFrame(columns=df.columns)
    mask = (df['spearman_p'].fillna(1.0) < min_p)
    mask &= df['spearman_rho'].abs().fillna(0.0) > min_abs_rho
    if 'donor_frac' in df.columns:
        mask &= df['donor_frac'].fillna(0.0) >= min_donor_frac
    if {'n_nonzero_donors', 'n_donors_modeled'}.issubset(df.columns):
        min_nonzero = np.ceil(min_donor_frac * df['n_donors_modeled'].fillna(0).astype(float))
        mask &= df['n_nonzero_donors'].fillna(0).astype(float) >= min_nonzero
    if 'value_sd' in df.columns:
        mask &= df['value_sd'].fillna(0.0) > min_sd
    pool = df[mask].copy()
    if pool.empty:
        return pool
    pool['abs_spearman_rho'] = pool['spearman_rho'].abs()
    pool['aging_direction'] = np.where(pool['spearman_rho'] > 0, 'aging-up',
                               np.where(pool['spearman_rho'] < 0, 'aging-down', 'flat'))
    sort_cols = ['spearman_p', 'abs_spearman_rho']
    asc = [True, False]
    if 'donor_frac' in pool.columns:
        sort_cols.append('donor_frac'); asc.append(False)
    return pool.sort_values(sort_cols, ascending=asc)


def _entity_sender_receiver(entity):
    text = str(entity)
    if '::' in text:
        text = text.split('::', 1)[0]
    if '|' in text:
        left, right = text.split('|', 1)
    elif '->' in text:
        left, right = text.split('->', 1)
    else:
        left, right = text, text
    return left.strip(), right.strip()


def _take_diverse_rows(df, entity_col, n, max_per_sender=None, max_per_receiver=None,
                       sender_counts=None, receiver_counts=None):
    if df is None or df.empty or n <= 0:
        return [], sender_counts or {}, receiver_counts or {}
    sender_counts = {} if sender_counts is None else dict(sender_counts)
    receiver_counts = {} if receiver_counts is None else dict(receiver_counts)
    rows = []
    used = set()
    for _, row in df.drop_duplicates(entity_col).iterrows():
        ent = row[entity_col]
        if ent in used:
            continue
        sender, receiver = _entity_sender_receiver(ent)
        if max_per_sender is not None and sender_counts.get(sender, 0) >= max_per_sender:
            continue
        if max_per_receiver is not None and receiver_counts.get(receiver, 0) >= max_per_receiver:
            continue
        rows.append(row)
        used.add(ent)
        sender_counts[sender] = sender_counts.get(sender, 0) + 1
        receiver_counts[receiver] = receiver_counts.get(receiver, 0) + 1
        if len(rows) >= n:
            break
    return rows, sender_counts, receiver_counts


def _interleave_direction_rows(up_rows, down_rows):
    out = []
    max_len = max(len(up_rows), len(down_rows))
    for i in range(max_len):
        if i < len(up_rows):
            out.append(up_rows[i])
        if i < len(down_rows):
            out.append(down_rows[i])
    return out


def select_balanced_visual_entities(model_df, entity_col, top_n=12, min_up=None, min_down=None,
                                    max_per_sender=None, max_per_receiver=None, label=None):
    """Select strict nominal discovery hits while balancing aging-up and aging-down directions."""
    pool = strict_nominal_visual_pool(model_df)
    if pool.empty or entity_col not in pool.columns:
        if label:
            print(f'{label}: no entities passed strict nominal discovery filter')
        return []
    up_pool = pool[pool['spearman_rho'] > 0].copy()
    down_pool = pool[pool['spearman_rho'] < 0].copy()
    target_up = min_up if min_up is not None else top_n // 2
    target_down = min_down if min_down is not None else top_n - target_up
    if target_up + target_down > top_n:
        target_down = max(0, top_n - target_up)
    sender_counts, receiver_counts = {}, {}
    up_rows, sender_counts, receiver_counts = _take_diverse_rows(
        up_pool, entity_col, target_up, max_per_sender, max_per_receiver, sender_counts, receiver_counts)
    down_rows, sender_counts, receiver_counts = _take_diverse_rows(
        down_pool, entity_col, target_down, max_per_sender, max_per_receiver, sender_counts, receiver_counts)
    selected_rows = _interleave_direction_rows(up_rows, down_rows)
    selected_entities = [r[entity_col] for r in selected_rows]

    if len(selected_entities) < top_n:
        remaining = pool[~pool[entity_col].isin(selected_entities)].copy()
        fill_rows, sender_counts, receiver_counts = _take_diverse_rows(
            remaining, entity_col, top_n - len(selected_entities),
            max_per_sender, max_per_receiver, sender_counts, receiver_counts)
        selected_rows.extend(fill_rows)
        selected_entities = [r[entity_col] for r in selected_rows]

    if len(selected_entities) < top_n:
        remaining = pool[~pool[entity_col].isin(selected_entities)].drop_duplicates(entity_col)
        selected_entities.extend(remaining[entity_col].head(top_n - len(selected_entities)).tolist())

    selected_entities = list(dict.fromkeys(selected_entities))[:top_n]
    if label:
        sel = pool[pool[entity_col].isin(selected_entities)]
        n_up = int((sel['spearman_rho'] > 0).sum())
        n_down = int((sel['spearman_rho'] < 0).sum())
        print(f'{label}: selected {len(selected_entities)} entities ({n_up} aging-up, {n_down} aging-down) from strict nominal discovery pool')
        if n_up == 0:
            print(f'{label}: no aging-up entities passed P < 0.05, |rho| > 0.30, donor_frac >= 0.35')
        if n_down == 0:
            print(f'{label}: no aging-down entities passed P < 0.05, |rho| > 0.30, donor_frac >= 0.35')
    return selected_entities


def save_visual_selection_table(model_df, entity_col, entities, outfile):
    """Save the exact entities used by a main figure for source-data review."""
    if model_df is None or model_df.empty or not entities or entity_col not in model_df.columns:
        return pd.DataFrame()
    df = model_df[model_df[entity_col].isin(entities)].copy()
    if df.empty:
        return df
    df['_entity_order'] = pd.Categorical(df[entity_col], categories=entities, ordered=True)
    if 'aging_direction' not in df.columns and 'spearman_rho' in df.columns:
        df['aging_direction'] = np.where(df['spearman_rho'] > 0, 'aging-up',
                                  np.where(df['spearman_rho'] < 0, 'aging-down', 'flat'))
    keep = [c for c in [entity_col, 'metric', 'aging_direction', 'spearman_rho', 'spearman_p',
                        'n_nonzero_donors', 'n_donors_modeled', 'donor_frac',
                        'cohort_sensitivity_label', 'discovery_trend_label',
                        'interpretation_tier', 'recommended_location'] if c in df.columns]
    out = df.sort_values('_entity_order')[keep].drop_duplicates()
    out.to_csv(TAB_DIR / f'{outfile}.tsv', sep='	', index=False)
    return out


def annotate_trend_axis(ax, row, p_col=None):
    parts = []
    if row is not None and len(row):
        if 'spearman_rho' in row and pd.notna(row['spearman_rho']):
            parts.append(f"rho={float(row['spearman_rho']):.2f}")
        if 'spearman_p' in row and pd.notna(row['spearman_p']):
            parts.append(f"nominal P={float(row['spearman_p']):.2g}")
        if 'n_nonzero_donors' in row and 'n_donors_modeled' in row and pd.notna(row.get('n_nonzero_donors', np.nan)) and pd.notna(row.get('n_donors_modeled', np.nan)):
            parts.append(f"nonzero donors={int(row['n_nonzero_donors'])}/{int(row['n_donors_modeled'])}")
        label = row.get('cohort_sensitivity_label', row.get('discovery_trend_label', ''))
        if isinstance(label, str) and label and label != 'not selected':
            parts.append(label)
    if parts:
        ax.text(0.03, 0.97, ' | '.join(parts), transform=ax.transAxes,
                ha='left', va='top', fontsize=12,
                bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=2))


def plot_spearman_lollipop(model_df, entity_col, entities, title, outfile, figsize=(8, 6), label_col=None):
    if model_df.empty or not entities:
        return None
    p_sp = first_existing_col(model_df, ['spearman_p'], None)
    df = model_df[model_df[entity_col].isin(entities)].copy()
    df = visual_candidate_df(df, min_nonzero=0, min_sd=0.0)
    if df.empty:
        return None
    df = df.drop_duplicates(entity_col).sort_values('spearman_rho')
    y = np.arange(len(df))
    labels_trend = df.get('discovery_trend_label', pd.Series('exploratory trend', index=df.index))
    colors = np.where(df['spearman_rho'] >= 0, UP_COLOR, DOWN_COLOR)
    alpha = np.where(labels_trend.eq('nominal Spearman trend'), 0.88, 0.52)
    edge = np.where(labels_trend.eq('nominal Spearman trend'), 'white', '#6E6E6E')
    sizes = np.full(len(df), 90.0)
    if p_sp is not None:
        sizes = np.clip(-np.log10(df[p_sp].clip(lower=1e-300)).values * 26, 45, 260)
    fig, ax = plt.subplots(figsize=figsize)
    ax.hlines(y, 0, df['spearman_rho'], color='#B8B8B8', lw=1.2)
    for i, (_, row) in enumerate(df.iterrows()):
        ax.scatter(row['spearman_rho'], y[i], s=sizes[i], c=colors[i], alpha=alpha[i],
                   edgecolors=edge[i], linewidths=2.3, zorder=3)
    ax.axvline(0, ls='--', lw=1.0, color='black')
    labels = df[label_col] if label_col and label_col in df.columns else df[entity_col]
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel('Spearman rho with age')
    ax.set_title(title)
    caption = 'Nominal Spearman discovery trends; dot size is based on Spearman p.'
    ax.text(0.0, -0.13, caption, transform=ax.transAxes, ha='left', va='top', fontsize=12, color='#555555')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def donor_ordered_heatmap(value_df, entity_col, value_col, entities, title, outfile, figsize=(13, 6), row_label_map=None, clip_z=None, caption=None):
    if value_df.empty or not entities:
        return None
    donor_meta = donor_base.copy()
    donor_meta['sample_id'] = donor_meta['sample_id'].astype(str)
    donor_order = donor_meta.sort_values('age')['sample_id'].tolist()
    ages = donor_meta.sort_values('age')['age'].to_numpy()
    df = value_df[value_df[entity_col].isin(entities)].copy()
    df['sample_id'] = df['sample_id'].astype(str)
    hm = (df.pivot_table(index=entity_col, columns='sample_id', values=value_col, aggfunc='mean')
          .reindex(index=entities, columns=donor_order))
    hm_z = hm.sub(hm.mean(axis=1), axis=0).div(hm.std(axis=1).replace(0, np.nan), axis=0).fillna(0)
    if clip_z is not None:
        hm_z = hm_z.clip(lower=clip_z[0], upper=clip_z[1])
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(hm_z, cmap='RdBu_r', center=0, ax=ax,
                cbar_kws={'label': f'row z-score {value_col}'}, xticklabels=False)
    if len(donor_order) > 0:
        tick_idx = np.linspace(0, len(donor_order)-1, min(8, len(donor_order)), dtype=int)
        ax.set_xticks(tick_idx + 0.5)
        ax.set_xticklabels([f'{ages[i]:.0f}' for i in tick_idx], rotation=0)
    ax.set_xlabel('Donors ordered by age')
    if row_label_map:
        ax.set_yticklabels([row_label_map.get(str(t.get_text()), str(t.get_text())) for t in ax.get_yticklabels()])
    ax.set_ylabel(entity_col)
    ax.set_title(title)
    if caption:
        ax.text(0.0, -0.12, caption, transform=ax.transAxes, ha='left', va='top', fontsize=12, color='#555555')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def plot_trend_grid(value_df, model_df, entity_col, value_col, entities, title_prefix, outfile, ncols=3):
    if value_df.empty or not entities:
        return None
    p_col = choose_trend_pcol(model_df)
    n = len(entities)
    ncols = min(ncols, max(1, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.6*ncols, 4.45*nrows), squeeze=False)
    axes = axes.flatten()
    for ax, ent in zip(axes, entities):
        sub = value_df[value_df[entity_col] == ent]
        rm = model_df[model_df[entity_col] == ent]
        row = rm.iloc[0] if len(rm) else None
        plot_loess_trajectory(sub['age'], sub[value_col], sub['age_group'],
                              label=str(ent)[:32], beta=None, q=None, n_boot=100, ax=ax)
        if row is not None:
            annotate_trend_axis(ax, row, p_col=p_col)
    for ax in axes[n:]:
        ax.axis('off')
    fig.suptitle(title_prefix, y=1.01, fontweight='bold', fontsize=13)
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig

In [ ]:


def _celltype_superclass(celltype):
    """Collapse detailed PFC labels into coarse communication compartments."""
    ct = str(celltype)
    ctl = ct.lower()
    if any(x in ctl for x in ['astro', 'micro', 'oligo', 'opc']):
        return 'Glia'
    if any(x in ctl for x in ['endo', 'vlmc', 'pericyte', 'vascular']):
        return 'Vascular'
    inhibitory_terms = ['sst', 'pvalb', 'vip', 'lamp5', 'sncg', 'pax6', 'chandelier', 'rosehip', 'in-']
    if any(x in ctl for x in inhibitory_terms):
        return 'Inhibitory'
    excitatory_terms = ['l2', 'l3', 'l4', 'l5', 'l6', 'it', 'ct', 'et', 'np', 'pc', 'excit']
    if any(x in ctl for x in excitatory_terms):
        return 'Excitatory'
    return 'Other'


def _age_group_diff_matrix(panel, sender_col, receiver_col, value_col='sum_strength',
                           left='Young adult', right='Elderly', aggregate=None):
    if panel is None or panel.empty:
        return pd.DataFrame(), pd.DataFrame()
    df = panel.copy()
    if aggregate == 'coarse':
        df['sender_plot'] = df[sender_col].map(_celltype_superclass)
        df['receiver_plot'] = df[receiver_col].map(_celltype_superclass)
    else:
        df['sender_plot'] = df[sender_col].astype(str)
        df['receiver_plot'] = df[receiver_col].astype(str)
    df = df[df['age_group'].isin([left, right])].copy()
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()
    donor_pair = (df.groupby(['sample_id','age_group','sender_plot','receiver_plot'], observed=True)[value_col]
                    .sum().reset_index())
    group_mean = (donor_pair.groupby(['age_group','sender_plot','receiver_plot'], observed=True)[value_col]
                    .mean().reset_index())
    wide = group_mean.pivot_table(index=['sender_plot','receiver_plot'], columns='age_group',
                                  values=value_col, fill_value=0.0)
    for g in [left, right]:
        if g not in wide.columns:
            wide[g] = 0.0
    wide['delta'] = wide[right] - wide[left]
    wide = wide.reset_index()
    mat = wide.pivot(index='sender_plot', columns='receiver_plot', values='delta').fillna(0.0)
    return mat, wide


def plot_age_group_differential_heatmaps(panel, sender_col, receiver_col, title, outfile,
                                         value_col='sum_strength', aggregate=None,
                                         comparisons=None, figsize=(14, 4.8), max_labels=28):
    """Draw Young adult/Adult/Elderly pairwise sender-receiver difference heatmaps."""
    if comparisons is None:
        comparisons = [('Young adult', 'Elderly', 'Old - Young'),
                       ('Young adult', 'Adult', 'Adult - Young adult'),
                       ('Adult', 'Elderly', 'Elderly - Adult')]
    mats, source_tables = [], []
    for left, right, label in comparisons:
        mat, wide = _age_group_diff_matrix(panel, sender_col, receiver_col, value_col, left, right, aggregate)
        if not mat.empty:
            mats.append((label, mat))
            wide['comparison'] = label
            source_tables.append(wide)
    if not mats:
        return None
    source = pd.concat(source_tables, ignore_index=True) if source_tables else pd.DataFrame()
    source.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)

    all_abs = np.concatenate([np.ravel(m.values) for _, m in mats if m.size])
    vmax = np.nanpercentile(np.abs(all_abs), 98) if len(all_abs) else 1.0
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0
    n = len(mats)
    fig, axes = plt.subplots(1, n, figsize=(max(figsize[0], 5.2*n), max(figsize[1], 4.6)), squeeze=False)
    axes = axes.flatten()
    for ax, (label, mat) in zip(axes, mats):
        row_order = mat.abs().sum(axis=1).sort_values(ascending=False).index.tolist()
        col_order = mat.abs().sum(axis=0).sort_values(ascending=False).index.tolist()
        mat = mat.reindex(index=row_order[:max_labels], columns=col_order[:max_labels])
        sns.heatmap(mat, ax=ax, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax,
                    linewidths=0.65, linecolor='#EEEEEE',
                    cbar=(ax is axes[-1]), cbar_kws={'label': 'Mean strength difference'})
        ax.set_title(label, fontsize=12)
        ax.set_xlabel('Receiver')
        ax.set_ylabel('Sender' if ax is axes[0] else '')
        ax.tick_params(axis='x', rotation=70)
        ax.tick_params(axis='y', rotation=0)
    fig.suptitle(title, y=1.03, fontsize=12)
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


set_pub_style()

if not broad_models.empty:
    bm_s = broad_models[broad_models['metric'] == 'mean_strength'].copy()
    pb = choose_trend_pcol(bm_s)
    top_broad = select_balanced_visual_entities(bm_s, 'broad_pair', top_n=12, min_up=6, min_down=6, label='Figure 2 broad pairs (mean_strength)')
    if not top_broad:
        top_broad = select_visual_entities(bm_s, 'broad_pair', top_n=12)
    save_visual_selection_table(bm_s, 'broad_pair', top_broad, 'Figure2_balanced_broad_pair_selection')

    # 2A: Spearman lollipop ranked by discovery trend and cohort sensitivity
    plot_df = bm_s[bm_s['broad_pair'].isin(top_broad)].copy()
    if not plot_df.empty:
        plot_spearman_lollipop(plot_df, 'broad_pair', top_broad,
                               '2A Broad pair Spearman age trends',
                               'Figure2A_broad_spearman_lollipop', figsize=(7.5, 7))

    # 2B: Continuous age trajectories
    top_traj = top_broad[:6]
    plot_trend_grid(broad_pair_full, bm_s, 'broad_pair', 'mean_strength', top_traj,
                    '2B Broad pair mean-strength continuous-age trajectories',
                    'Figure2B_broad_mean_strength_trajectories', ncols=3)

    # 2C: Donor-ordered heatmap
    donor_ordered_heatmap(broad_pair_full, 'broad_pair', 'mean_strength', top_broad,
                          '2C Broad pair mean-strength donor-ordered aging heatmap (row z-score; ordered by Spearman rho)',
                          'Figure2C_broad_mean_strength_donor_ordered_heatmap', figsize=(13, max(4.5, 0.35*len(top_broad))))



    # 2D: Broad sender-receiver differential heatmaps for the three age comparisons.
    if not broad_pair_full.empty and {'sender','receiver','age_group','sum_strength'}.issubset(broad_pair_full.columns):
        plot_age_group_differential_heatmaps(
            broad_pair_full, 'sender', 'receiver',
            '2D Broad sender-receiver communication burden differences across age groups',
            'Figure2D_broad_sender_receiver_pairwise_difference_heatmaps',
            value_col='sum_strength', aggregate=None, figsize=(15, 5.2), max_labels=24)

        # 2E: Coarse-grained overview: Excitatory/Inhibitory/Glia/Vascular compartments.
        plot_age_group_differential_heatmaps(
            broad_pair_full, 'sender', 'receiver',
            '2E Coarse communication burden differences across age groups',
            'Figure2E_coarse_compartment_pairwise_difference_heatmaps',
            value_col='sum_strength', aggregate='coarse', figsize=(10.5, 3.8), max_labels=8)

    # 2D: Young/Old network retained as supplementary visual context
    if g_young is not None:
        fig2d = plot_young_old_networks(g_young, g_old, figsize=(12, 5))
        save_fig(fig2d, 'FigureS2D_broad_young_old_networks_context', FIG_DIR)
        plt.show()

    print('Figure 2 saved')
else:
    print('No broad_models for Figure 2')


## Figure 3 - Fine Cell-Pair Continuous Age Trajectories

Emphasize selected fine-pair age effects, donor-ordered heatmaps, and continuous age trajectories. Grouped violin plots remain secondary validation panels.


In [ ]:


set_pub_style()

if not fine_models.empty:
    fm_s = fine_models[fine_models['metric'] == 'mean_strength'].copy()
    pf = choose_trend_pcol(fm_s)
    top_fine_stat = select_visual_entities(fm_s, 'cellpair', top_n=18)
    top_fine = select_balanced_visual_entities(fm_s, 'cellpair', top_n=12, min_up=6, min_down=6,
                                               max_per_sender=3, max_per_receiver=3, label='Figure 3 fine pairs (mean_strength)')
    if not top_fine:
        top_fine = top_fine_stat
    save_visual_selection_table(fm_s, 'cellpair', top_fine, 'Figure3_balanced_fine_pair_selection')
    save_visual_selection_table(fm_s, 'cellpair', top_fine_stat, 'FigureS3_top_statistical_fine_pair_selection')

    # 3A: Top continuous-age trajectories
    plot_trend_grid(fine_pair_full, fm_s, 'cellpair', 'mean_strength', top_fine[:9],
                    '3A Fine pair mean-strength continuous-age trajectories',
                    'Figure3A_fine_mean_strength_trajectories', ncols=3)

    # S3: Pure top statistical fine pairs retained as supplementary context
    if top_fine_stat:
        plot_trend_grid(fine_pair_full, fm_s, 'cellpair', 'mean_strength', top_fine_stat[:9],
                        'S3 Top statistical fine-pair mean-strength trajectories',
                        'FigureS3_top_statistical_fine_pair_mean_strength_trajectories', ncols=3)

    # 3B: Donor-ordered heatmap
    donor_ordered_heatmap(fine_pair_full, 'cellpair', 'mean_strength', top_fine,
                          '3B Fine pair mean-strength donor-ordered aging heatmap (row z-score; ordered by Spearman rho)',
                          'Figure3B_fine_mean_strength_donor_ordered_heatmap', figsize=(14, max(5, 0.30*len(top_fine))))

    # 3C: Metric consistency heatmap for the clearest top fine pairs
    if top_fine:
        qf_all = first_existing_col(fine_models, ['p_overall'], 'p_overall')
        fm_h = visual_candidate_df(fine_models[fine_models['cellpair'].isin(top_fine)].copy(), min_nonzero=8, min_sd=0.0)
        if not fm_h.empty:
            fm_h['signed_log10p'] = np.sign(fm_h['spearman_rho']) * (-np.log10(fm_h['spearman_p'].clip(lower=1e-300)))
            hm = (fm_h.pivot_table(index='cellpair', columns='metric', values='signed_log10p', aggfunc='mean')
                  .reindex(top_fine))
            hm = hm[[c for c in ['sum_strength','mean_strength','sig_lr_count'] if c in hm.columns]]
            fig3c, ax3c = plt.subplots(figsize=(7, max(5, 0.28 * len(hm))))
            sns.heatmap(hm, cmap='RdBu_r', center=0, ax=ax3c,
                        cbar_kws={'label': 'sign(rho) x -log10(pS)'},
                        linewidths=0.6, linecolor='white')
            ax3c.set_title('3C Fine pair Spearman metric consistency')
            ax3c.set_xlabel('Metric')
            ax3c.set_ylabel('Fine cell-pair')
            plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
            save_fig(fig3c, 'Figure3C_fine_metric_consistency_heatmap', FIG_DIR)
            plt.show()

    # 3D: Grouped violin retained as supplementary validation
    top_violin = top_fine[:8]
    if top_violin and not fine_pair_full.empty:
        fig3d, axes3d = plt.subplots(2, 4, figsize=(14, 7))
        axes3d = axes3d.flatten()
        for ax, pair in zip(axes3d, top_violin):
            sub = fine_pair_full[fine_pair_full['cellpair'] == pair].copy()
            rm  = fm_s[fm_s['cellpair'] == pair]
            mwu_col = first_existing_col(rm, ['mwu_yo_p'], None)
            mwu_p = rm[mwu_col].values[0] if (len(rm) and mwu_col is not None) else None
            if not sub.empty:
                sub = sub.rename(columns={'mean_strength': 'value'})
                plot_violin_strip(sub, title=pair[:18], mwu_p=mwu_p, ax=ax)
        for ax in axes3d[len(top_violin):]:
            ax.axis('off')
        plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
        save_fig(fig3d, 'FigureS3D_fine_grouped_violin_validation', FIG_DIR)
        plt.show()

    # 3E: Ordered pattern summary. Horizontal bars are easier to read than pie charts.
    pattern_col = 'ordered_curve_pattern' if 'ordered_curve_pattern' in fm_s.columns else 'curve_pattern'
    if pattern_col in fm_s.columns:
        cp_c = (fm_s[pattern_col].value_counts()
                .rename_axis('ordered_curve_pattern')
                .reset_index(name='n_fine_pairs'))
        if not cp_c.empty:
            cp_c.to_csv(TAB_DIR / 'S4_fine_ordered_curve_pattern_counts.tsv', sep='\t', index=False)
            cp_plot = cp_c.sort_values('n_fine_pairs', ascending=True).copy()
            fig3e, ax3e = plt.subplots(figsize=(6.8, max(3.8, 0.36 * len(cp_plot) + 1.2)))
            colors = [CURVE_PAL.get(p, '#7A9BB8') for p in cp_plot['ordered_curve_pattern']]
            ax3e.barh(cp_plot['ordered_curve_pattern'], cp_plot['n_fine_pairs'],
                      color=colors, edgecolor='white', linewidth=0.6)
            for y, v in enumerate(cp_plot['n_fine_pairs']):
                ax3e.text(v + 0.05, y, str(int(v)), va='center', ha='left', fontsize=12)
            ax3e.set_xlabel('Number of fine cell pairs')
            ax3e.set_ylabel('Ordered age-bin pattern')
            ax3e.set_title('3E Ordered age-pattern distribution', fontweight='bold')
            ax3e.spines['top'].set_visible(False)
            ax3e.spines['right'].set_visible(False)
            plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
            save_fig(fig3e, 'Figure3E_ordered_curve_pattern_bar', FIG_DIR)
            plt.show()

    print('Figure 3 saved')
else:
    print('No fine_models for Figure 3')


## Figure 4 - CPDB Classification Continuous Age Information Flow

Model CPDB `classification` categories by continuous age. Directionality and donor prevalence help separate stable communication categories from local or stage-biased signals.

 Adds CPDB classification enrichment for LR drivers as a functional interpretation layer.


In [ ]:


def plot_classification_information_flow(classification_panel, title, outfile, value_col='sum_strength',
                                         top_n=18):
    """CellChat-rankNet-inspired pathway/classification information-flow comparison."""
    if classification_panel is None or classification_panel.empty:
        return None
    required = {'sample_id', 'age_group', 'classification', value_col}
    if not required.issubset(classification_panel.columns):
        return None
    df = classification_panel[classification_panel['age_group'].isin(['Young adult', 'Adult', 'Elderly'])].copy()
    if df.empty:
        return None
    donor_cls = (df.groupby(['sample_id','age_group','classification'], observed=True)[value_col]
                   .sum().reset_index())
    means = donor_cls.pivot_table(index='classification', columns='age_group',
                                  values=value_col, aggfunc='mean', fill_value=0.0)
    for g in ['Young adult', 'Adult', 'Elderly']:
        if g not in means.columns:
            means[g] = 0.0
    stats = means.reset_index()
    stats['delta_old_young'] = stats['Elderly'] - stats['Young adult']
    stats['ratio_old_young'] = (stats['Elderly'] + 1e-9) / (stats['Young adult'] + 1e-9)
    pvals = []
    for cls in stats['classification']:
        y = donor_cls[(donor_cls['classification'] == cls) & (donor_cls['age_group'] == 'Young adult')][value_col]
        o = donor_cls[(donor_cls['classification'] == cls) & (donor_cls['age_group'] == 'Elderly')][value_col]
        if len(y) >= 3 and len(o) >= 3 and (y.nunique() > 1 or o.nunique() > 1):
            try:
                pvals.append(st.mannwhitneyu(y, o, alternative='two-sided').pvalue)
            except Exception:
                pvals.append(np.nan)
        else:
            pvals.append(np.nan)
    stats['young_old_mwu_p'] = pvals
    stats['abs_delta'] = stats['delta_old_young'].abs()
    stats = stats.sort_values('abs_delta', ascending=False)
    stats.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plot_df = stats.head(top_n).sort_values('delta_old_young')
    if plot_df.empty:
        return None
    colors = np.where(plot_df['delta_old_young'] >= 0, UP_COLOR, DOWN_COLOR)
    fig, ax = plt.subplots(figsize=(7.2, max(4.0, 0.28 * len(plot_df))))
    ax.barh(plot_df['classification'], plot_df['delta_old_young'], color=colors, alpha=0.86)
    ax.axvline(0, color='#333333', lw=1.0)
    for y_pos, (_, row) in enumerate(plot_df.iterrows()):
        pval = row.get('young_old_mwu_p', np.nan)
        if pd.notna(pval):
            label = f"p={pval:.2g}"
            x = row['delta_old_young']
            ax.text(x + (0.015 * plot_df['abs_delta'].max() if x >= 0 else -0.015 * plot_df['abs_delta'].max()),
                    y_pos, label, va='center', ha='left' if x >= 0 else 'right', fontsize=12, color='#4A4A4A')
    ax.set_xlabel('Old - Young mean information flow')
    ax.set_ylabel('CPDB classification')
    ax.set_title(title, fontsize=12)
    ax.grid(axis='x', alpha=0.18)
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def split_lr_genes(lr_pair):
    """Best-effort split of CPDB LR pair names into ligand and receptor gene tokens."""
    if pd.isna(lr_pair):
        return [], []
    text = str(lr_pair)
    if '_' not in text:
        toks = [g for g in re.split(r'[+;&|,\s]+', text) if g]
        return toks, []
    ligand_part, receptor_part = text.split('_', 1)
    ligands = [g for g in re.split(r'[+;&|,\s]+', ligand_part) if g]
    receptors = [g for g in re.split(r'[+;&|,\s]+', receptor_part) if g]
    return ligands, receptors


def build_lr_gene_sets_for_enrichment(lr_df, rho_col='spearman_rho'):
    """Create aging-up/down ligand/receptor gene sets from selected LR drivers."""
    if lr_df is None or lr_df.empty or 'lr_pair' not in lr_df.columns or rho_col not in lr_df.columns:
        return pd.DataFrame()
    records = []
    keep = lr_df.copy()
    if 'discovery_trend_label' in keep.columns:
        disc = keep[keep['discovery_trend_label'].isin(['nominal Spearman trend'])].copy()
        if not disc.empty:
            keep = disc
    for _, row in keep.dropna(subset=['lr_pair', rho_col]).iterrows():
        direction = 'aging_up' if row[rho_col] > 0 else 'aging_down' if row[rho_col] < 0 else 'flat'
        if direction == 'flat':
            continue
        ligands, receptors = split_lr_genes(row['lr_pair'])
        for gene in ligands:
            records.append({'gene_set': f'{direction}_ligands', 'direction': direction,
                            'side': 'ligand', 'gene': gene, 'lr_pair': row['lr_pair'],
                            'entity': row.get('entity', np.nan), 'spearman_rho': row[rho_col]})
        for gene in receptors:
            records.append({'gene_set': f'{direction}_receptors', 'direction': direction,
                            'side': 'receptor', 'gene': gene, 'lr_pair': row['lr_pair'],
                            'entity': row.get('entity', np.nan), 'spearman_rho': row[rho_col]})
    out = pd.DataFrame(records)
    if out.empty:
        return out
    out['gene'] = out['gene'].astype(str).str.upper()
    return out.drop_duplicates(['gene_set','gene']).sort_values(['gene_set','gene'])


def run_enrichr_gene_sets(gene_sets, libraries=None, min_genes=3):
    """Run optional Enrichr enrichment through gseapy when available; never hard-fail the notebook."""
    if gene_sets is None or gene_sets.empty:
        return pd.DataFrame()
    if libraries is None:
        libraries = ['GO_Biological_Process_2023', 'Reactome_2022']
    try:
        import gseapy as gp
    except Exception as exc:
        print('GO/Reactome enrichment skipped: install gseapy on the server to enable Enrichr dotplots.')
        print(f'gseapy import error: {exc}')
        return pd.DataFrame()
    records = []
    for gene_set, sub in gene_sets.groupby('gene_set'):
        genes = sorted(sub['gene'].dropna().unique().tolist())
        if len(genes) < min_genes:
            continue
        for lib in libraries:
            try:
                enr = gp.enrichr(gene_list=genes, gene_sets=lib, organism='human', outdir=None, cutoff=1.0, no_plot=True)
                res = enr.results.copy() if hasattr(enr, 'results') else pd.DataFrame()
                if res.empty:
                    continue
                res['gene_set'] = gene_set
                res['library'] = lib
                res['input_gene_n'] = len(genes)
                records.append(res)
            except Exception as exc:
                print(f'Enrichr skipped for {gene_set} / {lib}: {exc}')
    if not records:
        return pd.DataFrame()
    out = pd.concat(records, ignore_index=True)
    # Normalize common gseapy column names across versions.
    rename_map = {
        'P-value': 'p_value',
        'Adjusted P-value': 'adjusted_p',
        'Overlap': 'overlap',
        'Term': 'term',
        'Genes': 'genes',
    }
    out = out.rename(columns={k: v for k, v in rename_map.items() if k in out.columns})
    return out


def plot_clean_enrichment_dotplot(enrich_df, title, outfile, top_n=8):
    """Nature-style split-panel GO/Reactome enrichment dotplot.

    Top panel: aging-up LR genes. Bottom panel: aging-down LR genes.
    X-axis is -log10(adjusted p); dot size is overlap gene count; marker shape is ligand/receptor.
    """
    if enrich_df is None or enrich_df.empty:
        return None
    df = enrich_df.copy()
    p_col = 'adjusted_p' if 'adjusted_p' in df.columns else 'p_value'
    if p_col not in df.columns or 'term' not in df.columns:
        return None
    df[p_col] = pd.to_numeric(df[p_col], errors='coerce')
    df = df.dropna(subset=[p_col]).copy()

    def overlap_count(x):
        try:
            return int(str(x).split('/')[0])
        except Exception:
            return np.nan

    def parse_gene_set(gs):
        text = str(gs).lower()
        direction = 'aging-up' if ('aging_up' in text or 'up_' in text) else ('aging-down' if ('aging_down' in text or 'down_' in text) else np.nan)
        side = 'ligand' if 'ligand' in text else ('receptor' if 'receptor' in text else 'LR gene')
        return pd.Series({'direction_label': direction, 'gene_side': side})

    parsed = df['gene_set'].apply(parse_gene_set) if 'gene_set' in df.columns else pd.DataFrame(index=df.index)
    df = pd.concat([df, parsed], axis=1)
    df = df[df['direction_label'].isin(['aging-up', 'aging-down'])].copy()
    if df.empty:
        return None
    df['gene_side'] = df['gene_side'].fillna('LR gene')
    df['overlap_n'] = df['overlap'].map(overlap_count) if 'overlap' in df.columns else np.nan
    if df['overlap_n'].isna().all() and 'genes' in df.columns:
        df['overlap_n'] = df['genes'].fillna('').astype(str).map(lambda x: len([g for g in re.split(r'[;,]', x) if g.strip()]))
    df['overlap_n'] = df['overlap_n'].fillna(1).clip(lower=1)
    df['neg_log10_adjusted_p'] = -np.log10(df[p_col].clip(lower=1e-300))
    df['neg_log10_p'] = df['neg_log10_adjusted_p']
    df['term_clean'] = (df['term'].astype(str)
                        .str.replace(r'\s+', ' ', regex=True)
                        .str.replace(r'\s*\(GO:\d+\)', '', regex=True)
                        .str.slice(0, 78))

    panels = [('aging-up', UP_COLOR, 'Aging-up LR genes'),
              ('aging-down', DOWN_COLOR, 'Aging-down LR genes')]
    fig, axes = plt.subplots(2, 1, figsize=(7.2, 7.8), sharex=True,
                             gridspec_kw={'height_ratios': [1, 1], 'hspace': 0.38})
    marker_map = {'ligand': 'o', 'receptor': '^', 'LR gene': 's'}
    for ax, (direction, color, label) in zip(axes, panels):
        sub = df[df['direction_label'] == direction].copy()
        pieces = []
        for side, side_df in sub.sort_values(p_col).groupby('gene_side', sort=False):
            pieces.append(side_df.head(top_n))
        if pieces:
            sub = pd.concat(pieces, ignore_index=True)
        else:
            sub = sub.head(0)
        sub = sub.sort_values(p_col, ascending=False).reset_index(drop=True)
        y = np.arange(len(sub))
        for side, side_df in sub.groupby('gene_side', sort=False):
            idx = side_df.index.to_numpy()
            ax.scatter(side_df['neg_log10_p'], y[idx],
                       s=np.clip(side_df['overlap_n'].astype(float) * 34, 42, 300),
                       c=color, marker=marker_map.get(side, 'o'),
                       edgecolors='white', linewidths=0.65, alpha=0.88, label=side)
        ax.axvline(-np.log10(0.05), color='#999999', lw=1.0, ls=':', zorder=0)
        ax.set_yticks(y)
        ax.set_yticklabels(sub['term_clean'], fontsize=12)
        ax.set_title(label, loc='left', color=color, fontweight='bold', fontsize=12)
        ax.grid(axis='x', alpha=0.14)
        ax.set_ylabel('')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        if len(sub) == 0:
            ax.text(0.5, 0.5, 'No enriched terms', ha='center', va='center',
                    transform=ax.transAxes, color='#777777', fontsize=12)
    axes[-1].set_xlabel(r'$-\log_{10}$(adjusted $P$)')
    handles, labels = axes[0].get_legend_handles_labels()
    if not handles:
        handles, labels = axes[1].get_legend_handles_labels()
    if handles:
        axes[0].legend(handles, labels, frameon=False, loc='lower right', title='Gene side')
    fig.suptitle(title, y=0.995, fontsize=12, fontweight='bold')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def plot_enrichment_dotplot(enrich_df, title, outfile, top_n=8):
    return plot_clean_enrichment_dotplot(enrich_df, title, outfile, top_n=top_n)


def plot_cpdb_directional_fallback_dotplot(lr_df, title, outfile, top_n=16):
    """Fallback functional dotplot using CPDB classification when GO/Reactome enrichment is empty."""
    if lr_df is None or lr_df.empty or 'classification' not in lr_df.columns or 'spearman_rho' not in lr_df.columns:
        return None
    df = lr_df.dropna(subset=['classification', 'spearman_rho']).copy()
    if df.empty:
        return None
    df['direction_label'] = np.where(df['spearman_rho'] >= 0, 'aging-up', 'aging-down')
    summ = (df.groupby(['classification', 'direction_label'])
              .agg(n_lr=('lr_pair', 'nunique') if 'lr_pair' in df.columns else ('entity', 'nunique'),
                   mean_rho=('spearman_rho', 'mean'))
              .reset_index())
    summ['signed_score'] = summ['mean_rho']
    summ['abs_mean_rho'] = summ['mean_rho'].abs()
    summ = summ.sort_values(['n_lr', 'abs_mean_rho'], ascending=[False, False]).head(top_n)
    summ = summ.sort_values('signed_score').reset_index(drop=True)
    if summ.empty:
        return None
    vmax = max(float(summ['signed_score'].abs().max()), 0.1)
    norm = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    cmap = mpl.cm.get_cmap('RdBu_r')
    fig, ax = plt.subplots(figsize=(8.6, max(4.7, 0.34 * len(summ) + 1.0)))
    y = np.arange(len(summ))
    ax.scatter(summ['signed_score'], y,
               s=np.clip(summ['n_lr'].astype(float) * 35, 55, 320),
               c=summ['signed_score'], cmap=cmap, norm=norm,
               edgecolors='white', linewidths=0.7, alpha=0.92)
    ax.axvline(0, color='#333333', lw=1.0, ls='--')
    ax.set_yticks(y)
    ax.set_yticklabels(summ['classification'], fontsize=12)
    ax.set_xlabel('Mean Spearman rho among LR drivers')
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0.15)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = fig.colorbar(sm, ax=ax, pad=0.015)
    cbar.set_label('Mean Spearman rho')
    ax.text(0.0, -0.10, 'Fallback: CPDB communication categories when GO/Reactome enrichment is unavailable.',
            transform=ax.transAxes, ha='left', va='top', fontsize=12, color='#555555')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def cpdb_classification_enrichment(selected_df, background_df, class_col='classification', entity_col='entity'):
    """Fisher enrichment of CPDB classification among displayed/discovery LR drivers."""
    if selected_df is None or background_df is None or selected_df.empty or background_df.empty:
        return pd.DataFrame()
    bg_cols = [entity_col, class_col]
    if not set(bg_cols).issubset(background_df.columns) or not set(bg_cols).issubset(selected_df.columns):
        return pd.DataFrame()
    bg = background_df[bg_cols].dropna().drop_duplicates()
    sel = selected_df[bg_cols + [c for c in ['spearman_rho','discovery_trend_label','cohort_sensitivity_label'] if c in selected_df.columns]].dropna(subset=bg_cols).drop_duplicates(bg_cols)
    if bg.empty or sel.empty:
        return pd.DataFrame()
    bg_entities = set(bg[entity_col])
    sel = sel[sel[entity_col].isin(bg_entities)].copy()
    if sel.empty:
        return pd.DataFrame()
    n_bg = bg[entity_col].nunique()
    n_sel = sel[entity_col].nunique()
    records = []
    for cls, bg_cls in bg.groupby(class_col):
        bg_entities_cls = set(bg_cls[entity_col])
        sel_entities_cls = set(sel.loc[sel[class_col] == cls, entity_col])
        a = len(sel_entities_cls)
        if a == 0:
            continue
        b = n_sel - a
        c = len(bg_entities_cls - sel_entities_cls)
        d = max(n_bg - len(bg_entities_cls) - b, 0)
        odds, pval = st.fisher_exact([[a, b], [c, d]], alternative='greater')
        rho_vals = sel.loc[sel[class_col] == cls, 'spearman_rho'] if 'spearman_rho' in sel.columns else pd.Series(dtype=float)
        records.append({
            class_col: cls,
            'selected_lr_n': a,
            'background_lr_n': len(bg_entities_cls),
            'selected_total_n': n_sel,
            'background_total_n': n_bg,
            'selected_fraction': a / max(n_sel, 1),
            'background_fraction': len(bg_entities_cls) / max(n_bg, 1),
            'odds_ratio': odds,
            'fisher_p': pval,
            'mean_spearman_rho': rho_vals.mean() if len(rho_vals) else np.nan,
        })
    out = pd.DataFrame(records)
    if out.empty:
        return out
    out['odds_ratio_plot'] = out['odds_ratio'].replace([np.inf, -np.inf], np.nan)
    finite_max = out['odds_ratio_plot'].replace([np.inf, -np.inf], np.nan).max()
    out['odds_ratio_plot'] = out['odds_ratio_plot'].fillna((finite_max if pd.notna(finite_max) else 1.0) * 1.25)
    return out.sort_values(['fisher_p', 'selected_lr_n'], ascending=[True, False])


def plot_classification_enrichment_dotplot(enrich_df, title, outfile, top_n=15):
    if enrich_df is None or enrich_df.empty:
        return None
    df = enrich_df.head(top_n).iloc[::-1].copy()
    fig, ax = plt.subplots(figsize=(8.5, max(4.8, 0.36 * len(df))))
    sizes = np.clip(df['selected_lr_n'].astype(float).values * 28, 45, 360)
    colors = df['mean_spearman_rho'].fillna(0)
    sca = ax.scatter(df['odds_ratio_plot'], np.arange(len(df)), s=sizes, c=colors,
                     cmap='RdBu_r', vmin=-max(abs(colors.min()), abs(colors.max()), 0.1),
                     vmax=max(abs(colors.min()), abs(colors.max()), 0.1),
                     edgecolors='white', linewidths=0.7, alpha=0.88)
    ax.axvline(1, ls='--', lw=1.0, color='#555555')
    ax.set_yticks(np.arange(len(df)))
    ax.set_yticklabels(df['classification'])
    ax.set_xlabel('Fisher odds ratio among selected LR drivers')
    ax.set_title(title)
    cbar = fig.colorbar(sca, ax=ax, pad=0.02)
    cbar.set_label('Mean Spearman rho')
    for i, row in enumerate(df.itertuples(index=False)):
        ax.text(row.odds_ratio_plot, i, f'  n={int(row.selected_lr_n)}, p={row.fisher_p:.2g}',
                va='center', fontsize=12, color='#333333')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig

set_pub_style()

if not class_models.empty and not classification_donor_df.empty:
    cm_s = class_models[class_models['metric'] == 'mean_strength'].copy()
    pc4 = choose_trend_pcol(cm_s)
    cm_s = add_interpretation_annotations(cm_s, layer='classification')
    top_cls_main = select_main_text_entities(cm_s, 'classification', top_n=6, layer='classification', allow_tier2=True)
    cm_bal_source = cm_s[cm_s['classification'].isin(top_cls_main)].copy() if top_cls_main else cm_s.copy()
    top_cls = select_balanced_visual_entities(cm_bal_source, 'classification', top_n=6, min_up=3, min_down=3, label='Figure 4 classifications')
    if not top_cls:
        top_cls = top_cls_main[:6]
    save_visual_selection_table(cm_s, 'classification', top_cls, 'Figure4_balanced_classification_selection')

    # 4A: Classification continuous age trajectories
    plot_trend_grid(classification_donor_df, cm_s, 'classification', 'mean_strength', top_cls[:6],
                    '4A Classification mean-strength continuous-age trajectories',
                    'Figure4A_classification_mean_strength_trajectories', ncols=3)

    # 4B: Classification donor-ordered heatmap
    class_label_map = {}
    if top_cls:
        class_meta = cm_s.drop_duplicates('classification').set_index('classification')
        class_label_map = {cls: f"{cls} ({int(class_meta.loc[cls, 'n_nonzero_donors'])}/{int(class_meta.loc[cls, 'n_donors_modeled'])})"
                           for cls in top_cls if cls in class_meta.index}
    donor_ordered_heatmap(classification_donor_df, 'classification', 'mean_strength', top_cls,
                          '4B Classification mean-strength donor-ordered aging heatmap (row z-score clipped to [-2, 3])',
                          'Figure4B_classification_mean_strength_donor_ordered_heatmap', figsize=(13, max(5, 0.40*len(top_cls))),
                          row_label_map=class_label_map, clip_z=(-2, 3),
                          caption='Row z-scores were clipped to [-2, 3] for visualization only. Classification values show mean FastCCC strength among significant LR interactions; summed category burden is retained only as an auxiliary interpretation.')

    # 4C: Classification x directionality beta heatmap
    if not fastccc_long.empty:
        cls_dir = (fastccc_long.groupby(['sample_id','classification','directionality'])
                   .agg(sum_strength=('strength','sum'))
                   .reset_index())
        if not cls_dir.empty:
            cls_dir['entity'] = cls_dir['classification'].astype(str) + '::' + cls_dir['directionality'].astype(str)
            entity_meta = cls_dir[['entity','classification','directionality']].drop_duplicates()
            full_idx = pd.MultiIndex.from_product(
                [donor_base['sample_id'].unique(), entity_meta['entity'].values],
                names=['sample_id','entity']
            ).to_frame(index=False)
            cls_dir_full = (full_idx
                            .merge(entity_meta, on='entity', how='left')
                            .merge(cls_dir[['sample_id','entity','sum_strength']], on=['sample_id','entity'], how='left')
                            .merge(donor_base[['sample_id','age','age_group','cohort_group','log_total_cells']],
                                   on='sample_id', how='left'))
            cls_dir_full['sum_strength'] = cls_dir_full['sum_strength'].fillna(0.0)
            cls_dir_full['sender_fraction'] = 0.0
            cls_dir_full['receiver_fraction'] = 0.0
            cls_dir_models = dual_track(cls_dir_full, 'entity', metrics=['sum_strength'])
            if not cls_dir_models.empty:
                cls_dir_models[['classification','directionality']] = (
                    cls_dir_models['entity'].str.split('::', expand=True).iloc[:, :2])
                cls_dir_models.to_csv(TAB_DIR / 'S6b_classification_directionality_models.tsv', sep='\t', index=False)
                hm_df = visual_candidate_df(cls_dir_models[
                    (cls_dir_models['metric'] == 'sum_strength') &
                    (cls_dir_models['classification'].isin(top_cls))
                ].copy(), min_nonzero=8, min_sd=0.0)
                if not hm_df.empty:
                    hm = hm_df.pivot_table(index='classification', columns='directionality', values='spearman_rho', aggfunc='mean')
                    hm = hm.reindex(top_cls)
                    fig4c, ax4c = plt.subplots(figsize=(8, max(5, 0.35 * len(hm))))
                    sns.heatmap(hm, cmap='RdBu_r', center=0, ax=ax4c,
                                cbar_kws={'label': 'Spearman rho'},
                                linewidths=0.6, linecolor='white')
                    ax4c.set_title('4C Classification x directionality Spearman heatmap')
                    ax4c.set_xlabel('Directionality')
                    ax4c.set_ylabel('Classification')
                    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
                    save_fig(fig4c, 'Figure4C_classification_directionality_heatmap', FIG_DIR)
                    plt.show()

    # 4D: Compact Spearman lollipop, replacing overcrowded annotated scatter
    cls_plot = cm_s[cm_s['classification'].isin(top_cls)].copy()
    if not cls_plot.empty:
        plot_spearman_lollipop(cls_plot, 'classification', top_cls,
                               '4D Top classification Spearman age trends',
                               'Figure4D_classification_spearman_lollipop',
                               figsize=(8, max(5, 0.34*len(cls_plot))))


    # 4E: Functional interpretation of LR drivers by CPDB classification enrichment
    lr_driver_source = selected_lr_discovery.copy() if 'selected_lr_discovery' in globals() and selected_lr_discovery is not None and not selected_lr_discovery.empty else (selected_lr.copy() if 'selected_lr' in globals() and selected_lr is not None and not selected_lr.empty else pd.DataFrame())
    lr_background_source = lr_models_raw.copy() if 'lr_models_raw' in globals() and lr_models_raw is not None and not lr_models_raw.empty else pd.DataFrame()
    if not lr_driver_source.empty and not lr_background_source.empty:
        lr_driver_for_enrich = lr_driver_source.copy()
        if 'discovery_trend_label' in lr_driver_for_enrich.columns:
            disc = lr_driver_for_enrich[lr_driver_for_enrich['discovery_trend_label'].isin(['nominal Spearman trend'])].copy()
            if not disc.empty:
                lr_driver_for_enrich = disc
        lr_class_enrichment = cpdb_classification_enrichment(lr_driver_for_enrich, lr_background_source)
        if not lr_class_enrichment.empty:
            lr_class_enrichment.to_csv(TAB_DIR / 'S7_lr_driver_cpdb_classification_enrichment.tsv', sep='\t', index=False)
            plot_classification_enrichment_dotplot(
                lr_class_enrichment,
                '4E CPDB classification enrichment among selected LR drivers',
                'Figure4E_LR_driver_CPDB_classification_enrichment',
                top_n=15)
            print(f'LR driver classification enrichment: {len(lr_class_enrichment)} classes')


    # 4F: Optional GO/Reactome enrichment of aging-up/down LR genes.
    # Requires gseapy on the server; without it, the LR gene-set table is still saved.
    if not lr_driver_source.empty:
        lr_gene_sets = build_lr_gene_sets_for_enrichment(lr_driver_source)
        if not lr_gene_sets.empty:
            lr_gene_sets.to_csv(TAB_DIR / 'S7_lr_driver_aging_direction_gene_sets.tsv', sep='\t', index=False)
            lr_gene_enrichment = run_enrichr_gene_sets(lr_gene_sets)
            if not lr_gene_enrichment.empty:
                lr_gene_enrichment.to_csv(TAB_DIR / 'S7_lr_driver_GO_Reactome_enrichment.tsv', sep='\t', index=False)
                plot_clean_enrichment_dotplot(
                    lr_gene_enrichment,
                    '4F Directional GO/Reactome enrichment of LR genes',
                    'Figure4F_GO_Reactome_directional_enrichment',
                    top_n=8)
            else:
                print('LR driver gene sets saved; GO/Reactome enrichment empty; drawing CPDB directional fallback.')
                plot_cpdb_directional_fallback_dotplot(
                    lr_driver_source,
                    '4F CPDB communication categories by aging direction',
                    'Figure4F_CPDB_directional_fallback',
                    top_n=16)
        else:
            print('LR driver gene set table empty; drawing CPDB directional fallback.')
            plot_cpdb_directional_fallback_dotplot(
                lr_driver_source,
                '4F CPDB communication categories by aging direction',
                'Figure4F_CPDB_directional_fallback',
                top_n=16)



    # 4G: Pathway/classification information-flow comparison, inspired by CellChat rankNet.
    plot_classification_information_flow(
        classification_donor_df,
        'S4G Supplementary grouped CPDB classification information flow: Old versus Young',
        'FigureS4G_classification_information_flow_old_vs_young',
        value_col='sum_strength',
        top_n=18)

    print('Figure 4 saved')
else:
    print('No classification_models for Figure 4')


## Figure 4/5 - Continuous LR Drivers, Enrichment, and Grouped Companion LRI Visualization

Continuous LR driver plots use `selected_lr_continuous` from Stage 7A. FastCCC Fig.4h-style Young adult/Adult/Elderly grouped LRI matrices are shown afterward as companion visualizations and do not participate in continuous-age filtering.


In [ ]:


def summarize_age_group_lri_sets(sig_long, selected_entities=None, min_donor_frac=0.10):
    """Summarize Young adult/Adult/Elderly LRI presence sets, adapted from FastCCC group-overlap application."""
    if sig_long is None or sig_long.empty:
        return pd.DataFrame(), pd.DataFrame()
    df = sig_long.copy()
    if 'entity' not in df.columns:
        df['entity'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    if selected_entities:
        df = df[df['entity'].isin(selected_entities)].copy()
    df = df.merge(donor_base[['sample_id','age_group']], on='sample_id', how='left', suffixes=('', '_donor'))
    if 'age_group_donor' in df.columns:
        df['age_group'] = df['age_group'].fillna(df['age_group_donor']) if 'age_group' in df.columns else df['age_group_donor']
    df = df[df['age_group'].isin(AGE_ORDER)].copy()
    if df.empty:
        return pd.DataFrame(), pd.DataFrame()
    group_n = donor_base[donor_base['age_group'].isin(AGE_ORDER)].groupby('age_group')['sample_id'].nunique().to_dict()
    support = (df.groupby(['age_group','entity','cellpair','lr_pair'], observed=True)
               .agg(donor_n=('sample_id','nunique'), mean_strength=('strength','mean'), sum_strength=('strength','sum'))
               .reset_index())
    support['group_donor_n'] = support['age_group'].map(group_n)
    support['donor_frac'] = support['donor_n'] / support['group_donor_n'].replace(0, np.nan)
    support['present'] = support['donor_frac'] >= min_donor_frac
    present = support[support['present']].copy()
    if present.empty:
        return support, pd.DataFrame()
    membership = (present.pivot_table(index=['entity','cellpair','lr_pair'], columns='age_group', values='present', aggfunc='max', fill_value=False)
                  .reset_index())
    for g in AGE_ORDER:
        if g not in membership.columns:
            membership[g] = False
    membership['group_pattern'] = membership.apply(
        lambda r: '&'.join([g for g in AGE_ORDER if bool(r[g])]) or 'none', axis=1)
    membership['elderly_specific'] = membership['Elderly'] & ~membership['Young adult'] & ~membership['Adult']
    membership['young_adult_specific'] = membership['Young adult'] & ~membership['Adult'] & ~membership['Elderly']
    membership['aging_gain'] = membership['Elderly'] & ~membership['Young adult']
    membership['aging_loss'] = membership['Young adult'] & ~membership['Elderly']
    return support, membership


def plot_age_group_lri_overlap(membership, title, outfile):
    if membership is None or membership.empty:
        return None
    counts = membership['group_pattern'].value_counts().reset_index()
    counts.columns = ['group_pattern', 'n_lri']
    counts['group_pattern'] = pd.Categorical(
        counts['group_pattern'],
        categories=['Young adult', 'Adult', 'Elderly', 'Young adult&Adult', 'Young adult&Elderly', 'Adult&Elderly', 'Young adult&Adult&Elderly'],
        ordered=True)
    counts = counts.sort_values(['group_pattern', 'n_lri'], ascending=[True, False]).dropna(subset=['group_pattern'])
    if counts.empty:
        return None
    fig, ax = plt.subplots(figsize=(8.6, 4.8))
    colors = []
    for pat in counts['group_pattern'].astype(str):
        if pat == 'Elderly': colors.append('#C84E4E')
        elif pat == 'Young adult': colors.append('#3B74AF')
        elif pat == 'Adult': colors.append('#D39C32')
        else: colors.append('#7A7A7A')
    ax.bar(counts['group_pattern'].astype(str), counts['n_lri'], color=colors, edgecolor='white')
    ax.set_xlabel('Age-group LRI membership')
    ax.set_ylabel('N LRIs')
    ax.set_title(title)
    ax.set_xticklabels(counts['group_pattern'].astype(str), rotation=35, ha='right')
    for i, v in enumerate(counts['n_lri']):
        ax.text(i, v, str(int(v)), ha='center', va='bottom', fontsize=12)
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def plot_specific_lri_network(membership, mode='elderly_specific', title=None, outfile=None, top_n_edges=28):
    if membership is None or membership.empty or mode not in membership.columns:
        return None
    sub = membership[membership[mode]].copy()
    if sub.empty:
        return None
    edge_counts = (sub.groupby('cellpair')['lr_pair'].nunique()
                   .sort_values(ascending=False).head(top_n_edges).reset_index(name='n_lri'))
    if edge_counts.empty:
        return None
    parts = edge_counts['cellpair'].astype(str).str.split('|', n=1, expand=True)
    edge_counts['sender'] = parts[0]
    edge_counts['receiver'] = parts[1] if parts.shape[1] > 1 else parts[0]
    G = nx.DiGraph()
    for _, row in edge_counts.iterrows():
        G.add_edge(row['sender'], row['receiver'], weight=float(row['n_lri']))
    pos = nx.circular_layout(G)
    fig, ax = plt.subplots(figsize=(7.2, 6.8))
    weights = np.array([G[u][v]['weight'] for u, v in G.edges()])
    nx.draw_networkx_nodes(G, pos, node_size=850, node_color='#F2F2F2', edgecolors='#333333', linewidths=2.3, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
    nx.draw_networkx_edges(G, pos, width=np.clip(weights / max(weights.max(), 1) * 6, 0.8, 6),
                           arrows=True, arrowstyle='-|>', arrowsize=12,
                           edge_color='#C84E4E' if 'old' in mode else '#3B74AF', alpha=0.72, ax=ax,
                           connectionstyle='arc3,rad=0.12')
    ax.set_title(title or mode)
    ax.axis('off')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    if outfile:
        save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def load_top_lr_expression_panel(entities, top_n=8):
    """Read h5ad files only for selected LR genes and sender/receiver cell types."""
    if not entities:
        return pd.DataFrame()
    ent_meta = []
    source_df = selected_lr_continuous if 'selected_lr_continuous' in globals() and selected_lr_continuous is not None and not selected_lr_continuous.empty else (selected_lr_discovery if 'selected_lr_discovery' in globals() and selected_lr_discovery is not None and not selected_lr_discovery.empty else (selected_lr if 'selected_lr' in globals() and selected_lr is not None and not selected_lr.empty else pd.DataFrame()))
    if source_df.empty:
        return pd.DataFrame()
    source_df = source_df[source_df['entity'].isin(entities)].drop_duplicates('entity').head(top_n)
    for _, row in source_df.iterrows():
        cellpair = str(row['cellpair'])
        parts = cellpair.split('|')
        sender = parts[0].strip()
        receiver = parts[1].strip() if len(parts) > 1 else sender
        ligands, receptors = split_lr_genes(row['lr_pair']) if 'split_lr_genes' in globals() else ([], [])
        for g in ligands[:3]:
            ent_meta.append({'entity': row['entity'], 'lr_pair': row['lr_pair'], 'cellpair': cellpair,
                             'celltype': sender, 'gene': g.upper(), 'side': 'ligand'})
        for g in receptors[:4]:
            ent_meta.append({'entity': row['entity'], 'lr_pair': row['lr_pair'], 'cellpair': cellpair,
                             'celltype': receiver, 'gene': g.upper(), 'side': 'receptor'})
    meta = pd.DataFrame(ent_meta).drop_duplicates()
    if meta.empty:
        return pd.DataFrame()
    wanted_genes = set(meta['gene'])
    wanted_ct = set(meta['celltype'])
    records = []
    for _, sample in samples.iterrows():
        sid = str(sample['sample_id'])
        try:
            adata = sc.read_h5ad(sample['h5ad_path'])
            if META_KEY not in adata.obs.columns:
                continue
            obs_ct = adata.obs[META_KEY].astype(str).str.strip()
            var_upper = pd.Index(adata.var_names.astype(str).str.upper())
            gene_map = {g: adata.var_names[var_upper.get_loc(g)] for g in wanted_genes if g in set(var_upper)}
            if not gene_map:
                del adata
                continue
            for _, m in meta.iterrows():
                if m['gene'] not in gene_map:
                    continue
                mask = (obs_ct == m['celltype']).values
                if mask.sum() == 0:
                    continue
                vals = adata[mask, gene_map[m['gene']]].X
                if hasattr(vals, 'toarray'):
                    vals = vals.toarray()
                vals = np.asarray(vals).ravel()
                records.append({
                    'sample_id': sid, 'age': sample['age'], 'age_group': sample['age_group'],
                    'entity': m['entity'], 'lr_pair': m['lr_pair'], 'cellpair': m['cellpair'],
                    'celltype': m['celltype'], 'gene': m['gene'], 'side': m['side'],
                    'mean_expression': float(np.mean(vals)),
                    'pct_expressing': float((vals > 0).mean()),
                    'n_cells': int(mask.sum()),
                })
            del adata
            gc.collect()
        except Exception as exc:
            print(f'Expression validation skipped for {sid}: {exc}')
    out = pd.DataFrame(records)
    if not out.empty:
        out.to_csv(TAB_DIR / 'S7_lr_driver_ligand_receptor_expression_validation.tsv', sep='\t', index=False)
    return out


def plot_lr_expression_validation(expr_df, title, outfile, max_panels=8):
    if expr_df is None or expr_df.empty:
        return None
    expr_df = expr_df.copy()
    expr_df['label'] = expr_df['gene'] + ' in ' + expr_df['celltype'].astype(str) + ' (' + expr_df['side'] + ')'
    labels = expr_df['label'].drop_duplicates().head(max_panels).tolist()
    n = len(labels)
    if n == 0:
        return None
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4*ncols, 4.25*nrows), squeeze=False)
    axes = axes.flatten()
    for ax, label in zip(axes, labels):
        sub = expr_df[expr_df['label'] == label]
        plot_loess_trajectory(sub['age'], sub['mean_expression'], sub['age_group'],
                              label=label[:34], ylabel='Mean expression', n_boot=100, ax=ax)
    for ax in axes[n:]:
        ax.axis('off')
    fig.suptitle(title, y=1.02, fontweight='bold')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def plot_fastccc_condition_circleplot(sig_long, selected_entities, title, outfile, top_n_lr=24, top_n_cellpairs=26):
    """FastCCC paper-style condition circle plot, faceted by Young adult/Adult/Elderly.

    This follows the paper's application logic more closely than offsetting all groups in one
    axis: each age group gets the same cellpair x LRI matrix. Dot size shows group support
    (-log10 min FastCCC p when available, otherwise donor occupancy), and color shows
    group mean strength. Red row labels indicate Elderly-specific LRIs; blue row labels indicate
    Young adult-specific LRIs under the selected donor-support threshold.
    """
    if sig_long is None or sig_long.empty or not selected_entities:
        return None
    df = sig_long.copy()
    if 'entity' not in df.columns:
        df['entity'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    df = df[df['entity'].isin(selected_entities)].copy()
    if df.empty:
        return None
    df = df.merge(donor_base[['sample_id','age_group']], on='sample_id', how='left', suffixes=('', '_donor'))
    if 'age_group_donor' in df.columns:
        df['age_group'] = df['age_group'].fillna(df['age_group_donor']) if 'age_group' in df.columns else df['age_group_donor']
    df = df[df['age_group'].isin(AGE_ORDER)].copy()
    if df.empty:
        return None
    p_candidates = ['p_value', 'pvalue', 'p_val', 'p', 'pval', 'significant_p']
    p_col = first_existing_col(df, p_candidates, None)
    agg_dict = {
        'mean_strength': ('strength', 'mean'),
        'sum_strength': ('strength', 'sum'),
        'donor_n': ('sample_id', 'nunique'),
    }
    if p_col is not None:
        agg_dict['min_p'] = (p_col, 'min')
    grp = (df.groupby(['age_group','cellpair','lr_pair'], observed=True)
           .agg(**agg_dict)
           .reset_index())
    group_donor_n = donor_base[donor_base['age_group'].isin(AGE_ORDER)].groupby('age_group')['sample_id'].nunique().to_dict()
    grp['donor_total'] = grp['age_group'].map(group_donor_n).fillna(grp['donor_n'])
    grp['donor_frac'] = grp['donor_n'] / grp['donor_total'].replace(0, np.nan)
    if p_col is not None:
        grp['support_score'] = -np.log10(grp['min_p'].clip(lower=1e-300))
        size_label = '-log10(min FastCCC p)'
    else:
        grp['support_score'] = grp['donor_frac']
        size_label = 'donor occupancy'

    # Select rows/columns by support, but keep the same matrix across facets.
    lr_rank = (grp.groupby('lr_pair')['support_score'].max()
               .sort_values(ascending=False).head(top_n_lr).index.tolist())
    cellpair_rank = (grp[grp['lr_pair'].isin(lr_rank)]
                     .groupby('cellpair')['support_score'].max()
                     .sort_values(ascending=False).head(top_n_cellpairs).index.tolist())
    plot_df = grp[grp['lr_pair'].isin(lr_rank) & grp['cellpair'].isin(cellpair_rank)].copy()
    if plot_df.empty:
        return None

    # Determine specificity at the displayed matrix level.
    membership = (plot_df.assign(present=plot_df['donor_frac'] >= 0.10)
                  .pivot_table(index='lr_pair', columns='age_group', values='present', aggfunc='max', fill_value=False))
    for g in AGE_ORDER:
        if g not in membership.columns:
            membership[g] = False
    elderly_specific = set(membership.index[membership['Elderly'] & ~membership['Young adult'] & ~membership['Adult']])
    young_adult_specific = set(membership.index[membership['Young adult'] & ~membership['Adult'] & ~membership['Elderly']])

    lr_order = lr_rank[::-1]
    cellpair_order = cellpair_rank
    xmap = {cp: i for i, cp in enumerate(cellpair_order)}
    ymap = {lr: i for i, lr in enumerate(lr_order)}
    max_support = max(float(plot_df['support_score'].max()), 1e-9)
    vmax_strength = np.nanpercentile(plot_df['mean_strength'], 95) if plot_df['mean_strength'].notna().any() else 1.0
    vmax_strength = max(float(vmax_strength), 1e-9)

    fig_w = max(13, 0.32 * len(cellpair_order) * len(AGE_ORDER) + 3.5)
    fig_h = max(6.5, 0.30 * len(lr_order) + 2.0)
    fig, axes = plt.subplots(1, len(AGE_ORDER), figsize=(fig_w, fig_h), sharey=True,
                             gridspec_kw={'wspace': 0.04})
    if len(AGE_ORDER) == 1:
        axes = [axes]
    cmap = mpl.cm.YlOrRd
    norm = mpl.colors.Normalize(vmin=0, vmax=vmax_strength)
    for ax, age_group in zip(axes, AGE_ORDER):
        sub = plot_df[plot_df['age_group'] == age_group].copy()
        if not sub.empty:
            x = sub['cellpair'].map(xmap).astype(float)
            y = sub['lr_pair'].map(ymap).astype(float)
            sizes = np.clip((sub['support_score'] / max_support) * 260, 12, 280)
            colors = cmap(norm(sub['mean_strength'].clip(lower=0)))
            ax.scatter(x, y, s=sizes, c=colors, alpha=0.92,
                       edgecolors='#3A3A3A', linewidths=0.65)
        for xi in np.arange(len(cellpair_order) + 1) - 0.5:
            ax.axvline(xi, color='#D0D0D0', lw=1.055, ls='--', zorder=0)
        ax.set_xlim(-0.6, len(cellpair_order) - 0.4)
        ax.set_xticks(np.arange(len(cellpair_order)))
        ax.set_xticklabels([cp.replace('|', ' -> ') for cp in cellpair_order], rotation=90, fontsize=12)
        ax.set_title(age_group, color=AGE_PAL.get(age_group, '#333333'), fontweight='bold')
        ax.set_xlabel('Sender→receiver')
    axes[0].set_yticks(np.arange(len(lr_order)))
    axes[0].set_yticklabels(lr_order, fontsize=12)
    for tick in axes[0].get_yticklabels():
        text = tick.get_text()
        if text in elderly_specific:
            tick.set_color('#C84E4E')
            tick.set_fontweight('bold')
        elif text in young_adult_specific:
            tick.set_color('#3B74AF')
            tick.set_fontweight('bold')
    axes[0].set_ylabel('Ligand-receptor pair')
    fig.suptitle(title, y=1.02, fontweight='bold')
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, fraction=0.018, pad=0.012)
    cbar.set_label('Mean FastCCC strength')
    size_ax = axes[-1]
    for val in [0.25, 0.50, 1.00]:
        size_ax.scatter([], [], s=np.clip(val * 260, 40, 320), c='#888888', alpha=0.65,
                        edgecolors='#3A3A3A', linewidths=0.65, label=f'{val * max_support:.2g}')
    size_ax.legend(title=size_label, frameon=False, loc='lower left', bbox_to_anchor=(1.02, 0.02), fontsize=12, title_fontsize=12)
    fig.text(0.01, 0.01, 'Red LR labels: Elderly-specific; blue LR labels: Young adult-specific (displayed matrix, donor_frac >= 0.10).',
             fontsize=12, color='#555555')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    plot_df.to_csv(TAB_DIR / 'S7_lr_driver_fastccc_condition_circleplot_values.tsv', sep='\t', index=False)
    return fig




def prepare_fig4h_group_lr(sig_long, selected_entities=None):
    """Build the group-level LRI table expected by the FastCCC Fig.4h-style contrast plot."""
    if sig_long is None or sig_long.empty:
        return pd.DataFrame()
    df = sig_long.copy()
    if 'entity' not in df.columns:
        df['entity'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    if selected_entities:
        df = df[df['entity'].isin(selected_entities)].copy()
    if df.empty:
        return pd.DataFrame()
    df = df.merge(donor_base[['sample_id', 'age_group']], on='sample_id', how='left', suffixes=('', '_donor'))
    if 'age_group_donor' in df.columns:
        df['age_group'] = df['age_group'].fillna(df['age_group_donor']) if 'age_group' in df.columns else df['age_group_donor']
    df = df[df['age_group'].isin(['Young adult', 'Adult', 'Elderly'])].copy()
    if df.empty:
        return pd.DataFrame()

    p_col = first_existing_col(df, ['p_value', 'pvalue', 'p_val', 'p', 'pval', 'significant_p'], None)
    agg_dict = {'frac_sig_n': ('sample_id', 'nunique'), 'mean_strength': ('strength', 'mean')}
    if p_col is not None:
        agg_dict['mean_pval'] = (p_col, 'mean')
        agg_dict['min_p'] = (p_col, 'min')
    group_lr = (df.groupby(['age_group', 'cellpair', 'lr_pair'], observed=True)
                  .agg(**agg_dict).reset_index().rename(columns={'age_group': 'group'}))
    group_n = donor_base[donor_base['age_group'].isin(['Young adult', 'Adult', 'Elderly'])].groupby('age_group')['sample_id'].nunique().to_dict()
    group_lr['group_donor_n'] = group_lr['group'].map(group_n).fillna(group_lr['frac_sig_n'])
    group_lr['frac_sig'] = group_lr['frac_sig_n'] / group_lr['group_donor_n'].replace(0, np.nan)
    if p_col is not None:
        group_lr['mean_pval'] = group_lr['mean_pval'].fillna(1.0)
        group_lr['neglog10_p'] = -np.log10(group_lr['min_p'].clip(lower=1e-300))
    else:
        group_lr['mean_pval'] = np.nan
        group_lr['neglog10_p'] = np.minimum(group_lr['frac_sig'] * 3.0, 3.0)

    def _parse_cellpair_fig4h(cellpair):
        text = str(cellpair)
        if '|' in text:
            a, b = text.split('|', 1)
        elif '->' in text:
            a, b = text.split('->', 1)
        else:
            a = b = text
        return a.strip(), b.strip()

    def _lr_label_fig4h(lr_pair):
        text = str(lr_pair)
        if '_' in text:
            lig, rec = text.split('_', 1)
            return f'{lig} -> {rec}'
        return text

    parsed = group_lr['cellpair'].map(_parse_cellpair_fig4h)
    group_lr['sender'] = [x[0] for x in parsed]
    group_lr['receiver'] = [x[1] for x in parsed]
    group_lr['lr_label'] = group_lr['lr_pair'].map(_lr_label_fig4h)
    return group_lr


def plot_fig4h_lri_contrast(group_lr, g1, g2, top_n=30, min_frac=0.05, outfile=None,
                            title=None, global_vmax=3.0):
    """Exact FastCCC Fig.4h-style two-group LRI matrix.

    Rows are LRI labels, columns are sender->receiver cell pairs. Dot size and color
    jointly encode -log10(P) support in the two compared groups.
    """
    import matplotlib.patches as mpatches
    import matplotlib.colors as mcolors
    from matplotlib.transforms import blended_transform_factory
    from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    if group_lr is None or group_lr.empty:
        return None

    COLOR_G1 = np.array(mcolors.to_rgb('#253494'))
    COLOR_G2 = np.array(mcolors.to_rgb('#F4A53A'))
    COLOR_BOTH = np.array(mcolors.to_rgb('#D71958'))
    COLOR_WEAK = np.array(mcolors.to_rgb('#F6E8C3'))
    LIGAND_COLOR = '#1F4E79'
    RECEPTOR_COLOR = '#D71958'

    def _parse_cellpair_fig4h(cellpair):
        text = str(cellpair)
        if '|' in text:
            a, b = text.split('|', 1)
        elif '->' in text:
            a, b = text.split('->', 1)
        else:
            a = b = text
        return a.strip(), b.strip()

    def _norm_s(v, cap):
        return (np.clip(float(v), 0, cap) / cap) ** 0.55

    def _norm_d(d, cap):
        return np.clip(float(d), -cap, cap) / cap

    def _dot_style(nlp1, nlp2, vmax=global_vmax):
        left_val, right_val = float(nlp1), float(nlp2)
        strength = _norm_s(left_val + right_val, vmax * 2)
        diff = _norm_d(left_val - right_val, vmax)
        size = 12 + 220 * strength
        if strength < 0.12:
            rgb = COLOR_WEAK
        else:
            if diff > 0.15:
                dom = min(abs(diff), 1.0)
                rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G1
            elif diff < -0.15:
                dom = min(abs(diff), 1.0)
                rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G2
            else:
                rgb = COLOR_BOTH
            rgb = COLOR_WEAK * (1 - strength) + rgb * strength
        rgb = np.clip(rgb, 0, 1)
        return (*rgb, 1.0), size

    def _make_legend_img(vmax=global_vmax, N=50):
        img = np.zeros((N, N, 4))
        vals = np.linspace(0, vmax, N)
        for ix in range(N):
            for iy in range(N):
                rgba, _ = _dot_style(vals[iy], vals[ix], vmax)
                img[N - 1 - iy, ix] = rgba
        return img

    cols = ['cellpair', 'lr_pair', 'lr_label', 'sender', 'receiver', 'mean_pval', 'frac_sig', 'neglog10_p']
    d1 = group_lr.loc[group_lr['group'] == g1, cols].copy()
    d2 = group_lr.loc[group_lr['group'] == g2, ['cellpair', 'lr_pair', 'mean_pval', 'frac_sig', 'neglog10_p']].copy()
    mg = d1.merge(d2, on=['cellpair', 'lr_pair'], suffixes=(f'_{g1}', f'_{g2}'), how='outer')
    for s in [g1, g2]:
        mg[f'mean_pval_{s}'] = mg[f'mean_pval_{s}'].fillna(1.0)
        mg[f'frac_sig_{s}'] = mg[f'frac_sig_{s}'].fillna(0.0)
        mg[f'neglog10_p_{s}'] = mg[f'neglog10_p_{s}'].fillna(0.0)
    if mg.empty:
        print(f'No LRI for {g1} vs {g2}')
        return None

    if mg['sender'].isna().any():
        parsed_missing = mg.loc[mg['sender'].isna(), 'cellpair'].map(_parse_cellpair_fig4h)
        mg.loc[mg['sender'].isna(), 'sender'] = [x[0] for x in parsed_missing]
        parsed_missing = mg.loc[mg['receiver'].isna(), 'cellpair'].map(_parse_cellpair_fig4h)
        mg.loc[mg['receiver'].isna(), 'receiver'] = [x[1] for x in parsed_missing]
    if mg['lr_label'].isna().any():
        lm = group_lr[['lr_pair', 'lr_label']].drop_duplicates('lr_pair')
        mg = mg.drop(columns='lr_label').merge(lm, on='lr_pair', how='left')
        mg['lr_label'] = mg['lr_label'].fillna(mg['lr_pair'])

    mg['max_nlp'] = mg[[f'neglog10_p_{g1}', f'neglog10_p_{g2}']].max(axis=1)
    mg = mg[(mg[f'frac_sig_{g1}'] >= min_frac) | (mg[f'frac_sig_{g2}'] >= min_frac)].copy()
    if mg.empty:
        print(f'No LRI for {g1} vs {g2}')
        return None

    top_lr = mg.groupby('lr_label')['max_nlp'].max().nlargest(top_n).index.tolist()
    mg = mg[mg['lr_label'].isin(top_lr)].copy()
    lr_order = mg.groupby('lr_label')['max_nlp'].max().sort_values(ascending=True).index.tolist()
    active_cp = set(mg['cellpair'].unique())
    cp_order = (mg[['cellpair', 'sender', 'receiver']].drop_duplicates()
                  .sort_values(['sender', 'receiver'])['cellpair'].tolist())
    cp_order = [c for c in cp_order if c in active_cp]
    n_y, n_x = len(lr_order), len(cp_order)
    if n_x == 0 or n_y == 0:
        print(f'Empty for {g1} vs {g2}')
        return None

    col_w = max(0.30, min(0.55, 18 / n_x))
    row_h = max(0.28, min(0.42, 14 / n_y))
    fig, ax = plt.subplots(figsize=(max(10, n_x * col_w + 5), max(6, n_y * row_h + 3.8)))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    for _, r in mg.iterrows():
        if r['lr_label'] not in lr_order or r['cellpair'] not in cp_order:
            continue
        xi, yi = cp_order.index(r['cellpair']), lr_order.index(r['lr_label'])
        n1, n2 = r[f'neglog10_p_{g1}'], r[f'neglog10_p_{g2}']
        rgba, sz = _dot_style(n1, n2)
        ax.scatter(xi, yi, s=sz, c=[rgba[:3]], alpha=rgba[3],
                   edgecolors='k' if max(n1, n2) > 0.3 else 'none',
                   linewidths=0.6, zorder=3)

    senders = [_parse_cellpair_fig4h(c)[0] for c in cp_order]
    for i in range(1, n_x):
        if senders[i] != senders[i - 1]:
            ax.axvline(i - 0.5, color='#AAAAAA', ls='--', lw=1.05, zorder=1)
    for yi in range(n_y):
        ax.axhline(yi, color='#EEEEEE', lw=0.95, zorder=0)

    ax.set_xlim(-0.5, n_x - 0.5)
    ax.set_ylim(-0.5, n_y - 0.5)
    ax.set_xticks(range(n_x))
    ax.set_xticklabels([''] * n_x)
    ax.set_yticks(range(n_y))
    ax.set_yticklabels([''] * n_y)
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)

    for i, lr in enumerate(lr_order):
        pts = [p.strip() for p in str(lr).split(' -> ')]
        if len(pts) == 2:
            children = [
                TextArea(pts[0], textprops=dict(color=LIGAND_COLOR, size=10, weight='bold')),
                TextArea('-', textprops=dict(color='#555555', size=10)),
                TextArea(pts[1], textprops=dict(color=RECEPTOR_COLOR, size=10, weight='bold')),
            ]
        else:
            children = [TextArea(str(lr), textprops=dict(color='black', size=10))]
        ax.add_artist(AnnotationBbox(HPacker(children=children, pad=0, sep=1, align='center'),
                                     xy=(-0.005, i), xycoords=('axes fraction', 'data'),
                                     box_alignment=(1.0, 0.5), frameon=False))

    xt = blended_transform_factory(ax.transData, ax.transAxes)
    for i, cp in enumerate(cp_order):
        _, recv = _parse_cellpair_fig4h(cp)
        ax.text(i, -0.015, recv, transform=xt, ha='right', va='top',
                rotation=45, fontsize=12, color=RECEPTOR_COLOR, weight='bold',
                style='italic', clip_on=False)

    sender_groups = {}
    for i, cp in enumerate(cp_order):
        sender, _ = _parse_cellpair_fig4h(cp)
        sender_groups.setdefault(sender, []).append(i)
    for sender, idxs in sender_groups.items():
        mid = (idxs[0] + idxs[-1]) / 2
        ax.plot([idxs[0] - 0.35, idxs[-1] + 0.35], [-0.17, -0.17],
                color='#AFAFAF', lw=1.0, transform=xt, clip_on=False)
        ax.text(mid, -0.31, sender, transform=xt, ha='center', va='top',
                rotation=90, fontsize=12, color=LIGAND_COLOR, clip_on=False)

    ax.set_title(title or f'Differential LRI - {g1} vs {g2}', size=11, weight='bold', pad=8, loc='left')

    ax_l = inset_axes(ax, width='9%', height='13%', loc='lower left', borderpad=2.2)
    ax_l.imshow(_make_legend_img(), aspect='equal', origin='lower', extent=[0, global_vmax, 0, global_vmax])
    ax_l.set_xlabel(f'-log10({g2} P)', fontsize=12, labelpad=1)
    ax_l.set_ylabel(f'-log10({g1} P)', fontsize=12, labelpad=1)
    ax_l.set_xticks([0, 1, 2, global_vmax])
    ax_l.set_xticklabels(['0', '1', '2', '>3'], fontsize=12)
    ax_l.set_yticks([0, 1, 2, global_vmax])
    ax_l.set_yticklabels(['0', '1', '2', '>3'], fontsize=12)
    ax_l.tick_params(length=2, pad=1)
    for sp in ax_l.spines.values():
        sp.set_linewidth(0.4)

    size_handles = []
    for v in [1, 2, 3]:
        _, sz = _dot_style(v, 0)
        size_handles.append(plt.scatter([], [], s=sz, c=[[*COLOR_G1, 0.85]],
                                        edgecolors='k', linewidths=0.6, label=f'-log10(P)={v}'))
    color_handles = [
        mpatches.Patch(facecolor=COLOR_G1, label=f'{g1} dominant'),
        mpatches.Patch(facecolor=COLOR_G2, label=f'{g2} dominant'),
        mpatches.Patch(facecolor=COLOR_BOTH, label='Both high'),
        mpatches.Patch(facecolor=COLOR_WEAK, label='Both weak'),
    ]
    ax.legend(handles=size_handles + color_handles, loc='upper left', bbox_to_anchor=(1.01, 0.50),
              frameon=True, framealpha=0.92, fontsize=12, title='Legend',
              title_fontsize=12, edgecolor='#CCCCCC', handletextpad=0.8)

    plt.subplots_adjust(bottom=0.26, left=0.18, right=0.87, top=0.93)
    if outfile:
        save_fig(fig, outfile, FIG_DIR)
        mg.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    plt.show()
    return fig

def plot_pairwise_lri_triptych(sig_long, selected_entities, title, outfile,
                               comparisons=None, top_n_lr=30, top_n_cellpairs=28,
                               min_frac=0.05, global_vmax=3.0):
    """FastCCC Fig. 4h-style pairwise LRI bubble matrices for Young adult/Adult/Elderly.

    One dot jointly encodes the two compared groups: size is combined support,
    color is left-dominant / right-dominant / both-high / both-weak support.
    """
    import matplotlib.patches as mpatches
    import matplotlib.colors as mcolors
    from matplotlib.transforms import blended_transform_factory
    from matplotlib.offsetbox import TextArea, HPacker, AnnotationBbox
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    if comparisons is None:
        comparisons = [('Young adult', 'Adult', 'early'), ('Adult', 'Elderly', 'late'), ('Young adult', 'Elderly', 'full aging')]
    if sig_long is None or sig_long.empty or not selected_entities:
        return None

    COLOR_G1 = np.array(mcolors.to_rgb('#253494'))
    COLOR_G2 = np.array(mcolors.to_rgb('#F4A53A'))
    COLOR_BOTH = np.array(mcolors.to_rgb('#D71958'))
    COLOR_WEAK = np.array(mcolors.to_rgb('#F6E8C3'))
    LIGAND_COLOR = '#1F4E79'
    RECEPTOR_COLOR = '#D71958'

    def _parse_cellpair_local(cellpair):
        text = str(cellpair)
        if '|' in text:
            a, b = text.split('|', 1)
        elif '->' in text:
            a, b = text.split('->', 1)
        elif '->' in text:
            a, b = text.split('->', 1)
        else:
            a = b = text
        return a.strip(), b.strip()

    def _lr_label(lr_pair):
        text = str(lr_pair)
        if '_' in text:
            lig, rec = text.split('_', 1)
            return f'{lig} -> {rec}'
        return text

    def _norm_s(v, cap):
        return (np.clip(float(v), 0, cap) / cap) ** 0.55

    def _norm_d(d, cap):
        return np.clip(float(d), -cap, cap) / cap

    def _dot_style(nlp1, nlp2, vmax=global_vmax):
        g1, g2 = float(nlp1), float(nlp2)
        strength = _norm_s(g1 + g2, vmax * 2)
        diff = _norm_d(g1 - g2, vmax)
        size = 12 + 220 * strength
        if strength < 0.12:
            rgb = COLOR_WEAK
        else:
            if diff > 0.15:
                dom = min(abs(diff), 1.0)
                rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G1
            elif diff < -0.15:
                dom = min(abs(diff), 1.0)
                rgb = (1 - dom) * COLOR_BOTH + dom * COLOR_G2
            else:
                rgb = COLOR_BOTH
            rgb = COLOR_WEAK * (1 - strength) + rgb * strength
        rgb = np.clip(rgb, 0, 1)
        return (*rgb, 1.0), size

    def _make_legend_img(vmax=global_vmax, N=50):
        img = np.zeros((N, N, 4))
        vals = np.linspace(0, vmax, N)
        for ix in range(N):
            for iy in range(N):
                rgba, _ = _dot_style(vals[iy], vals[ix], vmax)
                img[N - 1 - iy, ix] = rgba
        return img

    df = sig_long.copy()
    if 'entity' not in df.columns:
        df['entity'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    df = df[df['entity'].isin(selected_entities)].copy()
    if df.empty:
        return None
    df = df.merge(donor_base[['sample_id', 'age_group']], on='sample_id', how='left', suffixes=('', '_donor'))
    if 'age_group_donor' in df.columns:
        df['age_group'] = df['age_group'].fillna(df['age_group_donor']) if 'age_group' in df.columns else df['age_group_donor']
    groups_needed = list(dict.fromkeys([g for left, right, _ in comparisons for g in [left, right]]))
    df = df[df['age_group'].isin(groups_needed)].copy()
    if df.empty:
        return None

    p_col = first_existing_col(df, ['p_value', 'pvalue', 'p_val', 'p', 'pval', 'significant_p'], None)
    agg_dict = {'donor_n': ('sample_id', 'nunique'), 'mean_strength': ('strength', 'mean')}
    if p_col is not None:
        agg_dict['mean_pval'] = (p_col, 'mean')
        agg_dict['min_p'] = (p_col, 'min')
    group_lr = (df.groupby(['age_group', 'cellpair', 'lr_pair'], observed=True)
                .agg(**agg_dict).reset_index().rename(columns={'age_group': 'group'}))
    group_n = donor_base[donor_base['age_group'].isin(groups_needed)].groupby('age_group')['sample_id'].nunique().to_dict()
    group_lr['donor_total'] = group_lr['group'].map(group_n).fillna(group_lr['donor_n'])
    group_lr['frac_sig'] = group_lr['donor_n'] / group_lr['donor_total'].replace(0, np.nan)
    if p_col is not None:
        group_lr['neglog10_p'] = -np.log10(group_lr['min_p'].clip(lower=1e-300))
    else:
        group_lr['mean_pval'] = np.nan
        group_lr['neglog10_p'] = np.minimum(group_lr['frac_sig'] * global_vmax, global_vmax)
    group_lr['lr_label'] = group_lr['lr_pair'].map(_lr_label)
    parsed = group_lr['cellpair'].map(_parse_cellpair_local)
    group_lr['sender'] = [x[0] for x in parsed]
    group_lr['receiver'] = [x[1] for x in parsed]
    group_lr.to_csv(TAB_DIR / 'S7_grouped_LRI_Fig4h_support_table.tsv', sep='	', index=False)

    def _plot_one(ax, g1, g2, subtitle, is_first=False):
        cols = ['cellpair', 'lr_pair', 'lr_label', 'sender', 'receiver', 'mean_pval', 'frac_sig', 'neglog10_p']
        d1 = group_lr.loc[group_lr['group'] == g1, cols].copy()
        d2 = group_lr.loc[group_lr['group'] == g2, ['cellpair', 'lr_pair', 'mean_pval', 'frac_sig', 'neglog10_p']].copy()
        mg = d1.merge(d2, on=['cellpair', 'lr_pair'], suffixes=(f'_{g1}', f'_{g2}'), how='outer')
        for s in [g1, g2]:
            mg[f'mean_pval_{s}'] = mg[f'mean_pval_{s}'].fillna(1.0)
            mg[f'frac_sig_{s}'] = mg[f'frac_sig_{s}'].fillna(0.0)
            mg[f'neglog10_p_{s}'] = mg[f'neglog10_p_{s}'].fillna(0.0)
        if mg['sender'].isna().any():
            parsed_missing = mg['cellpair'].map(_parse_cellpair_local)
            mg['sender'] = mg['sender'].fillna(pd.Series([x[0] for x in parsed_missing], index=mg.index))
            mg['receiver'] = mg['receiver'].fillna(pd.Series([x[1] for x in parsed_missing], index=mg.index))
        if mg['lr_label'].isna().any():
            lm = group_lr[['lr_pair', 'lr_label']].drop_duplicates('lr_pair')
            mg = mg.drop(columns='lr_label').merge(lm, on='lr_pair', how='left')
            mg['lr_label'] = mg['lr_label'].fillna(mg['lr_pair'])
        mg['max_nlp'] = mg[[f'neglog10_p_{g1}', f'neglog10_p_{g2}']].max(axis=1)
        mg = mg[(mg[f'frac_sig_{g1}'] >= min_frac) | (mg[f'frac_sig_{g2}'] >= min_frac)].copy()
        if mg.empty:
            ax.text(0.5, 0.5, f'No LRI for {g1} vs {g2}', transform=ax.transAxes, ha='center', va='center')
            ax.axis('off')
            return None
        top_lr = mg.groupby('lr_label')['max_nlp'].max().nlargest(top_n_lr).index.tolist()
        mg = mg[mg['lr_label'].isin(top_lr)].copy()
        lr_order = mg.groupby('lr_label')['max_nlp'].max().sort_values(ascending=True).index.tolist()
        cp_order = (mg[['cellpair', 'sender', 'receiver']].drop_duplicates().sort_values(['sender', 'receiver'])['cellpair'].tolist())[:top_n_cellpairs]
        mg = mg[mg['cellpair'].isin(cp_order)].copy()
        if not lr_order or not cp_order:
            ax.axis('off')
            return None
        for _, r in mg.iterrows():
            xi, yi = cp_order.index(r['cellpair']), lr_order.index(r['lr_label'])
            n1, n2 = r[f'neglog10_p_{g1}'], r[f'neglog10_p_{g2}']
            rgba, sz = _dot_style(n1, n2)
            ax.scatter(xi, yi, s=sz, c=[rgba[:3]], alpha=rgba[3], edgecolors='k' if max(n1, n2) > 0.3 else 'none', linewidths=0.6, zorder=3)
        senders = [_parse_cellpair_local(c)[0] for c in cp_order]
        for i in range(1, len(cp_order)):
            if senders[i] != senders[i - 1]:
                ax.axvline(i - 0.5, color='#AAAAAA', ls='--', lw=1.05, zorder=1)
        for yi in range(len(lr_order)):
            ax.axhline(yi, color='#EEEEEE', lw=0.95, zorder=0)
        ax.set_xlim(-0.5, len(cp_order) - 0.5)
        ax.set_ylim(-0.5, len(lr_order) - 0.5)
        ax.set_xticks(range(len(cp_order)))
        ax.set_xticklabels([''] * len(cp_order))
        ax.set_yticks(range(len(lr_order)))
        ax.set_yticklabels([''] * len(lr_order))
        ax.tick_params(length=0)
        for sp in ax.spines.values():
            sp.set_visible(False)
        if is_first:
            for i, lr in enumerate(lr_order):
                pts = [p.strip() for p in lr.split(' -> ')]
                if len(pts) == 2:
                    children = [TextArea(pts[0], textprops=dict(color=LIGAND_COLOR, size=10, weight='bold')), TextArea('-', textprops=dict(color='#555555', size=10)), TextArea(pts[1], textprops=dict(color=RECEPTOR_COLOR, size=10, weight='bold'))]
                else:
                    children = [TextArea(lr, textprops=dict(color='black', size=10))]
                ax.add_artist(AnnotationBbox(HPacker(children=children, pad=0, sep=1, align='center'), xy=(-0.01, i), xycoords=('axes fraction', 'data'), box_alignment=(1.0, 0.5), frameon=False))
        xt = blended_transform_factory(ax.transData, ax.transAxes)
        for i, cp in enumerate(cp_order):
            _, recv = _parse_cellpair_local(cp)
            ax.text(i, -0.015, recv, transform=xt, ha='right', va='top', rotation=45, fontsize=12, color=RECEPTOR_COLOR, weight='bold', style='italic', clip_on=False)
        sender_groups = {}
        for i, cp in enumerate(cp_order):
            sender, _ = _parse_cellpair_local(cp)
            sender_groups.setdefault(sender, []).append(i)
        for sender, idxs in sender_groups.items():
            mid = (idxs[0] + idxs[-1]) / 2
            ax.plot([idxs[0] - 0.35, idxs[-1] + 0.35], [-0.17, -0.17], color='#AFAFAF', lw=1.0, transform=xt, clip_on=False)
            ax.text(mid, -0.31, sender, transform=xt, ha='center', va='top', rotation=90, fontsize=12, color=LIGAND_COLOR, clip_on=False)
        ax.set_title(f'{g1} vs {g2} - {subtitle}', size=10, weight='bold', pad=8, loc='left')
        ax_l = inset_axes(ax, width='18%', height='18%', loc='lower left', borderpad=1.8)
        ax_l.imshow(_make_legend_img(), aspect='equal', origin='lower', extent=[0, global_vmax, 0, global_vmax])
        ax_l.set_xlabel(f'-log10({g2} P)', fontsize=12, labelpad=1)
        ax_l.set_ylabel(f'-log10({g1} P)', fontsize=12, labelpad=1)
        ax_l.set_xticks([0, 1, 2, global_vmax]); ax_l.set_xticklabels(['0', '1', '2', '>3'], fontsize=12)
        ax_l.set_yticks([0, 1, 2, global_vmax]); ax_l.set_yticklabels(['0', '1', '2', '>3'], fontsize=12)
        ax_l.tick_params(length=2, pad=1)
        for sp in ax_l.spines.values(): sp.set_linewidth(0.4)
        return mg

    fig_w = max(13.0, 5.4 * len(comparisons))
    fig_h = max(7.2, 0.34 * top_n_lr + 3.5)
    fig, axes = plt.subplots(1, len(comparisons), figsize=(fig_w, fig_h), squeeze=False)
    axes = axes.flatten()
    source_parts = []
    for i, (ax, (g1, g2, subtitle)) in enumerate(zip(axes, comparisons)):
        src_part = _plot_one(ax, g1, g2, subtitle, is_first=(i == 0))
        if src_part is not None:
            src_part['contrast'] = f'{g1}_vs_{g2}'
            source_parts.append(src_part)
    size_handles = []
    for v in [1, 2, 3]:
        _, sz = _dot_style(v, 0)
        size_handles.append(plt.scatter([], [], s=sz, c=[[*COLOR_G1, 0.85]], edgecolors='k', linewidths=0.6, label=f'-log10(P)={v}'))
    color_handles = [mpatches.Patch(facecolor=COLOR_G1, label='left group dominant'), mpatches.Patch(facecolor=COLOR_G2, label='right group dominant'), mpatches.Patch(facecolor=COLOR_BOTH, label='both high'), mpatches.Patch(facecolor=COLOR_WEAK, label='both weak')]
    axes[-1].legend(handles=size_handles + color_handles, loc='upper left', bbox_to_anchor=(1.02, 0.92), frameon=True, framealpha=0.92, fontsize=12, title='2D LRI support', title_fontsize=12, edgecolor='#CCCCCC', handletextpad=0.8)
    fig.suptitle(title, y=0.995, fontsize=12, fontweight='bold')
    plt.subplots_adjust(bottom=0.34, left=0.18, right=0.86, top=0.86, wspace=0.38)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    if source_parts:
        pd.concat(source_parts, ignore_index=True).to_csv(TAB_DIR / 'S7_lr_driver_pairwise_Fig4h_values.tsv', sep='	', index=False)
    return fig


def plot_young_middle_old_lri_circleplot(sig_long, selected_entities, title, outfile, top_n_lr=22, top_n_cellpairs=18):
    """FastCCC Fig. 4h-inspired LRI circle plot for three aging groups.

    Rows are LRIs; each sender-receiver cell pair is split into Young adult, Adult, and Elderly
    subcolumns. Dot size and color intensity both encode group support/significance.
    """
    if sig_long is None or sig_long.empty or not selected_entities:
        return None
    df = sig_long.copy()
    if 'entity' not in df.columns:
        df['entity'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    df = df[df['entity'].isin(selected_entities)].copy()
    if df.empty:
        return None
    df = df.merge(donor_base[['sample_id','age_group']], on='sample_id', how='left', suffixes=('', '_donor'))
    if 'age_group_donor' in df.columns:
        df['age_group'] = df['age_group'].fillna(df['age_group_donor']) if 'age_group' in df.columns else df['age_group_donor']
    df = df[df['age_group'].isin(AGE_ORDER)].copy()
    if df.empty:
        return None
    p_col = first_existing_col(df, ['p_value', 'pvalue', 'p_val', 'p', 'pval', 'significant_p'], None)
    agg_dict = {'mean_strength': ('strength', 'mean'), 'donor_n': ('sample_id', 'nunique')}
    if p_col is not None:
        agg_dict['min_p'] = (p_col, 'min')
    grp = (df.groupby(['age_group','cellpair','lr_pair'], observed=True)
           .agg(**agg_dict).reset_index())
    group_n = donor_base[donor_base['age_group'].isin(AGE_ORDER)].groupby('age_group')['sample_id'].nunique().to_dict()
    grp['donor_total'] = grp['age_group'].map(group_n).fillna(grp['donor_n'])
    grp['donor_frac'] = grp['donor_n'] / grp['donor_total'].replace(0, np.nan)
    if p_col is not None:
        grp['support_score'] = -np.log10(grp['min_p'].clip(lower=1e-300))
        support_label = '-log10(min FastCCC p)'
    else:
        grp['support_score'] = grp['donor_frac']
        support_label = 'donor occupancy'

    piv = grp.pivot_table(index=['cellpair','lr_pair'], columns='age_group',
                          values='support_score', aggfunc='max', fill_value=0).reset_index()
    for g in AGE_ORDER:
        if g not in piv.columns:
            piv[g] = 0.0
    piv['range_support'] = piv[AGE_ORDER].max(axis=1) - piv[AGE_ORDER].min(axis=1)
    piv['max_support'] = piv[AGE_ORDER].max(axis=1)
    lr_order = (piv.groupby('lr_pair')[['range_support','max_support']].max()
                .sort_values(['range_support','max_support'], ascending=False)
                .head(top_n_lr).index.tolist())
    cellpair_order = (piv[piv['lr_pair'].isin(lr_order)]
                      .groupby('cellpair')[['range_support','max_support']].max()
                      .sort_values(['range_support','max_support'], ascending=False)
                      .head(top_n_cellpairs).index.tolist())
    plot_df = grp[grp['lr_pair'].isin(lr_order) & grp['cellpair'].isin(cellpair_order)].copy()
    if plot_df.empty:
        return None

    mem = plot_df.assign(present=plot_df['donor_frac'] >= 0.10).pivot_table(
        index='lr_pair', columns='age_group', values='present', aggfunc='max', fill_value=False)
    for g in AGE_ORDER:
        if g not in mem.columns:
            mem[g] = False
    elderly_specific = set(mem.index[mem['Elderly'] & ~mem['Young adult'] & ~mem['Adult']])
    young_adult_specific = set(mem.index[mem['Young adult'] & ~mem['Adult'] & ~mem['Elderly']])
    middle_specific = set(mem.index[mem['Adult'] & ~mem['Young adult'] & ~mem['Elderly']])

    lr_order = lr_order[::-1]
    ymap = {lr: i for i, lr in enumerate(lr_order)}
    base_x = {cp: i * 3.45 for i, cp in enumerate(cellpair_order)}
    offsets = {'Young adult': 0.00, 'Adult': 0.72, 'Elderly': 1.44}
    group_colors = {
        'Young adult': np.array(mpl.colors.to_rgb(YOUNG_COLOR)),
        'Adult': np.array(mpl.colors.to_rgb(AGE_PAL.get('Adult', '#D8A03D'))),
        'Elderly': np.array(mpl.colors.to_rgb(OLD_COLOR)),
    }
    max_support = max(float(plot_df['support_score'].max()), 1e-9)
    fig_w = max(13.4, 0.72 * len(cellpair_order) + 4.8)
    fig_h = max(6.4, 0.30 * len(lr_order) + 2.5)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    for age_group in AGE_ORDER:
        sub = plot_df[plot_df['age_group'] == age_group].copy()
        if sub.empty:
            continue
        scaled = (sub['support_score'] / max_support).clip(0, 1).values
        colors = [tuple((1 - s) * np.array([0.94, 0.94, 0.94]) + s * group_colors[age_group]) for s in scaled]
        sizes = np.clip(16 + scaled * 245, 14, 275)
        xs = sub['cellpair'].map(base_x).astype(float) + offsets[age_group]
        ys = sub['lr_pair'].map(ymap).astype(float)
        ax.scatter(xs, ys, s=sizes, c=colors, edgecolors='white',
                   linewidths=0.755, alpha=0.96, zorder=3, label=age_group)

    for cp in cellpair_order:
        x0 = base_x[cp] - 0.42
        ax.axvline(x0, color='#D0D0D0', lw=1.0, ls='--', zorder=0)
        ax.text(base_x[cp] + 0.72, -0.15, cp.replace('|', '→'),
                ha='right', va='top', rotation=90, fontsize=12,
                transform=ax.get_xaxis_transform(), color='#24415E')
    ax.axvline(base_x[cellpair_order[-1]] + 1.95, color='#D0D0D0', lw=1.0, ls='--', zorder=0)

    xticks, xticklabels = [], []
    short = {'Young adult': 'YA', 'Adult': 'A', 'Elderly': 'E'}
    for cp in cellpair_order:
        for age_group in AGE_ORDER:
            xticks.append(base_x[cp] + offsets[age_group])
            xticklabels.append(short[age_group])
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, fontsize=12, rotation=0)
    for tick, label in zip(ax.get_xticklabels(), xticklabels):
        color = {'YA': YOUNG_COLOR, 'A': AGE_PAL.get('Adult', '#D8A03D'), 'E': OLD_COLOR}[label]
        tick.set_color(color)
        tick.set_fontweight('bold')

    ax.set_yticks(np.arange(len(lr_order)))
    ax.set_yticklabels(lr_order, fontsize=12)
    for tick in ax.get_yticklabels():
        lr = tick.get_text()
        if lr in elderly_specific:
            tick.set_color(OLD_COLOR); tick.set_fontweight('bold')
        elif lr in young_adult_specific:
            tick.set_color(YOUNG_COLOR); tick.set_fontweight('bold')
        elif lr in middle_specific:
            tick.set_color(AGE_PAL.get('Adult', '#D8A03D')); tick.set_fontweight('bold')

    ax.set_xlabel('Sender→receiver cell pair, split into Young adult (YA), Adult (A), and Elderly (E) support')
    ax.set_ylabel('Ligand-receptor interaction')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(-0.78, base_x[cellpair_order[-1]] + 2.22)
    ax.set_ylim(-0.7, len(lr_order) - 0.25)
    ax.grid(axis='y', alpha=0.10)

    group_handles = [
        mpl.lines.Line2D([0], [0], marker='o', color='none',
                         markerfacecolor=group_colors[g], markeredgecolor='white',
                         markersize=8, label=g)
        for g in AGE_ORDER
    ]
    size_handles = [
        plt.scatter([], [], s=np.clip(30 + frac * 260, 45, 330), c='#777777',
                    alpha=0.55, edgecolors='white', label=f'{frac * max_support:.2g}')
        for frac in [0.25, 0.50, 1.00]
    ]
    leg1 = ax.legend(handles=group_handles, frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1.0), title='Age group')
    ax.add_artist(leg1)
    ax.legend(handles=size_handles, frameon=False, loc='upper left', bbox_to_anchor=(1.01, 0.68), title=support_label)
    fig.text(0.01, 0.01, 'LRI label color marks group-specific support at donor fraction >= 0.10: Young adult blue, Adult gold, Elderly red.',
             fontsize=12, color='#555555')
    plt.subplots_adjust(bottom=0.31, right=0.82)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    plot_df.to_csv(TAB_DIR / 'S7_lr_driver_YMO_FastCCC_style_circleplot_values.tsv', sep='\t', index=False)
    return fig



def plot_young_old_differential_lri_circleplot(sig_long, selected_entities, title, outfile, top_n_lr=22, top_n_cellpairs=22):
    """FastCCC Fig. 4h-inspired grouped LRI circle plot for aging.

    Rows are LRIs; each sender-receiver pair is split into Young adult and Elderly subcolumns.
    Dot size and color intensity both encode group support/significance.
    """
    if sig_long is None or sig_long.empty or not selected_entities:
        return None
    df = sig_long.copy()
    if 'entity' not in df.columns:
        df['entity'] = df['cellpair'].astype(str) + '::' + df['lr_pair'].astype(str)
    df = df[df['entity'].isin(selected_entities)].copy()
    if df.empty:
        return None
    df = df.merge(donor_base[['sample_id','age_group']], on='sample_id', how='left', suffixes=('', '_donor'))
    if 'age_group_donor' in df.columns:
        df['age_group'] = df['age_group'].fillna(df['age_group_donor']) if 'age_group' in df.columns else df['age_group_donor']
    df = df[df['age_group'].isin(['Young adult','Elderly'])].copy()
    if df.empty:
        return None
    p_col = first_existing_col(df, ['p_value', 'pvalue', 'p_val', 'p', 'pval', 'significant_p'], None)
    agg_dict = {'mean_strength': ('strength', 'mean'), 'donor_n': ('sample_id', 'nunique')}
    if p_col is not None:
        agg_dict['min_p'] = (p_col, 'min')
    grp = (df.groupby(['age_group','cellpair','lr_pair'], observed=True)
           .agg(**agg_dict).reset_index())
    group_n = donor_base[donor_base['age_group'].isin(['Young adult','Elderly'])].groupby('age_group')['sample_id'].nunique().to_dict()
    grp['donor_total'] = grp['age_group'].map(group_n).fillna(grp['donor_n'])
    grp['donor_frac'] = grp['donor_n'] / grp['donor_total'].replace(0, np.nan)
    if p_col is not None:
        grp['support_score'] = -np.log10(grp['min_p'].clip(lower=1e-300))
        support_label = '-log10(min FastCCC p)'
    else:
        grp['support_score'] = grp['donor_frac']
        support_label = 'donor occupancy'

    piv = grp.pivot_table(index=['cellpair','lr_pair'], columns='age_group',
                          values='support_score', aggfunc='max', fill_value=0).reset_index()
    for g in ['Young adult','Elderly']:
        if g not in piv.columns:
            piv[g] = 0.0
    piv['diff_support'] = piv['Elderly'] - piv['Young adult']
    piv['max_support'] = piv[['Young adult','Elderly']].max(axis=1)
    piv['abs_diff_support'] = piv['diff_support'].abs()
    lr_order = (piv.groupby('lr_pair')[['abs_diff_support','max_support']].max()
                .sort_values(['abs_diff_support','max_support'], ascending=False)
                .head(top_n_lr).index.tolist())
    cellpair_order = (piv[piv['lr_pair'].isin(lr_order)]
                      .groupby('cellpair')[['abs_diff_support','max_support']].max()
                      .sort_values(['abs_diff_support','max_support'], ascending=False)
                      .head(top_n_cellpairs).index.tolist())
    plot_df = grp[grp['lr_pair'].isin(lr_order) & grp['cellpair'].isin(cellpair_order)].copy()
    if plot_df.empty:
        return None

    mem = plot_df.assign(present=plot_df['donor_frac'] >= 0.10).pivot_table(
        index='lr_pair', columns='age_group', values='present', aggfunc='max', fill_value=False)
    for g in ['Young adult','Elderly']:
        if g not in mem.columns:
            mem[g] = False
    elderly_specific = set(mem.index[mem['Elderly'] & ~mem['Young adult']])
    young_adult_specific = set(mem.index[mem['Young adult'] & ~mem['Elderly']])

    lr_order = lr_order[::-1]
    ymap = {lr: i for i, lr in enumerate(lr_order)}
    base_x = {cp: i * 2.65 for i, cp in enumerate(cellpair_order)}
    offsets = {'Young adult': 0.00, 'Elderly': 0.72}
    group_colors = {'Young adult': np.array(mpl.colors.to_rgb(YOUNG_COLOR)),
                    'Elderly': np.array(mpl.colors.to_rgb(OLD_COLOR))}
    max_support = max(float(plot_df['support_score'].max()), 1e-9)
    fig_w = max(12.8, 0.56 * len(cellpair_order) + 4.8)
    fig_h = max(6.4, 0.30 * len(lr_order) + 2.4)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    for age_group in ['Young adult', 'Elderly']:
        sub = plot_df[plot_df['age_group'] == age_group].copy()
        if sub.empty:
            continue
        scaled = (sub['support_score'] / max_support).clip(0, 1).values
        colors = [tuple((1 - s) * np.array([0.93, 0.93, 0.93]) + s * group_colors[age_group]) for s in scaled]
        sizes = np.clip(18 + scaled * 250, 16, 280)
        xs = sub['cellpair'].map(base_x).astype(float) + offsets[age_group]
        ys = sub['lr_pair'].map(ymap).astype(float)
        ax.scatter(xs, ys, s=sizes, c=colors, edgecolors='white',
                   linewidths=0.755, alpha=0.96, zorder=3, label=age_group)

    for cp in cellpair_order:
        x0 = base_x[cp] - 0.38
        ax.axvline(x0, color='#D2D2D2', lw=1.0, ls='--', zorder=0)
        ax.text(base_x[cp] + 0.36, -0.14, cp.replace('|', '→'),
                ha='right', va='top', rotation=90, fontsize=12,
                transform=ax.get_xaxis_transform(), color='#24415E')
    ax.axvline(base_x[cellpair_order[-1]] + 1.10, color='#D2D2D2', lw=1.0, ls='--', zorder=0)

    xticks, xticklabels = [], []
    for cp in cellpair_order:
        for age_group in ['Young adult', 'Elderly']:
            xticks.append(base_x[cp] + offsets[age_group])
            xticklabels.append('YA' if age_group == 'Young adult' else 'E')
    ax.set_xticks(xticks)
    ax.set_xticklabels(xticklabels, fontsize=12, rotation=0)
    for tick, label in zip(ax.get_xticklabels(), xticklabels):
        tick.set_color(YOUNG_COLOR if label == 'Y' else OLD_COLOR)
        tick.set_fontweight('bold')
    ax.set_yticks(np.arange(len(lr_order)))
    ax.set_yticklabels(lr_order, fontsize=12)
    for tick in ax.get_yticklabels():
        lr = tick.get_text()
        if lr in elderly_specific:
            tick.set_color(OLD_COLOR); tick.set_fontweight('bold')
        elif lr in young_adult_specific:
            tick.set_color(YOUNG_COLOR); tick.set_fontweight('bold')
    ax.set_xlabel('Sender→receiver cell pair, split into Young adult (YA) and Elderly (E) support')
    ax.set_ylabel('Ligand-receptor interaction')
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(-0.75, base_x[cellpair_order[-1]] + 1.35)
    ax.set_ylim(-0.7, len(lr_order) - 0.25)
    ax.grid(axis='y', alpha=0.10)
    group_handles = [
        mpl.lines.Line2D([0], [0], marker='o', color='none', markerfacecolor=YOUNG_COLOR,
                         markeredgecolor='white', markersize=8, label='Young adult support'),
        mpl.lines.Line2D([0], [0], marker='o', color='none', markerfacecolor=OLD_COLOR,
                         markeredgecolor='white', markersize=8, label='Elderly support'),
    ]
    size_handles = [
        plt.scatter([], [], s=np.clip(32 + frac * 260, 45, 330), c='#777777',
                    alpha=0.55, edgecolors='white', label=f'{frac * max_support:.2g}')
        for frac in [0.25, 0.50, 1.00]
    ]
    leg1 = ax.legend(handles=group_handles, frameon=False, loc='upper left', bbox_to_anchor=(1.01, 1.0), title='Age group')
    ax.add_artist(leg1)
    ax.legend(handles=size_handles, frameon=False, loc='upper left', bbox_to_anchor=(1.01, 0.72), title=support_label)
    fig.text(0.01, 0.01, 'Red LRI labels: Elderly-specific; blue LRI labels: Young adult-specific (donor fraction >= 0.10).',
             fontsize=12, color='#555555')
    plt.subplots_adjust(bottom=0.30, right=0.82)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    plot_df.to_csv(TAB_DIR / 'S7_lr_driver_young_old_FastCCC_style_circleplot_values.tsv', sep='\t', index=False)
    return fig


def plot_lr_gene_pathway_alluvial(gene_sets=None, enrich_df=None, title='LR genes and enriched pathways', outfile='Figure5K_LR_gene_pathway_alluvial', top_terms=8, top_genes=28):
    """Paper Fig. 4i-style alluvial connecting LR genes to enriched pathways.

    Uses GO/Reactome enrichment when available. If enrichment failed, falls back to CPDB
    classification of selected LR drivers so the biological interpretation panel is still drawn.
    """
    links = []
    if enrich_df is not None and not enrich_df.empty and {'gene_set','term'}.issubset(enrich_df.columns):
        df = enrich_df.copy()
        if 'adjusted_p' in df.columns:
            df = df.sort_values('adjusted_p')
        for _, row in df.head(top_terms).iterrows():
            genes = []
            if 'genes' in row and pd.notna(row['genes']):
                genes = [g.strip().upper() for g in re.split(r'[;,]', str(row['genes'])) if g.strip()]
            for gene in genes[:top_genes]:
                links.append({'gene': gene, 'pathway': str(row['term'])[:46], 'group': str(row.get('gene_set', 'enriched')), 'weight': 1.0})
    if not links and 'selected_lr_continuous' in globals() and selected_lr_continuous is not None and not selected_lr_continuous.empty:
        df = selected_lr_continuous.copy()
    elif not links and 'selected_lr_discovery' in globals() and selected_lr_discovery is not None and not selected_lr_discovery.empty:
        df = selected_lr_discovery.copy()
    elif not links and 'selected_lr' in globals() and selected_lr is not None and not selected_lr.empty:
        df = selected_lr.copy()
        if 'discovery_trend_label' in df.columns:
            tmp = df[df['discovery_trend_label'].isin(['nominal Spearman trend'])].copy()
            if not tmp.empty:
                df = tmp
        df = df.head(60)
        for _, row in df.iterrows():
            ligs, recs = split_lr_genes(row['lr_pair']) if 'split_lr_genes' in globals() else ([], [])
            direction = 'aging_up' if row.get('spearman_rho', 0) > 0 else 'aging_down'
            pathway = str(row.get('classification', 'CPDB classification'))[:46]
            for gene in (ligs + recs):
                links.append({'gene': gene.upper(), 'pathway': pathway, 'group': direction, 'weight': 1.0})
    link_df = pd.DataFrame(links)
    if link_df.empty:
        return None
    link_df = (link_df.groupby(['gene','pathway','group'], as_index=False)['weight'].sum()
               .sort_values('weight', ascending=False))
    top_gene_names = link_df.groupby('gene')['weight'].sum().sort_values(ascending=False).head(top_genes).index.tolist()
    top_pathways = link_df.groupby('pathway')['weight'].sum().sort_values(ascending=False).head(top_terms).index.tolist()
    link_df = link_df[link_df['gene'].isin(top_gene_names) & link_df['pathway'].isin(top_pathways)].copy()
    if link_df.empty:
        return None
    link_df.to_csv(TAB_DIR / 'S7_lr_driver_gene_pathway_alluvial_links.tsv', sep='\t', index=False)
    genes = top_gene_names
    pathways = top_pathways
    x_gene, x_path = 0.08, 0.92
    y_gene = {g: 1 - (i + 0.5) / max(len(genes), 1) for i, g in enumerate(genes)}
    y_path = {p: 1 - (i + 0.5) / max(len(pathways), 1) for i, p in enumerate(pathways)}
    base_palette = {'aging_up': '#C44E52', 'aging-down': '#3B74AF', 'aging_down': '#3B74AF', 'aging-up': '#C44E52'}
    palette = {g: base_palette.get(str(g), ('#C44E52' if 'up' in str(g).lower() else ('#3B74AF' if 'down' in str(g).lower() else '#8A8A8A'))) for g in sorted(link_df['group'].unique())}
    fig, ax = plt.subplots(figsize=(12, max(5.5, 0.22 * max(len(genes), len(pathways)) + 3)))
    for _, row in link_df.iterrows():
        g, p = row['gene'], row['pathway']
        if g not in y_gene or p not in y_path:
            continue
        y0, y1 = y_gene[g], y_path[p]
        verts = [(x_gene, y0), (0.36, y0), (0.64, y1), (x_path, y1)]
        path = mpl.path.Path(verts, [mpl.path.Path.MOVETO, mpl.path.Path.CURVE4, mpl.path.Path.CURVE4, mpl.path.Path.CURVE4])
        patch = mpl.patches.PathPatch(path, facecolor='none', edgecolor=palette[row['group']],
                                      lw=np.clip(row['weight'], 0.7, 4.0), alpha=0.42)
        ax.add_patch(patch)
    for gene, y in y_gene.items():
        ax.text(x_gene - 0.015, y, gene, ha='right', va='center', fontsize=12)
        ax.plot([x_gene, x_gene + 0.015], [y, y], color='#333333', lw=1.2)
    for pathway, y in y_path.items():
        ax.text(x_path + 0.015, y, pathway, ha='left', va='center', fontsize=12)
        ax.plot([x_path - 0.015, x_path], [y, y], color='#333333', lw=1.2)
    for group, color in palette.items():
        ax.plot([], [], color=color, lw=4, alpha=0.6, label=group)
    ax.legend(frameon=False, loc='lower center', bbox_to_anchor=(0.5, -0.06), ncol=min(4, len(palette)))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(title, fontweight='bold')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig

def plot_age_group_lr_bubble(lr_panel, model_df, entities, title, outfile, top_n=18):
    """FastCCC-style grouped LR bubble: color is age-group mean strength, size is donor occupancy."""
    if lr_panel is None or lr_panel.empty or not entities:
        return None
    entities = list(dict.fromkeys(entities))[:top_n]
    df = lr_panel[lr_panel['entity'].isin(entities)].copy()
    if df.empty:
        return None
    label_map = {}
    if model_df is not None and not model_df.empty and {'entity','cellpair','lr_pair'}.issubset(model_df.columns):
        meta = model_df.drop_duplicates('entity').set_index('entity')
        for ent in entities:
            if ent in meta.index:
                label_map[ent] = f"{str(meta.loc[ent, 'cellpair'])[:18]} | {str(meta.loc[ent, 'lr_pair'])[:30]}"
    grouped = (df.groupby(['entity','age_group'])
               .agg(mean_strength=('sum_strength','mean'),
                    sem_strength=('sum_strength', lambda x: float(pd.Series(x).sem()) if len(x) > 1 else 0.0),
                    donor_frac=('sum_strength', lambda x: float((pd.Series(x) > 0).mean())),
                    n_donors=('sample_id','nunique'))
               .reset_index())
    grouped['age_group'] = pd.Categorical(grouped['age_group'], categories=AGE_ORDER, ordered=True)
    grouped = grouped.dropna(subset=['age_group'])
    if grouped.empty:
        return None
    piv = grouped.pivot(index='entity', columns='age_group', values='mean_strength').reindex(index=entities, columns=AGE_ORDER)
    z = piv.sub(piv.mean(axis=1), axis=0).div(piv.std(axis=1).replace(0, np.nan), axis=0).fillna(0)
    z_long = z.reset_index().melt(id_vars='entity', var_name='age_group', value_name='row_z_strength')
    plot_df = grouped.merge(z_long, on=['entity','age_group'], how='left')
    ent_order = entities[::-1]
    xmap = {g: i for i, g in enumerate(AGE_ORDER)}
    ymap = {e: i for i, e in enumerate(ent_order)}
    fig, ax = plt.subplots(figsize=(8.8, max(5.2, 0.34 * len(ent_order))))
    sca = ax.scatter(plot_df['age_group'].map(xmap), plot_df['entity'].map(ymap),
                     s=np.clip(plot_df['donor_frac'] * 340, 35, 340),
                     c=plot_df['row_z_strength'], cmap='RdBu_r', vmin=-2.5, vmax=2.5,
                     edgecolors='white', linewidths=0.7, alpha=0.9)
    ax.set_xticks(range(len(AGE_ORDER)))
    ax.set_xticklabels(AGE_ORDER)
    ax.set_yticks(range(len(ent_order)))
    ax.set_yticklabels([label_map.get(e, e) for e in ent_order], fontsize=12)
    ax.set_xlabel('Age group')
    ax.set_ylabel('Cell pair | ligand-receptor')
    ax.set_title(title)
    cbar = fig.colorbar(sca, ax=ax, pad=0.02)
    cbar.set_label('row z-score mean strength')
    for frac, size in [(0.25, 85), (0.50, 170), (0.75, 255)]:
        ax.scatter([], [], s=size, c='#888888', alpha=0.55, edgecolors='white', label=f'{frac:.0%} donors')
    ax.legend(title='Occupancy', frameon=False, loc='lower right', bbox_to_anchor=(1.28, 0.0))
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    grouped.to_csv(TAB_DIR / 'S7_lr_driver_age_group_bubble_values.tsv', sep='\t', index=False)
    return fig

set_pub_style()


def plot_ranked_lr_spearman_dotplot(model_df, entities, title, outfile, top_n=20):
    """Ranked LR driver dotplot: Spearman rho is the main visual effect size."""
    if model_df is None or model_df.empty or not entities:
        return None
    df = model_df[model_df['entity'].isin(entities)].copy()
    if df.empty:
        return None
    df = visual_candidate_df(df, min_nonzero=0, min_sd=0.0)
    if df.empty:
        return None
    df = df.drop_duplicates('entity').copy()
    df['abs_spearman_rho'] = df['spearman_rho'].abs()
    df = df.sort_values(['spearman_p', 'abs_spearman_rho'], ascending=[True, False]).head(top_n)
    df = df.sort_values('spearman_rho').reset_index(drop=True)
    df['display_label'] = (df['cellpair'].astype(str).str.replace('|', '→', regex=False).str.slice(0, 24)
                           + ' | ' + df['lr_pair'].astype(str).str.replace('_', '–', regex=False).str.slice(0, 34))
    p_col = first_existing_col(df, ['spearman_p'], 'spearman_p')
    size_source = df['spearman_p'].clip(lower=1e-300) if 'spearman_p' in df.columns else df[p_col].clip(lower=1e-300)
    sizes = np.asarray(np.clip(-np.log10(size_source) * 36, 45, 340), dtype=float)
    vmax = max(float(df['spearman_rho'].abs().max()), 0.1)
    norm = mpl.colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    cmap = mpl.cm.get_cmap('RdBu_r')
    edge = np.where(df['discovery_trend_label'].eq('nominal Spearman trend'), '#222222', '#9A9A9A')
    alpha_map = {'robust trend': 0.95, 'cohort-sensitive trend': 0.72, 'exploratory trend': 0.58}
    alphas = df.get('cohort_sensitivity_label', pd.Series('exploratory trend', index=df.index)).map(alpha_map).fillna(0.58)
    fig, ax = plt.subplots(figsize=(8.6, max(5.2, 0.34 * len(df) + 1.2)))
    y = np.arange(len(df))
    ax.hlines(y, 0, df['spearman_rho'], color='#D6D6D6', lw=1.1, zorder=1)
    for i, row in df.iterrows():
        ax.scatter(row['spearman_rho'], y[i], s=sizes[i],
                   c=[cmap(norm(row['spearman_rho']))],
                   edgecolors=edge[i], linewidths=2.2,
                   alpha=float(alphas.iloc[i]), zorder=3)
    ax.axvline(0, color='#333333', lw=1.0, ls='--')
    ax.set_yticks(y)
    ax.set_yticklabels(df['display_label'], fontsize=12)
    ax.set_xlabel('Spearman rho with age')
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='x', alpha=0.16)
    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = fig.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label('Spearman rho')
    for p in [0.05, 0.01, 0.001]:
        ax.scatter([], [], s=np.clip(-np.log10(p) * 42, 65, 380),
                   c='#888888', edgecolors='white', label=f'p={p:g}')
    ax.legend(title='Spearman p', frameon=False, loc='lower right')
    ax.text(0.0, -0.11, 'Dot color: rho; dot size: Spearman p. Nominal Spearman p is used for discovery visualization.',
            transform=ax.transAxes, ha='left', va='top', fontsize=12, color='#555555')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    df.to_csv(TAB_DIR / 'S7_ranked_LR_spearman_dotplot_values.tsv', sep='\t', index=False)
    return fig


def select_lr_driver_entities(selected_lr_df, selected_lr_anchor=None, top_n=24):
    """Select balanced aging-up/down LRIs from the strict nominal discovery pool."""
    if selected_lr_df is None or selected_lr_df.empty:
        return [], 'no nominal discovery LR candidates'
    disc = selected_lr_df.copy()
    if 'aging_direction' not in disc.columns:
        disc['aging_direction'] = np.where(disc['spearman_rho'] > 0, 'aging-up',
                                           np.where(disc['spearman_rho'] < 0, 'aging-down', 'flat'))
    disc['abs_spearman_rho'] = disc['spearman_rho'].abs()
    disc = disc.sort_values(['spearman_p', 'abs_spearman_rho', 'donor_frac'], ascending=[True, False, False])
    up = disc[disc['spearman_rho'] > 0].head(max(1, top_n // 2))
    down = disc[disc['spearman_rho'] < 0].head(max(1, top_n // 2))
    balanced = pd.concat([up, down], ignore_index=True)
    if len(balanced) < top_n:
        balanced = pd.concat([balanced, disc[~disc['entity'].isin(balanced['entity'])]], ignore_index=True)
    balanced = balanced.drop_duplicates('entity').head(top_n)
    n_up = int((balanced['spearman_rho'] > 0).sum())
    n_down = int((balanced['spearman_rho'] < 0).sum())
    if n_up == 0:
        print('no aging-up LR passed donor_frac >= 0.35 and nominal discovery filters')
    if n_down == 0:
        print('no aging-down LR passed donor_frac >= 0.35 and nominal discovery filters')
    return balanced['entity'].tolist(), f'nominal Spearman discovery LRIs; not restricted to main-text Tier candidates ({n_up} up, {n_down} down)'


lr_plot_source = selected_lr_continuous if 'selected_lr_continuous' in globals() and not selected_lr_continuous.empty else (selected_lr_discovery if 'selected_lr_discovery' in globals() and not selected_lr_discovery.empty else selected_lr)
if not lr_plot_source.empty:
    pcl = first_existing_col(lr_plot_source, ['spearman_p','p_linear'], 'p_linear')
    top_lr_entities, lr_fallback_label = select_lr_driver_entities(
        lr_plot_source, selected_lr_anchor if 'selected_lr_anchor' in globals() else None, top_n=24)
    print(f'Figure 5 LR selection: {lr_fallback_label}')

    # 5A: Ranked LR driver dotplot, limited to prioritized LR drivers
    bubble_df = lr_plot_source[lr_plot_source['entity'].isin(top_lr_entities)].copy()
    if not bubble_df.empty:
        plot_ranked_lr_spearman_dotplot(
            bubble_df, top_lr_entities,
            f'5A Ranked LR drivers by continuous-age Spearman trend ({lr_fallback_label})',
            'Figure5A_ranked_LR_spearman_dotplot',
            top_n=20)

    # 5B: Top LR continuous age trajectories
    top6l = top_lr_entities[:6]
    if top6l and 'lr_agg' in globals() and not lr_agg.empty:
        plot_trend_grid(lr_agg, lr_plot_source, 'entity', 'sum_strength', top6l,
                        f'5B LR-level donor-strength continuous-age trajectories ({lr_fallback_label})',
                        'Figure5B_LR_level_donor_strength_trajectories', ncols=3)

    # 5C: AD/literature anchored LR dot map
    if 'selected_lr_anchor' in globals() and not selected_lr_anchor.empty:
        qa = first_existing_col(selected_lr_anchor, ['spearman_p','p_linear'], 'p_linear')
        ad_plot = visual_candidate_df(selected_lr_anchor.copy(), min_nonzero=8, min_sd=0.0)
        ad_plot['log10p'] = -np.log10(ad_plot[qa].clip(lower=1e-300))
        ad_plot = ad_plot.sort_values([qa, 'abs_spearman_rho'], ascending=[True, False]).head(20)
        ad_plot = ad_plot.iloc[::-1].copy()
        ad_plot['label'] = ad_plot['cellpair'].astype(str).str.slice(0, 16) + '\n' + ad_plot['lr_pair'].astype(str).str.slice(0, 26)
        fig5c, ax5c = plt.subplots(figsize=(9, 7))
        y = np.arange(len(ad_plot))
        colors = ['#C84E4E' if b >= 0 else '#3B74AF' for b in ad_plot['spearman_rho']]
        sizes = np.clip(ad_plot['log10p'].values * 24.0, 26, 280)
        ax5c.scatter(ad_plot['spearman_rho'], y, s=sizes, c=colors, alpha=0.85,
                     edgecolors='white', linewidths=0.6)
        ax5c.axvline(0, ls='--', lw=1.0, color='black')
        ax5c.set_yticks(y)
        ax5c.set_yticklabels(ad_plot['label'])
        ax5c.set_xlabel('Spearman rho with age')
        ax5c.set_title('5C Literature/AD anchored LR changes')
        plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
        save_fig(fig5c, 'Figure5C_literature_anchor_LR_dotmap', FIG_DIR)
        plt.show()

    # 5D: Donor-ordered LR heatmap
    if 'lr_agg' in globals() and not lr_agg.empty:
        donor_ordered_heatmap(lr_agg, 'entity', 'sum_strength', top_lr_entities,
                              f'5D LR-level donor-strength heatmap (row z-score; {lr_fallback_label})',
                              'Figure5D_LR_level_donor_strength_heatmap', figsize=(14, max(5, 0.30*len(top_lr_entities))))


    # 5F: FastCCC-style Young adult/Adult/Elderly grouped LR bubble as an interpretable companion to continuous age trends
    if 'lr_agg' in globals() and not lr_agg.empty and top_lr_entities:
        plot_age_group_lr_bubble(
            lr_agg, lr_plot_source, top_lr_entities,
            f'S5F Young adult/Adult/Elderly LR driver bubble ({lr_fallback_label})',
            'FigureS5F_LR_driver_age_group_bubble',
            top_n=18)


    # 5G: FastCCC paper-style grouped companion plots for continuous LR drivers.
    grouped_lri_entities = (selected_lr_continuous['entity'].drop_duplicates().tolist()
                            if 'selected_lr_continuous' in globals() and selected_lr_continuous is not None and not selected_lr_continuous.empty
                            else top_lr_entities)
    if 'fastccc_long' in globals() and not fastccc_long.empty and grouped_lri_entities:
        plot_fastccc_condition_circleplot(
            fastccc_long, grouped_lri_entities,
            f'S5G Young adult/Adult/Elderly FastCCC-style LRI small multiples ({lr_fallback_label})',
            'FigureS5G_YMO_FastCCC_style_LRI_small_multiples',
            top_n_lr=24,
            top_n_cellpairs=28)




    # 5G2: Exact FastCCC Fig.4h-style pairwise LRI matrices for Young adult/Adult/Elderly contrasts.
    # These grouped plots display the continuous LR driver set and do not define it.
    if 'fastccc_long' in globals() and not fastccc_long.empty and grouped_lri_entities:
        group_lr_fig4h = prepare_fig4h_group_lr(fastccc_long, selected_entities=grouped_lri_entities)
        if not group_lr_fig4h.empty:
            group_lr_fig4h.to_csv(TAB_DIR / 'S7_grouped_LRI_Fig4h_exact_support_table.tsv', sep='\t', index=False)
            for g1, g2, tag in [('Young adult', 'Adult', 'early'), ('Adult', 'Elderly', 'late'), ('Young adult', 'Elderly', 'full_aging')]:
                plot_fig4h_lri_contrast(
                    group_lr_fig4h, g1, g2, top_n=30, min_frac=0.05,
                    outfile=f'Figure5G_FastCCC_Fig4h_LRI_{g1}_vs_{g2}',
                    title=f'5G FastCCC Fig.4h-style LRI support - {g1} vs {g2} ({tag}; {lr_fallback_label})')
        plot_young_middle_old_lri_circleplot(
            fastccc_long, grouped_lri_entities,
            f'S5G Young adult/Adult/Elderly FastCCC-style LRI circle plot ({lr_fallback_label})',
            'FigureS5G_YMO_FastCCC_style_LRI_circleplot',
            top_n_lr=22,
            top_n_cellpairs=18)
    lr_gene_enrich_for_plot = globals().get('lr_gene_enrichment', pd.DataFrame())
    lr_gene_sets_for_plot = globals().get('lr_gene_sets', pd.DataFrame())
    plot_lr_gene_pathway_alluvial(
        gene_sets=lr_gene_sets_for_plot,
        enrich_df=lr_gene_enrich_for_plot,
        title='5K LR genes connected to enriched pathways or CPDB classes',
        outfile='Figure5K_LR_gene_pathway_alluvial',
        top_terms=8,
        top_genes=28)

    # 5H/5I: FastCCC application-style LRI set comparison and age-specific communication network.
    lri_support, lri_membership = summarize_age_group_lri_sets(
        fastccc_long if 'fastccc_long' in globals() else pd.DataFrame(),
        selected_entities=grouped_lri_entities if 'grouped_lri_entities' in globals() else top_lr_entities,
        min_donor_frac=0.10)
    if not lri_support.empty:
        lri_support.to_csv(TAB_DIR / 'S7_lr_driver_age_group_lri_support.tsv', sep='\t', index=False)
    if not lri_membership.empty:
        lri_membership.to_csv(TAB_DIR / 'S7_lr_driver_age_group_lri_membership.tsv', sep='\t', index=False)
        plot_age_group_lri_overlap(
            lri_membership,
            '5H Young adult/Adult/Elderly LRI set overlap among selected LR drivers',
            'Figure5H_age_group_LRI_set_overlap')
        plot_specific_lri_network(
            lri_membership, mode='elderly_specific',
            title='5I Elderly-specific LRI communication network',
            outfile='Figure5I_elderly_specific_LRI_network')
        plot_specific_lri_network(
            lri_membership, mode='young_adult_specific',
            title='Figure S5 Young adult-specific LRI communication network',
            outfile='FigureS5_young_adult_specific_LRI_network')

    # 5J: Ligand/receptor expression validation for top LR drivers.
    expr_entities = top_lr_entities[:8]
    lr_expr_panel = load_top_lr_expression_panel(expr_entities, top_n=8)
    if not lr_expr_panel.empty:
        plot_lr_expression_validation(
            lr_expr_panel,
            '5J Ligand/receptor expression validation for top LR drivers',
            'Figure5J_LR_ligand_receptor_expression_validation',
            max_panels=8)

    # 5E: Receptor-TF Sankey
    def _as_mpl_figure(obj):
        """Return the matplotlib Figure from helpers that may return fig, ax or (fig, ax)."""
        if obj is None:
            return None
        if hasattr(obj, 'savefig'):
            return obj
        if isinstance(obj, (tuple, list)):
            for item in obj:
                fig = _as_mpl_figure(item)
                if fig is not None:
                    return fig
        return None

    fig5e_obj = plot_receptor_tf_sankey(
        lr_plot_source[lr_plot_source['entity'].isin(top_lr_entities)],
        receptor_tf,
        lr_col='lr_pair',
        top_n_lr=10,
        figsize=(8, 8),
    )
    fig5e = _as_mpl_figure(fig5e_obj)
    if fig5e is not None:
        save_fig(fig5e, 'Figure5E_receptor_TF_sankey', FIG_DIR)
        plt.show()
    else:
        print('Figure 5E skipped: plot_receptor_tf_sankey returned no matplotlib Figure')

    print('Figure 5 saved')
else:
    print('No selected LR drivers for Figure 5')


## Figure 6 — 网络角色变化

 Shows donor-level trajectories, age-bin mean plus SEM, and age-bin top role transitions.


In [ ]:


def plot_sender_receiver_role_arrows(panel, title, outfile):
    """CellChat-style sender/receiver role scatter with Young-to-Old arrows."""
    if panel is None or panel.empty:
        return None
    metrics = ['sender_share', 'receiver_share']
    df = panel[panel['metric_name'].isin(metrics)].copy()
    df = df[df['age_group'].isin(['Young adult', 'Adult', 'Elderly'])].copy()
    if df.empty:
        return None
    means = (df.groupby(['celltype','age_group','metric_name'], observed=True)['metric_value']
               .mean().reset_index())
    wide = (means.pivot_table(index=['celltype','age_group'], columns='metric_name',
                              values='metric_value', fill_value=0.0)
              .reset_index())
    wide.to_csv(TAB_DIR / f'{outfile}_source.tsv', sep='\t', index=False)
    y = wide[wide['age_group'] == 'Young adult'].set_index('celltype')
    m = wide[wide['age_group'] == 'Adult'].set_index('celltype')
    o = wide[wide['age_group'] == 'Elderly'].set_index('celltype')
    common = sorted(set(y.index) & set(o.index))
    if not common:
        return None
    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    all_x = wide['sender_share'].values if 'sender_share' in wide.columns else np.array([0])
    all_y = wide['receiver_share'].values if 'receiver_share' in wide.columns else np.array([0])
    pad_x = (np.nanmax(all_x) - np.nanmin(all_x)) * 0.08 + 1e-6
    pad_y = (np.nanmax(all_y) - np.nanmin(all_y)) * 0.08 + 1e-6
    shifts = []
    for ct in common:
        x0, y0 = y.loc[ct, 'sender_share'], y.loc[ct, 'receiver_share']
        x1, y1 = o.loc[ct, 'sender_share'], o.loc[ct, 'receiver_share']
        shifts.append((ct, float(np.hypot(x1 - x0, y1 - y0))))
        ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='->', lw=1.0, color='#7A7A7A', alpha=0.65,
                                    shrinkA=3, shrinkB=3))
    ax.scatter(y.loc[common, 'sender_share'], y.loc[common, 'receiver_share'],
               s=52, c=AGE_PAL.get('Young adult', YOUNG_COLOR), label='Young adult',
               edgecolors='white', linewidths=0.85, zorder=3)
    if not m.empty:
        mid_common = [ct for ct in common if ct in m.index]
        ax.scatter(m.loc[mid_common, 'sender_share'], m.loc[mid_common, 'receiver_share'],
                   s=60, c=AGE_PAL.get('Adult', '#E3A53A'), label='Adult',
                   edgecolors='white', linewidths=0.85, zorder=3, alpha=0.9)
    ax.scatter(o.loc[common, 'sender_share'], o.loc[common, 'receiver_share'],
               s=46, c=AGE_PAL.get('Elderly', OLD_COLOR), label='Elderly',
               edgecolors='white', linewidths=0.85, zorder=4)
    top_label = [ct for ct, _ in sorted(shifts, key=lambda x: x[1], reverse=True)[:10]]
    for ct in top_label:
        ax.annotate(str(ct)[:18], (o.loc[ct, 'sender_share'], o.loc[ct, 'receiver_share']),
                    fontsize=12, xytext=(3, 3), textcoords='offset points')
    ax.set_xlim(np.nanmin(all_x) - pad_x, np.nanmax(all_x) + pad_x)
    ax.set_ylim(np.nanmin(all_y) - pad_y, np.nanmax(all_y) + pad_y)
    ax.set_xlabel('Outgoing communication role (mean sender share)')
    ax.set_ylabel('Incoming communication role (mean receiver share)')
    ax.set_title(title, fontsize=12)
    ax.grid(alpha=0.18)
    ax.legend(loc='best', frameon=False)
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def make_network_age_bins(panel, n_bins=5):
    df = panel.copy()
    age_lookup = donor_base[['sample_id','age']].drop_duplicates().copy()
    if 'age' not in df.columns:
        df = df.merge(age_lookup, on='sample_id', how='left')
    # Quantile bins keep multiple donors per bin even when ages are unevenly sampled.
    try:
        bins = pd.qcut(age_lookup['age'], q=min(n_bins, age_lookup['age'].nunique()), duplicates='drop')
        bin_map = dict(zip(age_lookup['sample_id'], bins.astype(str)))
        df['age_bin'] = df['sample_id'].map(bin_map)
    except Exception:
        df['age_bin'] = pd.cut(df['age'], bins=n_bins).astype(str)
    bin_stats = (df[['age_bin','sample_id','age']].dropna()
                 .drop_duplicates(['age_bin','sample_id'])
                 .groupby('age_bin', observed=True)
                 .agg(age_mid=('age','mean'), age_min=('age','min'),
                      age_max=('age','max'), n_donors=('sample_id','nunique'))
                 .reset_index())
    bin_stats['age_bin_label'] = bin_stats.apply(lambda r: f"{r.age_min:.0f}-{r.age_max:.0f}y", axis=1)
    order = bin_stats.sort_values('age_mid')['age_bin'].tolist()
    label_map = dict(zip(bin_stats['age_bin'], bin_stats['age_bin_label']))
    df['age_bin'] = pd.Categorical(df['age_bin'], categories=order, ordered=True)
    df['age_bin_label'] = df['age_bin'].astype(str).map(label_map)
    return df, bin_stats.sort_values('age_mid')


def plot_network_role_agebin_errorbars(panel, entities, title, outfile, ncols=4):
    if panel is None or panel.empty or not entities:
        return None
    df, bin_stats = make_network_age_bins(panel, n_bins=5)
    df = df[df['role_entity'].isin(entities)].copy()
    if df.empty:
        return None
    summary = (df.groupby(['role_entity','age_bin','age_bin_label'], observed=True)
               .agg(mean_value=('metric_value','mean'),
                    sem_value=('metric_value', lambda x: float(pd.Series(x).sem()) if len(x) > 1 else 0.0),
                    n_donors=('sample_id','nunique'),
                    age_mid=('age','mean'))
               .reset_index())
    summary.to_csv(TAB_DIR / 'S8_network_role_agebin_summary.tsv', sep='\t', index=False)
    n = len(entities)
    ncols = min(ncols, max(1, n))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.4*ncols, 4.3*nrows), squeeze=False)
    axes = axes.flatten()
    for ax, ent in zip(axes, entities):
        sub = summary[summary['role_entity'] == ent].sort_values('age_mid')
        ax.errorbar(sub['age_mid'], sub['mean_value'], yerr=sub['sem_value'],
                    marker='o', lw=1.8, ms=4.5, color='#2F6F73', ecolor='#8BB8BA', capsize=2)
        ax.set_title(str(ent).replace('::', ' | '), fontsize=12)
        ax.set_xlabel('Age bin midpoint')
        ax.set_ylabel('Mean role score')
        ax.grid(alpha=0.18)
    for ax in axes[n:]:
        ax.axis('off')
    fig.suptitle(title, y=1.02, fontweight='bold')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig


def plot_network_role_rank_transition(panel, metrics, title, outfile, top_k=3):
    if panel is None or panel.empty:
        return None
    df, bin_stats = make_network_age_bins(panel, n_bins=5)
    role_df = (df[df['metric_name'].isin(metrics)]
               .groupby(['metric_name','age_bin','age_bin_label','celltype'], observed=True)['metric_value']
               .mean().reset_index())
    if role_df.empty:
        return None
    role_df['rank'] = role_df.groupby(['metric_name','age_bin'], observed=True)['metric_value'].rank(ascending=False, method='first')
    top = role_df[role_df['rank'] <= top_k].copy()
    top['rank'] = top['rank'].astype(int)
    top = top.merge(bin_stats[['age_bin','age_mid']], on='age_bin', how='left')
    top = top.sort_values(['metric_name','age_mid','rank'])
    top.to_csv(TAB_DIR / 'S8_network_role_agebin_top_cells.tsv', sep='\t', index=False)
    metric_order = [m for m in metrics if m in top['metric_name'].unique()]
    if not metric_order:
        return None
    fig, axes = plt.subplots(len(metric_order), 1, figsize=(13.5, 3.1 * len(metric_order)), squeeze=False)
    axes = axes.flatten()
    bin_order = bin_stats['age_bin'].tolist()
    bin_labels = dict(zip(bin_stats['age_bin'], bin_stats['age_bin_label']))
    for ax, metric in zip(axes, metric_order):
        sub = top[top['metric_name'] == metric].copy()
        mat = np.full((top_k, len(bin_order)), np.nan)
        labels = [['' for _ in bin_order] for __ in range(top_k)]
        for _, row in sub.iterrows():
            if row['age_bin'] in bin_order:
                x = bin_order.index(row['age_bin'])
                y = int(row['rank']) - 1
                mat[y, x] = row['metric_value']
                labels[y][x] = str(row['celltype'])[:18]
        sns.heatmap(mat, ax=ax, cmap='YlGnBu', cbar=True, linewidths=0.85, linecolor='white',
                    xticklabels=[bin_labels.get(b, str(b)) for b in bin_order],
                    yticklabels=[f'Top {i}' for i in range(1, top_k + 1)])
        for y in range(top_k):
            for x in range(len(bin_order)):
                ax.text(x + 0.5, y + 0.5, labels[y][x], ha='center', va='center', fontsize=12, color='#1f1f1f')
        ax.set_title(f'{metric}: top cell-type role transition')
        ax.set_xlabel('Age bin')
        ax.set_ylabel('Rank')
    fig.suptitle(title, y=1.02, fontweight='bold')
    plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
    save_fig(fig, outfile, FIG_DIR)
    plt.show()
    return fig

set_pub_style()

if not network_models.empty:
    qcnet = first_existing_col(network_models, ['spearman_p','p_overall'], None)
    nm = network_models[network_models['metric_name'].isin(['sender_share','receiver_share','pagerank_ext','hub_score','betweenness'])].copy()
    nm['role_entity'] = nm['celltype'].astype(str) + '::' + nm['metric_name'].astype(str)
    npanel = network_panel.copy()
    npanel['role_entity'] = npanel['celltype'].astype(str) + '::' + npanel['metric_name'].astype(str)
    top_roles = select_balanced_visual_entities(nm, 'role_entity', top_n=12, min_up=6, min_down=6, label='Figure 6 network roles')
    if not top_roles:
        nm_tmp = nm.copy()
        nm_tmp['abs_spearman_rho'] = nm_tmp['spearman_rho'].abs()
        top_roles = nm_tmp.sort_values(['spearman_p','abs_spearman_rho'], ascending=[True, False])['role_entity'].drop_duplicates().head(12).tolist()

    save_visual_selection_table(nm, 'role_entity', top_roles, 'Figure6_balanced_network_role_selection')

    # 6A: Donor-level continuous trajectories, one point per donor age.
    plot_trend_grid(npanel, nm, 'role_entity', 'metric_value', top_roles[:8],
                    '6A Network role donor-level continuous trajectories',
                    'Figure6A_network_role_donor_level_trajectories', ncols=4)

    # 6B: Age-bin mean plus SEM, using multiple donors per bin.
    plot_network_role_agebin_errorbars(
        npanel, top_roles[:8],
        '6B Network role age-bin mean plus SEM',
        'Figure6B_network_role_agebin_errorbars', ncols=4)

    # 6C: Which cell types occupy top network roles across age bins.
    plot_network_role_rank_transition(
        npanel,
        metrics=['pagerank_ext','sender_share','receiver_share','hub_score'],
        title='6C Network core role transition across age bins',
        outfile='Figure6C_network_role_rank_transition_tiles',
        top_k=3)

    # 6D: Sender vs receiver Spearman scatter with limited labels.
    beta_plot_col = 'spearman_rho'
    nm_sr = visual_candidate_df(network_models.copy(), min_nonzero=8, min_sd=0.0)
    s_beta = (nm_sr[nm_sr['metric_name']=='sender_share']
              .set_index('celltype')[beta_plot_col].rename('sender_beta'))
    r_beta = (nm_sr[nm_sr['metric_name']=='receiver_share']
              .set_index('celltype')[beta_plot_col].rename('receiver_beta'))
    s_q = (nm_sr[nm_sr['metric_name']=='sender_share']
           .set_index('celltype')[qcnet].rename('sender_q') if qcnet in nm_sr.columns else pd.Series(dtype=float))
    r_q = (nm_sr[nm_sr['metric_name']=='receiver_share']
           .set_index('celltype')[qcnet].rename('receiver_q') if qcnet in nm_sr.columns else pd.Series(dtype=float))
    sr_df = pd.concat([s_beta, r_beta, s_q, r_q], axis=1).dropna(subset=['sender_beta','receiver_beta']).reset_index().rename(columns={'index':'celltype'})
    if not sr_df.empty:
        sr_df['change_score'] = sr_df[['sender_beta','receiver_beta']].abs().max(axis=1)
        fig6d, ax6d = plt.subplots(figsize=(6.5, 6.2))
        ax6d.scatter(sr_df['sender_beta'], sr_df['receiver_beta'], s=55,
                     c='#6C7A89', alpha=0.55, edgecolors='white', linewidths=0.85)
        ax6d.axhline(0, ls='--', lw=1.0, color='black')
        ax6d.axvline(0, ls='--', lw=1.0, color='black')
        for _, row in sr_df.nlargest(min(8, len(sr_df)), 'change_score').iterrows():
            ax6d.annotate(str(row['celltype'])[:18], (row['sender_beta'], row['receiver_beta']),
                          fontsize=12, xytext=(3,3), textcoords='offset points')
        ax6d.set_xlabel('sender role Spearman rho')
        ax6d.set_ylabel('receiver role Spearman rho')
        ax6d.set_title('6D Sender and receiver role shifts across age')
        plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
        save_fig(fig6d, 'Figure6D_sender_receiver_limited_labels', FIG_DIR)
        plt.show()

    # 6E: Compact global overview heatmap.
    nm_heat = visual_candidate_df(network_models.copy(), min_nonzero=8, min_sd=0.0)
    if not nm_heat.empty:
        hm = nm_heat.pivot_table(index='celltype', columns='metric_name', values='spearman_rho', aggfunc='mean')
        fig6e, ax6e = plt.subplots(figsize=(8, max(5, 0.28 * len(hm))))
        sns.heatmap(hm, cmap='RdBu_r', center=0, ax=ax6e,
                    cbar_kws={'label': 'Spearman rho'}, linewidths=0.6, linecolor='white')
        ax6e.set_title('6E Network role Spearman aging trends')
        ax6e.set_xlabel('Network role metric')
        ax6e.set_ylabel('Cell type')
        plt.tight_layout(pad=2.2, w_pad=2.2, h_pad=2.6)
        save_fig(fig6e, 'Figure6E_centrality_spearman_heatmap', FIG_DIR)
        plt.show()



    # 6F: CellChat-style role migration scatter; arrows show Young to Old shifts.
    plot_sender_receiver_role_arrows(
        network_panel,
        '6F Sender-receiver role migration from Young to Old',
        'Figure6F_sender_receiver_role_migration_arrows')

    print('Figure 6 saved')
else:
    print('No network_models for Figure 6')


## Analysis Complete

All outputs are saved under `OUT`:

- `tables/`: continuous-age models, JT ordered trends, LR-level trajectory tables, and unified evidence tables
- `/home/\u56fe\u7247/70`: manuscript-ready vector figures (`.pdf` and `.svg`)
- `fastccc_run_summary.tsv`: FastCCC run status

Main interpretation: continuous-age modeling is the primary evidence layer. Young adult/Adult/Elderly grouping, JT tests, and transition-style views are used only to explain early/late changes and improve readability.

Main limitations: sex is unavailable in the sample table, so the OLS models do not adjust for sex. FastCCC reference-aware analysis and external PFC validation can be added later as stronger validation layers.


In [ ]:
print('=' * 60)
print('Analysis complete!')
print(f'Output: {OUT}')
print()
print('Tables:')
for f in sorted(TAB_DIR.glob('*.tsv')):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')
print()
print('Figures (vector PDF/SVG):')
for f in sorted(list(FIG_DIR.glob('*.pdf')) + list(FIG_DIR.glob('*.svg'))):
    print(f'  {f.name}')